# Version 3: patient-level 4-class fundus classification on ODIR-5K (Cataract / referable DR / Glaucoma / Normal) at 384 px with label auditing and class balancing

Dataset (the only one used): Kaggle **`andrewmvd/ocular-disease-recognition-odir5k`** = the official ODIR-5K per-eye table
(`full_df.csv`: one row per eye image, its label and the doctor's diagnostic keywords for that eye, patient id) and its
**512 x 512 px** images (`preprocessed_images/`, 6,392 images). The Version 3 data analysis (`plan.md`) found that the
popular Kaggle copy `tanjemahamed/odir5k-classification` is the same set shrunk to 224 px, with red/blue swapped and patient ids
removed, so the official files are used directly.

1. **Classes C, D, G, N** (the paper's four). A / H / M / O eyes are read only for the duplicate and label-conflict audit.
2. **Referable DR:** a DR eye is kept only with moderate non-proliferative DR or worse (ICDR); mild NPDR and DR without a grade
   are removed (`dr_min_grade=2`). Mild NPDR (microaneurysms only) was the main error source of Version 2.
3. **Cleaning before any split** (excluded images are never trained, validated or tested on): eyes whose keywords name a second
   ODIR disease (single-diagnosis cohort), "suspected ..." labels, ungradable images (black / white-out / blurred; cataract exempt
   from the blur rule), near-duplicate copies and label conflicts. White letterbox corners are set to black.
4. **Patient-level 5-fold CV:** both eyes of a patient are always in the same fold (patient ids from the official table).
5. **Higher resolution:** ResNet50 at 384 px, DINOv2 ViT-B/14 at 336 px (Version 2: 224-256 px).
6. **Class balancing** (normal : glaucoma ~ 13 : 1): class-balanced sampling (1/sqrt(n)) + loss weight (1/sqrt(n)); the final
   decision prior is set by post-hoc logit adjustment (`decision_prior_power`, pre-specified 0.5); Part 12 shows 1.0 / 0.5 / 0.0.
7. **Protocol comparison:** ResNet50 is also trained once on a random image-level split (patients ignored), as in many published
   papers, to measure how much that protocol inflates accuracy (`tab_protocol`, reported separately, never as the main result).

Unchanged from Version 2: round FOV crop, CLAHE, stratified group K-fold, GPU augmentation, EMA, early stopping on validation NLL,
flip TTA, stacking + temperature scaling, McNemar/Holm, corrected t-test, Friedman, cluster bootstrap, shortcut baselines,
quantitative Grad-CAM. Run time: preset `fast` (default) ~3.7 h on GPU T4 x2; `lite` ~1.5 h; `full` ~5 h.

**Run on Kaggle:** Accelerator **GPU T4 x2**, Internet **on**, add `andrewmvd/ocular-disease-recognition-odir5k`, then
**Save Version -> Save & Run All (Commit)**. Results: `eyev3/v3_results.zip` (Output tab; also linked at the end). All figures
are 600 dpi PNG + vector PDF.

## Part 01: Install extra packages (grad-cam, timm) (`part01_install.py`)

Installs `grad-cam` and `timm` if missing, and upgrades `timm` to >= 1.0.20 (needed for the DINOv3 ViT-B/16 weights).

In [ ]:
# ==============================================================================
# PART 1/16: Install extra packages (grad-cam, timm >= 1.0.20)
# Kaggle: paste into the FIRST code cell of an empty notebook. Run the parts in order.
# ==============================================================================
import importlib.util, importlib.metadata, re, subprocess, sys
print("[Part 1] Checking extra dependencies (grad-cam, timm)...", flush=True)
for _mod, _pkg in (("pytorch_grad_cam", "grad-cam"), ("timm", "timm>=1.0.20")):
    if importlib.util.find_spec(_mod) is None:
        print(f"Installing {_pkg}...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)
# The DINOv3 ViT weights (vit_base_patch16_dinov3.lvd1689m) exist only in timm >= 1.0.20 (Sept 2025).
try:
    _tv = importlib.metadata.version("timm")
except importlib.metadata.PackageNotFoundError:
    _tv = "0"
if tuple(int(v) for v in re.findall(r"\d+", _tv)[:3]) < (1, 0, 20):
    print(f"Upgrading timm {_tv} -> >=1.0.20 (needed for the DINOv3 ViT weights)...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "timm>=1.0.20"], check=False)
    _tv = importlib.metadata.version("timm")
print(f"[Part 1 done] All dependencies ready (timm {_tv}).", flush=True)

## Part 02: Imports, settings (CFG), output folders, figure style (`part02_setup_config.py`)

**All settings are here.** `PRESET`: `fast` (default, 5 folds x 10 epochs, ResNet50 384 px + DINOv2 336 px, ~3.7 h on T4 x2),
`lite` (3 folds, DINOv2 at 224 px, ~1.5 h), `full` (5 folds x 14 epochs, ~5 h). Data cleaning: `drop_label_conflicts`,
`drop_non_fundus`, `drop_ungradable`, `dedup_keep_one`, `drop_multi_disease`, `drop_uncertain_labels`, `dr_min_grade`.
Balancing: `sampler_power`, `imbalance`, `decision_prior_power`. Changing a training-relevant setting gives a new *settings id*;
finished runs are only reused when the id matches.

In [ ]:
# ==============================================================================
# PART 2/16: Imports, settings (CFG), output folders, figure style   <-- edit settings here
# Kaggle: paste into a NEW code cell directly below Part 1. Run the parts in order.
# ==============================================================================
import os, json, time, math, glob, random, re, hashlib, shutil, warnings, zlib, gc, copy
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models
from torchvision.transforms import v2 as T
from sklearn.model_selection import StratifiedGroupKFold, train_test_split, cross_val_predict
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                             f1_score, cohen_kappa_score, matthews_corrcoef, roc_auc_score, roc_curve,
                             confusion_matrix)
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
T0 = time.time()


def pbar(it=None, **kw):
    """Progress bar: animated in the notebook editor, text in the commit log."""
    kw.setdefault("mininterval", 1.0)
    kw.setdefault("dynamic_ncols", True)
    return tqdm(it, **kw)


def step(msg):
    """Timestamped status line (clock time | minutes since start)."""
    print(f"[{time.strftime('%H:%M:%S')} | {(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)
IS_KAGGLE = os.path.isdir("/kaggle/input")

# Version 3 presets (times from the measured Version 2 epoch times on Kaggle "GPU T4 x2", scaled to ~4,350 images and the
# higher input sizes; Part 9 prints the projected end after every fold and never starts a run that would end after
# time_budget_h):
#   "fast" (default) 5 folds, 10 epochs, ResNet50 @384 + DINOv2 @336 + protocol-comparison run ........ ~3.7 h
#   "lite"           3 folds, 10 epochs, ResNet50 @384 + DINOv2 @224 + protocol-comparison run ........ ~1.5 h
#   "full"           5 folds, 14 epochs, as "fast" ...................................................... ~5 h
PRESET = "fast"

CFG = dict(
    # None = auto-detect the folder with full_df.csv + preprocessed_images (andrewmvd/ocular-disease-recognition-odir5k)
    data_root=None,
    # Classes = ODIR label letters: C = cataract, D = DR, G = glaucoma, N = normal (the paper's 4 classes). A = AMD,
    # H = hypertensive, M = myopia, O = other are read only for the duplicate / label-conflict audit.
    classes=["C", "D", "G", "N"],
    audit_all_folders=True,
    # Class balancing (normal : glaucoma ~ 13 : 1). Two parts that multiply:
    #   sampler_power: training images are drawn with probability ~ 1/(class size)^p (0 = plain shuffling,
    #                  0.5 = square-root balancing, 1 = every class equally often); drawn with replacement.
    #   imbalance:     loss weight "sqrt_inv" = 1/sqrt(class count), "inv" = 1/count, "none".
    # 0.5 + "sqrt_inv" = every class has the same total weight in training. No image is copied to disk or synthesised.
    sampler_power=0.5,
    imbalance="sqrt_inv",
    # Decision prior (Part 11): every model's probabilities are moved from its (balanced) training prior to
    # class frequency^decision_prior_power by post-hoc logit adjustment; the stacker uses the same prior.
    # 1.0 = natural frequencies (highest accuracy), 0.0 = balanced (highest balanced accuracy), 0.5 = pre-specified
    # compromise. Part 12 also reports the proposed model under 1.0 / 0.5 / 0.0 (tab_decision_prior).
    decision_prior_power=0.5,
    # One camera pipeline (ODIR-5K): no acquisition differences between classes to hide. The Version 2 random-resolution
    # and JPEG augmentations would only blur small lesions, so they are off.
    source_robust_aug=False,
    # Source-shift test (Version 2) needs several sources; ODIR-5K is one source, so it is off.
    source_shift_test=False, source_shift_normal_frac=0.2,
    source_shift_models=[],
    # Protocol comparison (Part 14): these models are also trained once on a RANDOM IMAGE-LEVEL split (patients and
    # fellow eyes ignored), the protocol of many published papers, to measure how much it inflates the numbers.
    naive_split_test=True, naive_split_models=["resnet50"], naive_test_frac=0.2,
    work_dir="/kaggle/working/eyev3" if IS_KAGGLE else "./eyev3_work",
    cache_dir="/tmp/eyev3_cache",          # preprocessed uint8 arrays (not saved as output)
    img_size=384,                         # cached global view (official images: 512 x 512); per-model sizes below
    disc_size=192,                        # optic-disc ROI (used for the XAI pointing game)
    # ---- Version 3 data handling ----
    swap_rb="auto",                       # colour-channel order check (the official files are normal RGB)
    remove_white_pad=True,                # some images have white letterbox corners: set them to black before the FOV crop
    drop_multi_disease=True,              # single-diagnosis cohort: the eye's keywords name only ONE of these diseases
    single_diagnosis_classes=["A", "C", "D", "G", "H", "M"],
    drop_uncertain_labels=True,           # label given only as "suspected ..." (32 "suspected glaucoma" eyes)
    dr_min_grade=2,                       # referable DR: 2 = moderate NPDR or worse (ICDR); 0 = keep every DR eye
    min_class_eyes=150,                   # class viability: at least this many unique eyes after cleaning
    # Per-model overrides. ViT-B/16 (torchvision) only exists at 224; VGG16 at 384 costs ~3x more GPU time for
    # the weakest baseline. DINOv2 ViT-B/14 stays at 224 (16x16 tokens): attention cost grows with the square of
    # the token count, so 336 px would cost ~4x more and not fit the Kaggle session.
    # ConvNeXt-T at 224 px: its depthwise convolutions are slow on T4 (35 min per fold in Version 1 at 256 px).
    # ResNet50 at 384 px (fast on T4); DINOv2 at 336 px (24 x 24 tokens; 448 would cost ~2x more). ConvNeXt-T is not used:
    # its depthwise convolutions are slow on T4 (24 min per fold at 224 px in Version 2).
    size_override={"resnet50": 384, "dinov2_vitb14": 336, "convnext_tiny": 224, "vit_b_16": 224, "dinov3_vitb16": 224},
    lr_override={"vit_b_16": 3e-5, "dinov2_vitb14": 5e-5, "dinov3_vitb16": 5e-5},
    bs_override={},                       # batch 32 for every model (with T4 x2: 16 per GPU)
    layer_decay=0.75,                     # layer-wise LR decay for DINOv2 (transformer fine-tuning)
    drop_path=0.1,                        # stochastic depth for timm models
    ema=True, ema_decay=0.998,            # exponential moving average of weights (evaluated + saved)
    disc_frac=0.30,                       # disc ROI side = 30% of the field-of-view width
    clahe=True,
    circle_mask=True,                     # round FOV crop (removes stretched / letterboxed borders)
    # ODIR-5K images are already cropped to the field of view (it touches the image border), so the circle keeps 98% of the
    # short side (fov_keep=1.0); Version 2 used 0.84 to equalise several sources.
    fov_keep=1.0,
    harmonise_side=0, harmonise_jpeg_q=0,      # one source: no resizing / JPEG re-encoding needed
    n_folds=5,                            # preset "lite": 3. 5 folds = 80% of the data for training (small classes!)
    inner_splits=8,                       # 1/8 of the training part -> validation
    seed=42,
    # 10 epochs (Version 2: 10 of 12 runs were still improving at its last epoch 8). Preset "full": 14.
    epochs=10, patience=3, min_epochs=5,
    batch_size=32,
    lr=1e-4, head_lr=1e-3, weight_decay=0.05, warmup_epochs=1,
    label_smoothing=0.1, mixup_alpha=0.2, mixup_prob=0.5, grad_clip=1.0,
    # Loader workers only copy bytes (augmentation runs on the GPU, Part 8); one core stays free for the main
    # process, which drives both GPUs.
    num_workers=min(3, max(1, (os.cpu_count() or 2) - 1)),
    multi_gpu=True,                       # Kaggle "GPU T4 x2": split every batch over both GPUs (DataParallel)
    tta=True,                             # flip test-time augmentation (4 views)
    # Version 3 models (Version 2 pooled accuracy, 4 classes, 224-256 px):
    #   resnet50       ResNet50 (83.97%), fast on T4 -> now at 384 px
    #   dinov2_vitb14  DINOv2 ViT-B/14, self-supervised vision transformer (84.64%) -> now at 336 px
    # Dropped: ConvNeXt-T (too slow on T4 at high resolution), DINOv3 ViT-B/16 (82.24%, last on every fold).
    global_models=["resnet50", "dinov2_vitb14"],
    disc_models=[],
    run_folds=None,                       # e.g. [0, 1] to split the work across sessions
    # Never start a run that would end after this many hours (from Part 2); Parts 10-16 need ~15 min after it.
    time_budget_h=4.6,                    # preset "lite": 2.2, "full": 6.2 (Kaggle limit: 12 h)
    save_weights_fold=0,                  # weights of this fold are kept for Grad-CAM
    dup_hash_bits=25,                     # pHash Hamming distance (of 256 bits, ~10%) for near-duplicates
    dup_cos=0.97,                         # deep-feature cosine threshold for near-duplicates
    dup_max_group=30,                     # raise the cosine threshold until no cluster is larger
    # Data cleaning (decided before any training; reported in data_quality.csv, audit.json and REPORT.md).
    drop_label_conflicts=True,            # drop duplicate clusters whose copies carry DIFFERENT labels
    dedup_keep_one=True,                  # keep ONE image per near-duplicate cluster (augmented / re-saved copies)
    # Ungradable images (Part 4): field of view almost black / white-out, or blurred (robust z-score of the log
    # Laplacian variance vs. all non-cataract images; cataract is exempt: lens opacity blurs the fundus).
    drop_ungradable=True, min_brightness=20, max_brightness=235, blur_z_global=-3.0, blur_z_group=-3.5,
    conflict_cos=0.99,                    # ...a conflict needs an MD5, pHash or cosine >= 0.99 link
    drop_non_fundus=True,                 # drop images that are not fundus photographs (e.g. external-eye photos)
    non_fundus_dark=0.2,                  # fundus = black corners around the round field of view
    n_boot=1000,
    xai_n=160,
    pretrained=True,
    # Optional external test sets in folder format (<root>/<class folder>/*.jpg), e.g.
    # [dict(name="MyHospital", path="/kaggle/input/my-set/images")]. Overlap with ODIR-5K is removed automatically.
    external=[],
    pipeline_version="2026-11-10-v3",     # bumped when training/evaluation code changes: old runs are not reused
)
if PRESET == "lite":
    CFG.update(n_folds=3, size_override={**CFG["size_override"], "dinov2_vitb14": 224}, time_budget_h=2.2)
if PRESET == "full":
    CFG.update(epochs=14, patience=4, time_budget_h=6.2)
if os.environ.get("EYEV3_CFG"):           # used only by the local smoke test
    CFG.update(json.loads(os.environ["EYEV3_CFG"]))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NGPU = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
USE_DP = bool(CFG["multi_gpu"]) and NGPU > 1      # training + inference batches split over all GPUs
CL = torch.channels_last if DEVICE.type == "cuda" else torch.contiguous_format
NW = CFG["num_workers"]
K = CFG["n_folds"]
SEED = CFG["seed"]


def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


seed_everything(SEED)
torch.backends.cudnn.benchmark = True

WORK = Path(CFG["work_dir"])
RUNS, FIGS, TABS, WTS, RUNS_SS = (WORK / d for d in ("runs", "figures", "tables", "weights", "runs_ss"))
for d in (WORK, RUNS, FIGS, TABS, WTS, RUNS_SS):
    d.mkdir(parents=True, exist_ok=True)


def pick_cache_dir():
    c = Path(CFG["cache_dir"])
    try:
        c.mkdir(parents=True, exist_ok=True)
        free_gb = shutil.disk_usage(c).free / 1e9
    except OSError:
        free_gb = 0
    if free_gb < 8:
        c = WORK / "cache"
        c.mkdir(exist_ok=True)
    return c


CACHE = pick_cache_dir()

# Settings fingerprint: finished runs are reused only by a notebook with exactly the same settings.
TRAIN_KEYS = ("img_size", "disc_size", "size_override", "lr_override", "bs_override", "layer_decay", "drop_path",
              "ema", "ema_decay", "disc_frac", "clahe", "circle_mask", "fov_keep", "pipeline_version", "harmonise_side", "harmonise_jpeg_q", "n_folds", "inner_splits", "seed", "epochs", "patience",
              "min_epochs", "batch_size", "lr", "head_lr", "weight_decay", "warmup_epochs", "label_smoothing",
              "mixup_alpha", "mixup_prob", "grad_clip", "tta", "dup_hash_bits", "dup_cos", "dup_max_group",
              "drop_label_conflicts", "conflict_cos", "drop_non_fundus", "non_fundus_dark", "pretrained", "external",
              "classes", "imbalance", "source_robust_aug", "source_shift_test", "source_shift_models",
              "source_shift_normal_frac", "audit_all_folders", "dedup_keep_one",
              "drop_ungradable", "min_brightness", "max_brightness", "blur_z_global", "blur_z_group", "global_models",
              "disc_models", "sampler_power", "swap_rb", "remove_white_pad", "drop_multi_disease",
              "single_diagnosis_classes", "drop_uncertain_labels", "dr_min_grade", "naive_split_test",
              "naive_split_models", "naive_test_frac")
FINGERPRINT = hashlib.md5(json.dumps({k: CFG[k] for k in TRAIN_KEYS}, sort_keys=True, default=str)
                          .encode()).hexdigest()[:12]
_fp = WORK / "fingerprint.txt"
if _fp.exists() and _fp.read_text().strip() != FINGERPRINT:
    _stale = Path("/tmp") / f"eyev3_stale_{_fp.read_text().strip()}"
    _stale.mkdir(parents=True, exist_ok=True)
    for _d in (RUNS, WTS, RUNS_SS):
        for _f in _d.iterdir():
            shutil.move(str(_f), str(_stale / (_d.name + "_" + _f.name)))
    if (WORK / "splits.csv").exists():
        shutil.move(str(WORK / "splits.csv"), str(_stale / "splits.csv"))
    print(f"Settings changed in this session: earlier runs moved to {_stale}; training starts fresh.")
_fp.write_text(FINGERPRINT)

# Resume: copy finished runs / splits / weights from a previous version of this notebook (same settings only).
if IS_KAGGLE:
    n_copied = 0
    for _prev in sorted({Path(p).parent for p in glob.glob(f"/kaggle/input/**/{WORK.name}/runs", recursive=True)}):
        _pfp = _prev / "fingerprint.txt"
        if not _pfp.exists() or _pfp.read_text().strip() != FINGERPRINT:
            print(f"Not resuming from {_prev}: it used different settings (or an older notebook version).")
            continue
        for _src, _dest in ((_prev / "runs", RUNS), (_prev / "runs_ss", RUNS_SS), (_prev / "weights", WTS),
                            (_prev, WORK)):
            for _f in _src.glob("*.npz" if _dest in (RUNS, RUNS_SS) else "*.pt" if _dest is WTS else "splits.csv"):
                if not (_dest / _f.name).exists():
                    shutil.copy(_f, _dest / _f.name); n_copied += 1
    if n_copied:
        print(f"Resumed {n_copied} files from previous outputs.")

print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE} | "
      f"{torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'} | GPUs: {NGPU}"
      f"{' (DataParallel on)' if USE_DP else ''}")
print(f"work={WORK}  cache={CACHE}  preset={PRESET}  img={CFG['img_size']}  folds={K}  settings id={FINGERPRINT}")
print("classes:", " ".join(CFG["classes"]), "| referable DR: grade >=", CFG["dr_min_grade"], "| balancing: sampler",
      CFG["sampler_power"], "+ loss", CFG["imbalance"], "| decision prior power:", CFG["decision_prior_power"])
print("models:", ", ".join(CFG["global_models"]), "| disc:", ", ".join(CFG["disc_models"]) or "-",
      "| external:", ", ".join(e["name"] for e in CFG["external"]) or "none")
if DEVICE.type != "cuda":
    print("WARNING: no GPU found. Settings -> Accelerator -> GPU T4, otherwise training takes days.")
step("Part 2 done: settings loaded.")

# ---- figure style: IEEE column widths, validated colour slots, recessive axes ----
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"     # categorical slots 1-3 (all-pairs CVD-safe)
BLUES = LinearSegmentedColormap.from_list(
    "seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
COL_W, PAGE_W = 3.5, 7.16
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 600, "savefig.bbox": "tight", "font.size": 8,
    "axes.titlesize": 8.5, "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "legend.frameon": False, "axes.edgecolor": AXIS, "axes.labelcolor": INK2,
    "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.5, "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 1.6, "pdf.fonttype": 42, "ps.fonttype": 42,
    "figure.facecolor": "white", "axes.facecolor": "white"})


def save_fig(fig, name):
    fig.savefig(FIGS / f"{name}.pdf")
    fig.savefig(FIGS / f"{name}.png", dpi=600)
    plt.show()
    plt.close(fig)


def to_latex(df, name, caption, label, col_fmt=None):
    """Minimal booktabs table writer (no jinja2 dependency)."""
    df.to_csv(TABS / f"{name}.csv", index=False)
    cols = list(df.columns)
    col_fmt = col_fmt or ("l" + "c" * (len(cols) - 1))
    esc = lambda s: "" if (not isinstance(s, str) and pd.isna(s)) else \
        str(s).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")
    lines = [r"\begin{table}[!t]", r"\centering", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\resizebox{\columnwidth}{!}{%", rf"\begin{{tabular}}{{{col_fmt}}}", r"\toprule",
             " & ".join(rf"\textbf{{{esc(c)}}}" for c in cols) + r" \\", r"\midrule"]
    for _, row in df.iterrows():
        lines.append(" & ".join(esc(v) for v in row.values) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}%", "}", r"\end{table}"]
    (TABS / f"{name}.tex").write_text("\n".join(lines), encoding="utf-8")

## Part 03: Read the official ODIR-5K table, preprocess every image once, build the cache (`part03_data_cache.py`)

Reads `full_df.csv` (one row per eye: label, patient id, the eye's diagnostic keywords) and the 512 px `preprocessed_images`,
checks the colour-channel order, and preprocesses every image once: white letterbox corners -> black, round field-of-view crop,
CLAHE, optic-disc ROI, plus two image-quality measures (brightness, sharpness) for the ungradable filter of Part 4.

In [ ]:
# ==============================================================================
# PART 3/16: Find the dataset, preprocess every image once, build the cache
# Kaggle: paste into a NEW code cell directly below Part 2. Run the parts in order.
# ==============================================================================
ALL_SHORT = dict(A="AMD", C="Cataract", D="DR", G="Glaucoma", H="Hypertensive", M="Myopia", N="Normal", O="Other")
CLASSES = list(CFG["classes"])                       # folder letters, e.g. ["A", "C", "D", "G", "H", "M", "N"]
SHORT = [ALL_SHORT[c] for c in CLASSES]              # names for tables
ABBR = CLASSES                                       # one-letter tick labels for crowded figures
NC = len(CLASSES)
IDX = {c: i for i, c in enumerate(CLASSES)}
assert "N" in IDX and "G" in IDX, "Normal (N) and glaucoma (G) must be kept"
NORMAL, GLAUCOMA = IDX["N"], IDX["G"]
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def canon_class(name, any_class=False):
    """Folder name -> class letter (None for unknown folders; also None for non-task classes unless any_class)."""
    k = re.sub(r"[^a-z]", "", str(name).lower())
    c = {"a": "A", "amd": "A", "armd": "A", "agerelatedmaculardegeneration": "A", "agedegeneration": "A",
         "diabetes": "D", "o": "O", "others": "O", "other": "O", "c": "C", "cataract": "C",
         "d": "D", "dr": "D", "diabeticretinopathy": "D", "g": "G", "glaucoma": "G", "h": "H", "hypertension": "H",
         "hypertensive": "H", "hypertensiveretinopathy": "H", "m": "M", "myopia": "M", "pathologicalmyopia": "M",
         "n": "N", "normal": "N", "healthy": "N"}.get(k)
    return c if (c in IDX or (any_class and c is not None)) else None


def source_of(cls, fname):
    """Source dataset of an image, from its file-name pattern (manan1717/ocular-disease-dataset).
    ODIR-2019: <id>_left/right.jpg | Mendeley eye-disease set: DR#/Glaucoma#/Myopia#.jpg |
    Kaggle eye-diseases-classification: _<a>_<b>.jpg and cataract_###.png | ARMD curated set: *_ARMD.png, aria_* |
    hypertension set: hexadecimal .png names. '*_ORID19_*_ARMD.png' are ODIR-2019 images re-saved by the ARMD set
    (the data audit found several of them duplicated as ODIR DR/glaucoma files)."""
    if re.fullmatch(r"\d+_(left|right)\.(jpg|jpeg|png)", fname, re.I): return "ODIR"
    if re.fullmatch(r"(DR|Glaucoma|Myopia)\d+\.jpe?g", fname, re.I): return "Mendeley"
    if re.fullmatch(r"_\d+_\d+\.jpe?g", fname): return "EyeDiseaseKaggle"
    if fname.lower().startswith("cataract_"): return "CataractPNG"
    if "_ORID" in fname: return "ODIR-copy (ARMD set)"
    if "_ARMD" in fname or "_aria_" in fname: return "ARMD (RFMiD/JSIEC/ARIA)"
    if cls == "H" and fname.lower().endswith(".png"): return "HRD"
    return "other"


def camera_of(src):
    """Camera source used for within-source evaluation and the source-shift test (ODIR files + ODIR copies)."""
    return "ODIR" if src.startswith("ODIR") else src


def find_odir():
    """Official ODIR-5K on Kaggle (andrewmvd/ocular-disease-recognition-odir5k): full_df.csv (one row per eye image: single
    label, patient id, the eye's diagnostic keywords) next to preprocessed_images/ (512 x 512 px)."""
    if CFG["data_root"]:
        r_ = Path(CFG["data_root"])
        return r_ / "full_df.csv", r_ / "preprocessed_images"
    for c_ in sorted(glob.glob("/kaggle/input/**/full_df.csv", recursive=True)):
        d_ = Path(c_).parent / "preprocessed_images"
        if d_.is_dir():
            return Path(c_), d_
    raise FileNotFoundError("Dataset not found: add 'andrewmvd/ocular-disease-recognition-odir5k' as input or set "
                            "CFG['data_root'] to the folder that holds full_df.csv and preprocessed_images/.")


def load_odir(csv_p, img_d):
    """One row per eye image: path, label letter, patient id (both eyes share it), the eye's own keywords."""
    ann = pd.read_csv(csv_p).drop_duplicates("filename")
    lab = ann["labels"].astype(str).str.extract(r"([A-Z])")[0].values
    eye = np.where(ann.filename.str.contains("left"), "left", "right")
    kw = np.where(eye == "left", ann["Left-Diagnostic Keywords"], ann["Right-Diagnostic Keywords"]).astype(str)
    df_ = pd.DataFrame({"path": [str(Path(img_d) / f) for f in ann.filename], "cls": lab,
                        "label": [IDX.get(c_, -1) for c_ in lab], "orig_split": "all", "file": ann.filename.values,
                        "source": "ODIR", "patient": ["odir_" + str(i) for i in ann.ID], "eye": eye,
                        "eye_key": ann.filename.values, "kw": kw})
    df_ = df_[[os.path.exists(p) for p in df_.path]]
    if not CFG["audit_all_folders"]:
        df_ = df_[df_.label >= 0]
    return df_.sort_values("path").reset_index(drop=True)


def scan_classes(d, split, any_class=False):
    rows = []
    for cdir in sorted(p for p in Path(d).iterdir() if p.is_dir()):
        cls = canon_class(cdir.name, any_class)
        if cls is None:
            continue
        for f in sorted(cdir.rglob("*")):
            if f.suffix.lower() in IMG_EXT:
                m = re.fullmatch(r"(\d+)_(left|right)\.\w+", f.name, re.I)
                rows.append(dict(path=str(f), cls=cls, label=IDX.get(cls, -1), orig_split=split, file=f.name,
                                 source=source_of(cls, f.name), patient=f"odir_{m.group(1)}" if m else ""))
    return rows


def scan_dataset(root, any_class=False):
    root = Path(root)
    subdirs = [p for p in root.iterdir() if p.is_dir()]
    if any(canon_class(p.name) for p in subdirs):
        return pd.DataFrame(scan_classes(root, "all", any_class))
    rows = []
    for sd in sorted(subdirs):
        split = {"valid": "val", "validation": "val"}.get(sd.name.lower(), sd.name.lower())
        rows += scan_classes(sd, split, any_class)
    return pd.DataFrame(rows)


def fundus_square(bgr, thr=10):
    """Crop to the field of view (largest bright contour) and zero-pad to a square.
    Also returns the dark share of the four corners of the cropped box: a fundus photograph has a round
    field of view, so these corners are black (~1.0); an external-eye photo fills them (~0.0)."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    _, th = cv2.threshold(gray, thr, 255, cv2.THRESH_BINARY)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        x, y, w, h = cv2.boundingRect(max(cnts, key=cv2.contourArea))
        if w > 0.2 * bgr.shape[1] and h > 0.2 * bgr.shape[0]:
            bgr, gray = bgr[y:y + h, x:x + w], gray[y:y + h, x:x + w]
    h, w = bgr.shape[:2]
    c = max(2, int(0.08 * min(h, w)))
    corners = np.concatenate([gray[:c, :c].ravel(), gray[:c, -c:].ravel(), gray[-c:, :c].ravel(), gray[-c:, -c:].ravel()])
    dark = float((corners < 25).mean())
    s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    sq = cv2.copyMakeBorder(bgr, top, s - h - top, left, s - w - left, cv2.BORDER_CONSTANT, value=0)
    if CFG.get("circle_mask", True):
        # Same round field of view for every source: removes camera-specific borders (flat top/bottom edges,
        # notches, oval stretched FOVs) that identify the source dataset but carry no disease information.
        # The image is then cropped to that circle, so the field of view fills the frame equally for every source.
        # Diameter = fov_keep x the TRUE FOV diameter (the long side of the box: Mendeley's FOV is cut at top and
        # bottom), so the retina has the same scale for every source; never wider than the short side, so no flat
        # edge is left inside the circle.
        r = max(8, int(min(CFG.get("fov_keep", 0.84) * max(h, w), 0.98 * min(h, w)) / 2))
        mask = np.zeros((s, s), np.uint8)
        cv2.circle(mask, (s // 2, s // 2), r, 255, -1)
        sq = cv2.bitwise_and(sq, sq, mask=mask)[s // 2 - r:s // 2 + r, s // 2 - r:s // 2 + r]
    return sq, dark


SWAP_RB = False                                   # set in this part (detect_swap) before any image is processed


def read_bgr(path):
    """Read an image as BGR (fixes swapped red/blue channels if detect_swap found them)."""
    bgr = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_COLOR)
    if bgr is not None and SWAP_RB:
        bgr = np.ascontiguousarray(bgr[..., ::-1])
    return bgr


def detect_swap(paths, n=300):
    """Fundus photographs are red-dominant. If the first channel cv2 returns (blue) is brighter than the third (red) in
    most images, the files store RGB in BGR order (true for the 224 px Kaggle copy tanjemahamed/odir5k-classification)."""
    rng_ = np.random.default_rng(0)
    votes = []
    for i in rng_.choice(len(paths), min(n, len(paths)), replace=False):
        im = cv2.imdecode(np.fromfile(str(paths[i]), np.uint8), cv2.IMREAD_COLOR)
        if im is None:
            continue
        m = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) > 15
        if m.sum() >= 100:
            votes.append(float(im[..., 0][m].mean()) > float(im[..., 2][m].mean()))
    return (bool(np.mean(votes) > 0.7) if votes else False), (float(np.mean(votes)) if votes else float("nan"))


def remove_white_pad(bgr):
    """Some ODIR-5K images were letterboxed with WHITE corners. Pure-white regions connected to the image border are set
    to black, so that the FOV crop works and white corners cannot become a class cue (data analysis: 5 of 100 sampled
    cataract images vs. 2 of 100 normal images). Returns (image, share of pixels changed)."""
    white = (bgr.min(2) > 235).astype(np.uint8)
    if not white.any():
        return bgr, 0.0
    n, lab = cv2.connectedComponents(white, connectivity=4)
    border = np.unique(np.r_[lab[0], lab[-1], lab[:, 0], lab[:, -1]])
    border = border[border > 0]
    if not len(border):
        return bgr, 0.0
    m = np.isin(lab, border)
    frac = float(m.mean())
    if frac < 0.002:
        return bgr, frac
    out = bgr.copy()
    out[m] = 0
    return out, frac


def lab_clahe(bgr, clip=2.0, tile=8):
    l, a, b = cv2.split(cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB))
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)


def locate_disc(sq):
    """Unsupervised optic-disc centre: brightest blob of blurred L channel inside the eroded FOV.
    Returns (cx, cy) relative to the square image, and a peak z-score as a confidence proxy."""
    s = 256
    im = cv2.resize(sq, (s, s), interpolation=cv2.INTER_AREA)
    L = cv2.cvtColor(im, cv2.COLOR_BGR2LAB)[..., 0].astype(np.float32)
    mask = (cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) > 15).astype(np.uint8)
    mask = cv2.erode(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))) > 0
    if mask.sum() < 100:
        return 0.5, 0.5, 0.0
    blur = cv2.GaussianBlur(L, (0, 0), sigmaX=s * 0.03)
    score = np.where(mask, blur, -1.0)
    y, x = np.unravel_index(int(np.argmax(score)), score.shape)
    vals = blur[mask]
    return x / s, y / s, float((blur[y, x] - vals.mean()) / (vals.std() + 1e-6))


def crop_disc(sq, cx, cy, frac, out):
    s = sq.shape[0]
    half = max(8, int(frac * s / 2))
    pad = cv2.copyMakeBorder(sq, half, half, half, half, cv2.BORDER_CONSTANT, value=0)
    x0, y0 = int(round(cx * s)), int(round(cy * s))
    crop = pad[y0:y0 + 2 * half, x0:x0 + 2 * half]
    interp = cv2.INTER_AREA if crop.shape[0] > out else cv2.INTER_CUBIC
    return cv2.resize(crop, (out, out), interpolation=interp)


_POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


HASH_WORDS = 4                                   # 256-bit hash = 4 x uint64


def _dct_hash(g64):
    d = cv2.dct(g64)[:16, :16].flatten()
    bits = d > np.median(d[1:])
    return np.packbits(bits).view(">u8").astype(np.uint64)


def phash_dihedral(gray):
    """256-bit DCT pHash of the high-pass, centre-cropped fundus, for all 8 rotations/flips.
    High-pass + centre crop removes what every fundus shares (circular FOV, vignetting) and keeps the
    eye-specific vessel/disc layout, so unrelated eyes do not collide while flip/rot90/brightness copies do."""
    f = gray.astype(np.float32)
    s = f.shape[0]
    hp = f - cv2.GaussianBlur(f, (0, 0), s / 30)
    c = int(s * 0.18)                            # square inscribed in the circular field of view
    g = cv2.resize(hp[c:s - c, c:s - c], (64, 64), interpolation=cv2.INTER_AREA)
    out = []
    for k in range(4):
        r = np.rot90(g, k)
        out.append(_dct_hash(np.ascontiguousarray(r)))
        out.append(_dct_hash(np.ascontiguousarray(r[:, ::-1])))
    return np.stack(out)                         # (8, HASH_WORDS)


def popcount64(x):
    x = np.ascontiguousarray(x, dtype=np.uint64)
    return _POP8[x.view(np.uint8)].reshape(*x.shape, 8).sum(-1)


def harmonise(bgr):
    """Bring every image to the dataset's lowest common quality: longest side <= 512 px, then JPEG (q=75).
    Several classes reached this dataset only as 512 x 512 copies (AMD, cataract, part of hypertensive), the rest
    as full-resolution files. Without this step that processing trace stays detectable after resizing to 224 px
    (data audit: 91% vs. 53% = chance with this step), and the model could use it as a class cue."""
    side, q = CFG["harmonise_side"], CFG["harmonise_jpeg_q"]
    if not side:
        return bgr
    h, w = bgr.shape[:2]
    f = side / max(h, w)
    if f < 1:
        bgr = cv2.resize(bgr, (max(1, round(w * f)), max(1, round(h * f))), interpolation=cv2.INTER_AREA)
    if q:
        bgr = cv2.imdecode(cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, int(q)])[1], cv2.IMREAD_COLOR)
    return bgr


def process_image(path):
    S, DS = CFG["img_size"], CFG["disc_size"]
    raw = Path(path).read_bytes()
    bgr = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        return None
    if SWAP_RB:
        bgr = np.ascontiguousarray(bgr[..., ::-1])
    shape0 = bgr.shape[:2]                           # original size (reported; used only by the shortcut baseline)
    wpad = 0.0
    if CFG["remove_white_pad"]:
        bgr, wpad = remove_white_pad(bgr)
    bgr = harmonise(bgr)
    sq, dark = fundus_square(bgr)
    cx, cy, conf = locate_disc(sq)
    g = cv2.resize(sq, (S, S), interpolation=cv2.INTER_AREA if sq.shape[0] > S else cv2.INTER_CUBIC)
    d = crop_disc(sq, cx, cy, CFG["disc_frac"], DS)
    h = phash_dihedral(cv2.cvtColor(g, cv2.COLOR_BGR2GRAY))
    # Image quality on the harmonised, cropped image BEFORE CLAHE (ungradable filter, Part 4): mean brightness and
    # Laplacian variance of the green channel inside the eroded field of view (the FOV rim is not counted as detail).
    gq = cv2.cvtColor(g, cv2.COLOR_BGR2GRAY)
    fov = cv2.erode((gq > 10).astype(np.uint8), np.ones((max(3, S // 25),) * 2, np.uint8)) > 0
    bright = float(gq[fov].mean()) if fov.any() else 0.0
    sharp = float(cv2.Laplacian(g[..., 1], cv2.CV_64F)[fov].var()) if fov.any() else 0.0
    if CFG["clahe"]:
        g, d = lab_clahe(g), lab_clahe(d)
    return (g[..., ::-1], d[..., ::-1], (cx, cy, conf), h, hashlib.md5(raw).hexdigest(), shape0, dark, sq.shape[0],
            bright, sharp, wpad)


def build_cache(paths, tag):
    S, DS = CFG["img_size"], CFG["disc_size"]
    key = (f"v9_{tag}_s{S}_d{DS}_c{int(CFG['clahe'])}_m{int(CFG['circle_mask'])}k{CFG.get('fov_keep', 0.84)}"
           f"_h{CFG['harmonise_side']}q{CFG['harmonise_jpeg_q']}_f{CFG['disc_frac']}_rb{int(SWAP_RB)}"
           f"_w{int(CFG['remove_white_pad'])}")
    gp, dp, mp, done = (CACHE / f"{key}_g.npy", CACHE / f"{key}_d.npy",
                        CACHE / f"{key}_meta.npz", CACHE / f"{key}.done")
    if done.exists():
        step(f"Cache found for '{tag}': reusing preprocessed images (no work needed).")
        z = np.load(mp, allow_pickle=False)
        return gp, dp, {k: z[k] for k in z.files}
    n = len(paths)
    step(f"Preprocessing {n} images for '{tag}' (crop, pad, CLAHE, disc ROI). Runs once per session.")
    G = np.lib.format.open_memmap(gp, mode="w+", dtype=np.uint8, shape=(n, S, S, 3))
    D = np.lib.format.open_memmap(dp, mode="w+", dtype=np.uint8, shape=(n, DS, DS, 3))
    meta = dict(disc=np.zeros((n, 3), np.float32), hashes=np.zeros((n, 8, HASH_WORDS), np.uint64),
                md5=np.array([""] * n, dtype="U32"), shapes=np.zeros((n, 2), np.int32), ok=np.zeros(n, bool),
                corner_dark=np.zeros(n, np.float32), crop_side=np.zeros(n, np.int32),
                bright=np.zeros(n, np.float32), sharp=np.zeros(n, np.float32), white_pad=np.zeros(n, np.float32))

    def work(i):
        r = process_image(paths[i])
        if r is None:
            return
        G[i], D[i] = r[0], r[1]
        meta["disc"][i], meta["hashes"][i], meta["md5"][i], meta["shapes"][i] = r[2], r[3], r[4], r[5]
        meta["corner_dark"][i], meta["crop_side"][i] = r[6], r[7]
        meta["bright"][i], meta["sharp"][i], meta["white_pad"][i] = r[8], r[9], r[10]
        meta["ok"][i] = True

    t = time.time()
    with ThreadPoolExecutor(max_workers=max(2, (os.cpu_count() or 2) * 2)) as ex:
        list(pbar(ex.map(work, range(n)), total=n, desc=f"Preprocess {tag}", unit="img"))
    G.flush(); D.flush(); del G, D
    np.savez(mp, **meta)
    done.touch()
    print(f"cached {tag}: {n} images in {time.time() - t:.0f}s ({(~meta['ok']).sum()} unreadable)")
    return gp, dp, meta


step("Part 3: reading the official ODIR-5K table ...")
ODIR_CSV, DATA_ROOT = find_odir()
DF = load_odir(ODIR_CSV, DATA_ROOT)               # every ODIR label (A/H/M/O: audit only)
print("table:", ODIR_CSV, "| images:", DATA_ROOT)
print("Images per ODIR label (one row per eye):")
print(pd.crosstab(DF.source, DF.cls, margins=True))
print(f"Patients: {DF.patient.nunique()} ({int((DF.groupby('patient').size() > 1).sum())} with both eyes in the table)")
if CFG["swap_rb"] == "auto":
    SWAP_RB, _vote = detect_swap(DF.path.tolist())
    print(f"Colour channels: blue > red in {_vote * 100:.0f}% of 300 sampled images -> "
          f"{'SWAPPED in the files, fixed while reading' if SWAP_RB else 'normal order'}.")
else:
    SWAP_RB = bool(CFG["swap_rb"])
_tag = "internal_" + hashlib.md5("\n".join(DF.path).encode()).hexdigest()[:8]   # new class list -> new cache
GPATH, DPATH, META = build_cache(DF.path.tolist(), _tag)
N = len(DF)
OK = META["ok"]
Y = DF.label.values.astype(np.int64)                              # -1 = folder outside the task (audit only)
LET = DF.cls.values                                               # class folder letter of every image
SRC = DF.source.values
CAM = np.array([camera_of(s) for s in SRC], dtype=object)       # camera source (ODIR copies count as ODIR)
FBYTES = np.array([os.path.getsize(p) for p in DF.path], np.int64)    # file size (used by the shortcut probe)
VALID = np.where(OK)[0]
HAS_ANN = True                                    # patient id, eye and keywords come from the official table
KW = DF.kw.astype(str).str.lower().values
step(f"Part 3 done: {len(VALID)} usable images ({int((Y[VALID] >= 0).sum())} in the task classes), "
     f"{NC} classes ({', '.join(SHORT)}).")

## Part 04: Duplicate + patient audit, data cleaning (fig_duplicate_audit, fig_removed_ungradable) (`part04_duplicate_audit.py`)

**Data audit and cleaning, decided before any split or training.** Near-duplicate clusters (MD5, pHash, deep features), label
conflicts (the same eye or a near-identical image filed under two labels, incl. A/H/M/O), non-fundus check, ungradable images,
one image per duplicate cluster, **multi-disease eyes** (keywords name a second ODIR disease), **uncertain ("suspected")
labels** and **DR below the referable grade**. Prints the per-class image-quality table and the class viability check.
Check the printed tables and `fig_removed_ungradable` before committing.

In [ ]:
# ==============================================================================
# PART 4/16: Duplicate + patient + source audit, data cleaning (fig_duplicate_audit, fig_sources)
# Kaggle: paste into a NEW code cell directly below Part 3. Run the parts in order.
# ==============================================================================
def hash_pairs(HA, HB=None, thr=25, chunk=32):
    """Pairs (i, j, dist) with min dihedral Hamming distance <= thr. HB=None -> within HA (i < j)."""
    same = HB is None
    HB = HA if same else HB
    nbits = 64 * HA.shape[-1]
    out, nn_d = [], np.full(len(HA), nbits, np.int64)
    for s in pbar(range(0, len(HA), chunk), desc="pHash compare", unit="chunk"):
        a = HA[s:s + chunk, 0][:, None, None, :]                      # (c, 1, 1, W)
        d = popcount64(np.bitwise_xor(a, HB[None])).sum(-1).min(-1).astype(np.int64)   # (c, Nb)
        if same:
            r = np.arange(d.shape[0])
            d[r, s + r] = nbits
        nn_d[s:s + d.shape[0]] = d.min(1)
        ii, jj = np.nonzero(d <= thr)
        dd = d[ii, jj]
        ii = ii + s
        if same:
            keep = jj > ii
            ii, jj, dd = ii[keep], jj[keep], dd[keep]
        out.append(np.stack([ii, jj, dd], 1))
    return (np.concatenate(out) if out else np.zeros((0, 3), np.int64)), nn_d


@torch.no_grad()
def deep_embed(npy_path, idx, bs=64):
    net = models.resnet50(weights="DEFAULT" if CFG["pretrained"] else None)
    net.fc = nn.Identity()
    net = net.to(DEVICE).eval()
    arr = np.load(npy_path, mmap_mode="r")
    mean = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)
    out = []
    for s in pbar(range(0, len(idx), bs), desc="Deep embeddings", unit="batch"):
        x = torch.from_numpy(np.stack([arr[i] for i in idx[s:s + bs]])).to(DEVICE)
        x = x.permute(0, 3, 1, 2).float().div(255)
        x = F.interpolate(x, size=(224, 224), mode="bilinear", antialias=True, align_corners=False)
        x = (x - mean) / std
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            f = net(x).float() + net(torch.flip(x, dims=[3])).float()
        out.append(F.normalize(f, dim=1).cpu())
    del net
    return torch.cat(out).numpy()


@torch.no_grad()
def cos_pairs(EA, EB=None, thr=0.97, chunk=1024):
    """Pairs (i, j, sim) with cosine >= thr, plus each row's nearest-neighbour similarity."""
    same = EB is None
    A = torch.from_numpy(EA).to(DEVICE)
    B = A if same else torch.from_numpy(EB).to(DEVICE)
    out, nn_s = [], np.zeros(len(EA), np.float32)
    for s in pbar(range(0, len(A), chunk), desc="Cosine similarity", unit="chunk"):
        S = A[s:s + chunk] @ B.T
        if same:
            r = torch.arange(S.shape[0], device=DEVICE)
            S[r, s + r] = -1.0
        nn_s[s:s + S.shape[0]] = S.max(1).values.cpu().numpy()
        ii, jj = torch.nonzero(S >= thr, as_tuple=True)
        vv = S[ii, jj]
        ii = ii + s
        if same:
            keep = jj > ii
            ii, jj, vv = ii[keep], jj[keep], vv[keep]
        out.append(np.stack([ii.cpu().numpy(), jj.cpu().numpy(), vv.float().cpu().numpy()], 1))
    return (np.concatenate(out) if out else np.zeros((0, 3))), nn_s


def components(n, pair_list):
    p = [q[:, :2].astype(np.int64) for q in pair_list if len(q)]
    if not p:
        return np.arange(n)
    p = np.concatenate(p)
    A = coo_matrix((np.ones(len(p)), (p[:, 0], p[:, 1])), shape=(n, n))
    return connected_components(A, directed=False)[1]




VALID = np.where(OK)[0]                  # all readable images (data cleaning at the end of this part)
H_valid = META["hashes"][VALID]
md5_valid = META["md5"][VALID]
_, md5_inv = np.unique(md5_valid, return_inverse=True)
_o = np.argsort(md5_inv, kind="stable")
_same = md5_inv[_o][1:] == md5_inv[_o][:-1]                   # consecutive images with identical bytes
md5_pairs = [np.stack([_o[:-1][_same], _o[1:][_same], np.zeros(int(_same.sum()), np.int64)], 1)]
step(f"Part 4 - audit 1/3: comparing perceptual hashes of {len(VALID)} images (CPU, ~2-5 min) ...")
hp_all, nn_hash = hash_pairs(H_valid, thr=CFG["dup_hash_bits"])
step("Part 4 - audit 2/3: deep-feature embeddings with ResNet-50 (GPU) ...")
E_valid = deep_embed(GPATH, VALID)
cp_all, nn_cos = cos_pairs(E_valid, thr=CFG["dup_cos"])
step("Part 4 - audit 3/3: clustering duplicates, patients and sources ...")

# Guard against chaining: tighten the hash threshold until no cluster exceeds dup_max_group.
hash_thr = CFG["dup_hash_bits"]
while True:
    hp = hp_all[hp_all[:, 2] <= hash_thr]
    max_size = np.bincount(components(len(VALID), md5_pairs + [hp])).max()
    if max_size <= CFG["dup_max_group"] or hash_thr <= 0:
        break
    hash_thr = max(0, hash_thr - 3)
if max_size > CFG["dup_max_group"]:
    print(f"WARNING: hash clusters too large ({max_size}); using exact (MD5) duplicates only for hashing.")
    hash_thr, hp = -1, hp_all[:0]

cos_thr = CFG["dup_cos"]
while True:
    cp = cp_all[cp_all[:, 2] >= cos_thr] if len(cp_all) else cp_all
    comp = components(len(VALID), md5_pairs + [hp, cp])
    max_size = np.bincount(comp).max()
    if max_size <= CFG["dup_max_group"] or cos_thr >= 0.995:
        break
    cos_thr = round(cos_thr + 0.005, 3)
if max_size > CFG["dup_max_group"]:
    # Deep features chain unrelated images together even at the strictest threshold: fall back to hashes.
    print(f"WARNING: deep-feature clusters too large ({max_size}); using MD5 + pHash clusters only.")
    cos_thr, cp = float("inf"), cp_all[:0]
    comp = components(len(VALID), md5_pairs + [hp])

# The same ODIR eye (identical file name <id>_<eye>.jpg) filed under two classes -> always the same cluster.
lab_v = Y[VALID]
file_v = DF.file.values[VALID]
src_v = SRC[VALID]
cam_v = CAM[VALID]
let_v = LET[VALID]                       # class folder (all 7 folders)
task_v = lab_v >= 0
ekey_v = DF.eye_key.values[VALID]       # official ODIR file name of each image (one eye = one key)
_eye = pd.Series(np.arange(len(VALID))).groupby(ekey_v).apply(list)
eye_pairs = np.array([[v[0], j] for v in _eye if len(v) > 1 for j in v[1:]], np.int64).reshape(-1, 2)
if len(eye_pairs):
    comp = components(len(VALID), md5_pairs + [hp, cp, eye_pairs])
sizes = np.bincount(comp)
in_multi = sizes[comp] > 1

# CV groups = duplicate clusters joined with ODIR patients (both eyes of a patient always share a fold).
pat_v = DF.patient.values[VALID]
_pp = pd.Series(np.arange(len(VALID)))[pat_v != ""].groupby(pat_v[pat_v != ""]).apply(list)
pat_pairs = np.array([[v[0], j] for v in _pp for j in v[1:]], np.int64).reshape(-1, 2)
comp_cv = components(len(VALID), md5_pairs + [hp, cp, eye_pairs, pat_pairs])
GROUP = np.full(N, -1, np.int64)
GROUP[VALID] = comp_cv


def leaked_fraction(train_rows, test_rows, grp=GROUP):
    tr_groups = set(grp[train_rows])
    return float(np.mean([grp[i] in tr_groups for i in test_rows])) if len(test_rows) else float("nan")


split_of = DF.orig_split.values
off_tr = VALID[split_of[VALID] == "train"]
off_te = VALID[np.isin(split_of[VALID], ["test", "val"])]
# What a naive image-level random 70/15/15 split would have done.
_tv = VALID[Y[VALID] >= 0]               # task images only
naive_tr, naive_tmp = train_test_split(_tv, test_size=0.30, stratify=Y[_tv], random_state=42)
_, naive_te = train_test_split(naive_tmp, test_size=0.50, stratify=Y[naive_tmp], random_state=42)
DUPG = np.full(N, -1, np.int64)
DUPG[VALID] = comp
PATG = np.full(N, -1, np.int64)
PATG[VALID] = components(len(VALID), [pat_pairs]) if len(pat_pairs) else np.arange(len(VALID))

# ---- Data quality: label conflicts and non-fundus images. Decided from image similarity, file names and the
# given labels only (never from model predictions), before any split or training. ----
_cp_strong = cp[cp[:, 2] >= CFG["conflict_cos"]] if len(cp) else cp
_links = [q[:, :2].astype(np.int64) for q in (md5_pairs[0], hp, _cp_strong, eye_pairs) if len(q)]
_links = np.concatenate(_links) if _links else np.zeros((0, 2), np.int64)
# Version 2: a label conflict is judged on the class FOLDER of each image (all 7 folders are scanned, Part 3), so a
# glaucoma photo that is also filed as myopia, or a DR photo that is also an AMD "ODIR copy", is found although
# A/H/M are not task classes.
_bad = _links[let_v[_links[:, 0]] != let_v[_links[:, 1]]]          # near-identical images, different class folders
conflict_groups = np.unique(comp[_bad[:, 0]])
conflict_v = np.isin(comp, conflict_groups)
conf_task_groups = np.unique(comp[conflict_v & task_v])             # conflicts that touch a task class
dark_v = META["corner_dark"][VALID] if "corner_dark" in META else np.ones(len(VALID))
nonfundus_v = dark_v < CFG["non_fundus_dark"]
combo = pd.Series([" + ".join(sorted({ALL_SHORT[let_v[j]] for j in np.where(comp == g)[0]})) for g in conf_task_groups],
                  dtype=object).value_counts()

# Image quality (gradability): objective rules on the harmonised, cropped image before CLAHE (Part 3), decided before
# any training. (1) Field of view almost black or white-out. (2) Blur: a DR / glaucoma / normal grade needs visible
# vessels and optic disc, so blur is judged against all non-cataract task images (robust z-score of the log Laplacian
# variance). Cataract is exempt from that rule (lens opacity blurs the fundus: it IS the sign), but every class still
# loses the extreme outliers of its own source x class group.
br_v = META["bright"][VALID].astype(float)
sh_v = np.log1p(META["sharp"][VALID].astype(float))
_ref = task_v & (lab_v != IDX.get("C", -99)) & ~conflict_v & ~nonfundus_v


def _robust_z(v, ref):
    if not ref.any():
        return np.zeros_like(v)
    med = np.median(v[ref])
    return (v - med) / max(1.4826 * np.median(np.abs(v[ref] - med)), 1e-6)


blur_z = _robust_z(sh_v, _ref)
blur_v = _ref & (blur_z < CFG["blur_z_global"])
for _key, _ix in pd.Series(np.arange(len(VALID)))[task_v].groupby([src_v[task_v], lab_v[task_v]]):
    _ix = _ix.values
    if len(_ix) >= 30:
        _m = np.zeros(len(VALID), bool)
        _m[_ix] = True
        blur_v[_ix[_robust_z(sh_v, _m)[_ix] < CFG["blur_z_group"]]] = True
exposure_v = (br_v < CFG["min_brightness"]) | (br_v > CFG["max_brightness"])
ungradable_v = task_v & (blur_v | exposure_v)

# ---- Version 3: keyword-based cleaning with the official ODIR-5K table. The diagnosis of every eye is a list of keywords;
# the label is one of them. An eye whose keywords ALSO name another ODIR disease (e.g. glaucoma + DR) has no single correct
# label (single-diagnosis cohort); an eye labelled only as "suspected ..." has an uncertain label; a DR eye below the
# referable grade (mild NPDR = microaneurysms only, or no grade given) is outside the referable-DR task. ----
# ODIR DR keywords: "mild nonproliferative retinopathy", "moderate non proliferative retinopathy", "severe ...", "(severe)
# proliferative diabetic retinopathy", "diabetic retinopathy" ("myopia retinopathy" and "hypertensive retinopathy" are not DR).
DIS_RX = {"A": r"macular degeneration", "C": r"cataract", "D": r"non ?proliferative retinopathy|diabetic retinopathy",
          "G": r"glaucoma", "H": r"hypertensive", "M": r"myopi"}
kw_v = KW[VALID]


def _mentions(text, c):
    return bool(re.search(DIS_RX[c], text)) if c in DIS_RX else False


def _strip_suspect(text):
    return re.sub(r"suspect\w*\s+[^,，;]+", " ", text)


has_kw_v = np.array([len(t.strip()) > 0 for t in kw_v])
multi_v = np.array([bool(has_kw_v[i]) and task_v[i] and any(_mentions(kw_v[i], c) for c in CFG["single_diagnosis_classes"]
                                                            if c != let_v[i]) for i in range(len(VALID))])
uncertain_v = np.array([bool(has_kw_v[i]) and task_v[i] and let_v[i] != "N" and _mentions(kw_v[i], let_v[i])
                        and not _mentions(_strip_suspect(kw_v[i]), let_v[i]) for i in range(len(VALID))])


def dr_grade(text):
    """ICDR grade from the ODIR keywords (highest grade named): 1 mild NPDR, 2 moderate, 3 severe, 4 proliferative,
    0 = DR without grade."""
    if "proliferative diabetic" in text:
        return 4
    return 3 if "severe non" in text else 2 if "moderate non" in text else 1 if "mild non" in text else 0


grade_v = np.array([dr_grade(kw_v[i]) if (has_kw_v[i] and let_v[i] == "D") else -1 for i in range(len(VALID))])
lowdr_v = task_v & (let_v == "D") & (grade_v < int(CFG["dr_min_grade"]))

# Exclusion reasons (the last assignment wins). Excluded images get no fold: never trained, validated or tested on.
reason = np.full(len(VALID), "", dtype=object)
reason[~task_v] = "not_in_task"
if CFG["drop_ungradable"]:
    reason[ungradable_v] = "ungradable"
if int(CFG["dr_min_grade"]) > 0:
    reason[lowdr_v] = "dr_below_referable"
if HAS_ANN and CFG["drop_uncertain_labels"]:
    reason[uncertain_v & task_v] = "uncertain_label"
if HAS_ANN and CFG["drop_multi_disease"]:
    reason[multi_v & task_v] = "multi_disease"
if CFG["drop_non_fundus"]:
    reason[nonfundus_v & task_v] = "non_fundus"
if CFG["drop_label_conflicts"]:
    reason[conflict_v & task_v] = "label_conflict"
# Near-duplicate copies inside one class (augmented or re-saved files): keep ONE image per duplicate cluster (the
# largest original file), so no eye is counted twice in a test fold.
dupcopy_v = np.zeros(len(VALID), bool)
if CFG["dedup_keep_one"]:
    _px = META["shapes"][VALID].astype(float).prod(1)
    _cand = np.where(reason == "")[0]
    _seen = set()
    for j in _cand[np.lexsort((_cand, -_px[_cand]))]:
        if comp[j] in _seen:
            dupcopy_v[j] = True
        else:
            _seen.add(comp[j])
    reason[dupcopy_v] = "duplicate_copy"
DQ = pd.DataFrame({"path": DF.path.values[VALID], "cls": let_v, "source": src_v, "camera": cam_v,
                   "patient": pat_v, "dup_cluster": comp, "cv_group": comp_cv, "label_conflict": conflict_v,
                   "corner_dark": np.round(dark_v, 3), "non_fundus": nonfundus_v, "brightness": np.round(br_v, 1),
                   "log_sharpness": np.round(sh_v, 3), "blur_z": np.round(blur_z, 2), "ungradable": ungradable_v,
                   "white_pad": np.round(META["white_pad"][VALID], 4), "keywords": kw_v, "multi_disease": multi_v,
                   "uncertain_label": uncertain_v, "dr_grade": grade_v, "duplicate_copy": dupcopy_v, "excluded": reason})
DQ.to_csv(WORK / "data_quality.csv", index=False)
used_v = reason == ""
DQ_TABLE = pd.DataFrame({
    "Class": SHORT,
    "Images": [int((lab_v == c).sum()) for c in range(NC)],
    "In duplicate clusters": [int((in_multi & (lab_v == c)).sum()) for c in range(NC)],
    "Label conflicts": [int(((reason == "label_conflict") & (lab_v == c)).sum()) for c in range(NC)],
    "Not fundus": [int(((reason == "non_fundus") & (lab_v == c)).sum()) for c in range(NC)],
    "Ungradable": [int(((reason == "ungradable") & (lab_v == c)).sum()) for c in range(NC)],
    "DR below referable": [int(((reason == "dr_below_referable") & (lab_v == c)).sum()) for c in range(NC)],
    "Multi-disease eye": [int(((reason == "multi_disease") & (lab_v == c)).sum()) for c in range(NC)],
    "Uncertain label": [int(((reason == "uncertain_label") & (lab_v == c)).sum()) for c in range(NC)],
    "Duplicate copies": [int(((reason == "duplicate_copy") & (lab_v == c)).sum()) for c in range(NC)],
    "Excluded": [int(((reason != "") & (lab_v == c)).sum()) for c in range(NC)],
    "Used": [int((used_v & (lab_v == c)).sum()) for c in range(NC)],
    "Unique eyes": [int(len(np.unique(comp[used_v & (lab_v == c)]))) for c in range(NC)]})
DQ_TABLE.loc[len(DQ_TABLE)] = ["Total"] + DQ_TABLE.iloc[:, 1:].sum().tolist()
to_latex(DQ_TABLE, "tab_data_quality", "Data-quality audit (all removals decided before splitting). Label conflicts: "
         "identical or near-identical copies (MD5, pHash, deep-feature or same ODIR eye) filed under different labels "
         "(including AMD, hypertensive, myopia and other); the whole duplicate cluster is removed. Not fundus: no dark "
         "corners around a round field of view. Ungradable: field of view almost black or white-out, or blurred (robust "
         "z-score of the Laplacian variance; cataract exempt from the global blur rule). DR below referable: mild NPDR or "
         "DR without a grade. Multi-disease eye: the eye's ODIR keywords also name another ODIR disease. Uncertain "
         "label: the class is given only as 'suspected'. Duplicate copies: one image is kept per near-duplicate cluster.",
         "tab:dataquality")

# Image quality per class (used images): a class that is systematically darker or blurrier could be recognised from image
# quality alone. Part 12 measures this with an image-quality-only baseline.
QUALITY = pd.DataFrame({"Class": SHORT,
                        "Median brightness": [round(float(np.median(br_v[used_v & (lab_v == c)])), 1)
                                              if (used_v & (lab_v == c)).any() else np.nan for c in range(NC)],
                        "Median log-sharpness": [round(float(np.median(sh_v[used_v & (lab_v == c)])), 3)
                                                 if (used_v & (lab_v == c)).any() else np.nan for c in range(NC)],
                        "White-padded (%)": [round(100 * float(np.mean(META["white_pad"][VALID][used_v & (lab_v == c)] > 0.002)), 1)
                                             if (used_v & (lab_v == c)).any() else np.nan for c in range(NC)]})
QUALITY.to_csv(TABS / "class_quality.csv", index=False)

# ---- Source audit: which source dataset each class comes from (after cleaning). ----
SOURCES = sorted(set(src_v[used_v]), key=lambda s: (s != "ODIR", s))
SRC_TABLE = pd.DataFrame({"Source": SOURCES, **{SHORT[c]: [int((used_v & (src_v == s) & (lab_v == c)).sum())
                                                            for s in SOURCES] for c in range(NC)}})
SRC_TABLE["Total"] = SRC_TABLE[SHORT].sum(1)
SRC_TABLE.loc[len(SRC_TABLE)] = ["Total"] + SRC_TABLE.iloc[:, 1:].sum().tolist()
to_latex(SRC_TABLE, "tab_sources", "Images used per class and source dataset (inferred from file names). "
         "Classes drawn from different sources can be separated by acquisition cues (camera, resolution, "
         "compression); see the shortcut baseline and the source-shift test.", "tab:sources")
_shp = META["shapes"][VALID]
SRC_PROPS = pd.DataFrame({"source": src_v, "w": _shp[:, 1], "h": _shp[:, 0], "kb": FBYTES[VALID] / 1024,
                          "corner_dark": dark_v}).groupby("source").agg(
    images=("w", "size"), width_median=("w", "median"), height_median=("h", "median"),
    file_kb_median=("kb", "median"), corner_dark_median=("corner_dark", "median")).round(2)
SRC_PROPS.to_csv(TABS / "source_properties.csv")
# Class viability: enough unique eyes, and part of the class comes from a source shared with other classes
# (otherwise the class can be recognised from its source alone and cannot be tested within one source).
_shared = {s for s in set(cam_v) if len(set(lab_v[used_v & (cam_v == s)])) >= 2}      # camera sources
VIABILITY = pd.DataFrame({
    "Class": SHORT,
    "Unique eyes": DQ_TABLE["Unique eyes"].values[:NC],
    "Sources": [", ".join(sorted(set(src_v[used_v & (lab_v == c)]))) for c in range(NC)],
    "In shared sources (%)": [round(100 * float(np.mean(np.isin(cam_v[used_v & (lab_v == c)], list(_shared))))
                                    if (used_v & (lab_v == c)).any() else 0, 1) for c in range(NC)]})
# Feasibility probe inside ONE camera source (ODIR): frozen ImageNet ResNet-50 features + logistic regression,
# group 5-fold. No fine-tuning, so the numbers are low; a class near zero here is recognised only via its source.
from sklearn.decomposition import PCA
_od = np.where(used_v & (cam_v == "ODIR"))[0]
probe_f1 = {}
if len(_od) >= 50 and len(np.unique(lab_v[_od])) >= 2:
    _pr = np.full(len(_od), -1)
    for tr_, te_ in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(_od, lab_v[_od], comp_cv[_od]):
        clf_ = make_pipeline(StandardScaler(), PCA(min(256, len(tr_) - 1), random_state=SEED),
                             LogisticRegression(C=0.05, max_iter=3000, class_weight="balanced"))
        _pr[te_] = clf_.fit(E_valid[_od[tr_]], lab_v[_od[tr_]]).predict(E_valid[_od[te_]])
    probe_f1 = {int(c): float(f1_score(lab_v[_od] == c, _pr == c)) for c in np.unique(lab_v[_od])}
VIABILITY["ODIR-camera probe F1"] = [round(probe_f1[c], 3) if c in probe_f1 else np.nan for c in range(NC)]
VIABILITY["OK"] = ((VIABILITY["Unique eyes"] >= CFG["min_class_eyes"]) & (VIABILITY["In shared sources (%)"] >= 20)
                   & ~(VIABILITY["ODIR-camera probe F1"] < 0.2))
VIABILITY.to_csv(TABS / "class_viability.csv", index=False)


# Harmonisation check at the TRAINING resolution. AMD (incl. its ODIR copies) and cataract (incl. the ODIR cataract
# files) exist only as 512 x 512 re-saved images; the other classes as full-resolution files. The data audit showed
# that harmonisation hides this processing trace at 224 px (53%, chance = 50%); at a higher input resolution more
# fine detail survives, so it is measured again here: the same full-resolution ODIR normal images are processed
# (a) from the original file and (b) after simulating the '512 x 512 + JPEG' chain; a frozen ResNet-50 at the
# training resolution + logistic regression tries to tell (a) from (b).
def _global_view(bgr):
    sq, _ = fundus_square(harmonise(bgr))
    S = CFG["img_size"]
    g = cv2.resize(sq, (S, S), interpolation=cv2.INTER_AREA if sq.shape[0] > S else cv2.INTER_CUBIC)
    return np.ascontiguousarray((lab_clahe(g) if CFG["clahe"] else g)[..., ::-1])


@torch.no_grad()
def _embed_native(X, bs=32):
    """ResNet-50 (ImageNet) global features at the images' own resolution (deep_embed always resizes to 224)."""
    net = models.resnet50(weights="DEFAULT" if CFG["pretrained"] else None)
    net.fc = nn.Identity()
    net = net.to(DEVICE).eval()
    mean = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)
    out = []
    for s in range(0, len(X), bs):
        x = torch.from_numpy(X[s:s + bs]).to(DEVICE).permute(0, 3, 1, 2).float().div(255)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(net((x - mean) / std).float().cpu())
    del net
    return torch.cat(out).numpy()


CHAIN_PROBE = None
_shp_all = META["shapes"]
_full = VALID[(SRC[VALID] == "ODIR") & (Y[VALID] == NORMAL) & (_shp_all[VALID].min(1) >= 1000)
              & (_shp_all[VALID].max(1) <= 1.05 * _shp_all[VALID].min(1))]          # full-resolution, square
if CFG["harmonise_side"] and len(_full) >= 60:
    from sklearn.model_selection import GroupKFold
    _ids = np.random.default_rng(SEED).choice(_full, min(300, len(_full)), replace=False)
    _XA, _XB = [], []
    for i in pbar(_ids, desc="Harmonisation check", unit="img"):
        bgr = cv2.imdecode(np.fromfile(DF.path[i], np.uint8), cv2.IMREAD_COLOR)
        _XA.append(_global_view(bgr))
        small = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
        small = cv2.imdecode(cv2.imencode(".jpg", small, [cv2.IMWRITE_JPEG_QUALITY, 75])[1], cv2.IMREAD_COLOR)
        _XB.append(_global_view(small))
    _E = _embed_native(np.stack(_XA + _XB))
    _yc = np.r_[np.zeros(len(_XA)), np.ones(len(_XB))].astype(int)
    _gc = np.r_[np.arange(len(_XA)), np.arange(len(_XB))]
    _pc = np.zeros(len(_yc), int)
    for tr_, te_ in GroupKFold(5).split(_E, _yc, _gc):
        _pc[te_] = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=3000)).fit(
            _E[tr_], _yc[tr_]).predict(_E[te_])
    CHAIN_PROBE = float((_pc == _yc).mean())
    del _XA, _XB, _E
    print(f"\nHARMONISATION CHECK at {CFG['img_size']} px: original vs. '512 x 512 + JPEG' copy of the same "
          f"{len(_ids)} ODIR images told apart with {CHAIN_PROBE * 100:.1f}% accuracy (50% = no trace; "
          f"data audit at 224 px: 53%).")
    if CHAIN_PROBE > 0.65:
        print("WARNING: the processing chain of the AMD/cataract images is visible at this resolution, so AMD and "
              "cataract (also inside ODIR) can be recognised from it. Lower CFG['img_size'] (e.g. 224 px) or report this "
              "number next to the AMD/cataract results.")

AUDIT = dict(
    n_images=int(N), n_unreadable=int((~OK).sum()), classes=CLASSES,
    exact_duplicate_pairs=int(len(md5_pairs[0])), phash_pairs=int(len(hp)), deep_pairs=int(len(cp)),
    hash_threshold_used_bits_of_256=int(hash_thr), cosine_threshold_used=cos_thr,
    n_clusters=int(len(sizes)), n_images_in_clusters_gt1=int(in_multi.sum()),
    largest_cluster=int(sizes.max()),
    odir_patients=int(len(_pp)), odir_patients_with_both_eyes=int(sum(len(v) > 1 for v in _pp)),
    n_cv_groups=int(len(np.unique(comp_cv))),
    official_eval_with_dup_in_train=leaked_fraction(off_tr, off_te) if len(off_tr) and len(off_te) else None,
    naive_70_15_15_test_with_dup_in_train=leaked_fraction(naive_tr, naive_te, DUPG),
    naive_70_15_15_test_with_same_patient_in_train=leaked_fraction(naive_tr, naive_te, PATG),
    naive_70_15_15_test_with_dup_or_patient_in_train=leaked_fraction(naive_tr, naive_te),
    nn_cos_quantiles={q: float(np.quantile(nn_cos, q)) for q in (0.5, 0.9, 0.99)},
    label_conflict_clusters=int(len(conf_task_groups)), label_conflict_images=int((conflict_v & task_v).sum()),
    label_conflict_classes={str(k_): int(v_) for k_, v_ in combo.items()},
    same_odir_eye_in_two_classes=int(sum(len(v) > 1 for v in _eye)),
    non_fundus_images=int((nonfundus_v & task_v).sum()),
    excluded_label_conflict=int((reason == "label_conflict").sum()),
    excluded_non_fundus=int((reason == "non_fundus").sum()),
    excluded_ungradable=int((reason == "ungradable").sum()),
    excluded_duplicate_copies=int((reason == "duplicate_copy").sum()),
    not_in_task_images=int((reason == "not_in_task").sum()),
    task_images_before_cleaning=int(task_v.sum()),
    n_used=int(used_v.sum()),
    imbalance_ratio_used=float(DQ_TABLE["Used"][:NC].max() / max(1, DQ_TABLE["Used"][:NC].min())),
    chain_probe_acc=CHAIN_PROBE, chain_probe_img_size=int(CFG["img_size"]),
    swap_rb=bool(SWAP_RB), patient_ids_available=bool(HAS_ANN),
    excluded_dr_below_referable=int((reason == "dr_below_referable").sum()),
    excluded_multi_disease=int((reason == "multi_disease").sum()),
    excluded_uncertain_label=int((reason == "uncertain_label").sum()),
    white_padded_task_images=int(((META["white_pad"][VALID] > 0.002) & task_v).sum()),
    dr_grades_used={str(g_): int(((grade_v == g_) & (reason == "")).sum()) for g_ in (0, 1, 2, 3, 4)},
)
(WORK / "audit.json").write_text(json.dumps(AUDIT, indent=2))
print(json.dumps(AUDIT, indent=2))
print("\nDATA-QUALITY TABLE (tab_data_quality):")
print(DQ_TABLE.to_string(index=False))
if len(combo):
    print("\nLabel-conflict clusters by class combination:\n" + combo.to_string())
print("\nIMAGE QUALITY PER CLASS (images used; class_quality.csv):")
print(QUALITY.to_string(index=False))
print("\nCLASS VIABILITY (OK = >= %d unique eyes and frozen-feature probe F1 >= 0.2):" % CFG["min_class_eyes"])
print(VIABILITY.drop(columns=["In shared sources (%)"]).to_string(index=False))
if not VIABILITY.OK.all():
    print("NOTE: classes marked False are hard to defend in a paper (too few eyes, or not separable from frozen "
          "features). See plan.md (Version 3).")

# Figure: nearest-neighbour similarity + the most similar pairs filed under different classes (visual check).
fig = plt.figure(figsize=(PAGE_W, 2.6))
gs = fig.add_gridspec(2, 5, width_ratios=[2.2, 1, 1, 1, 1])
ax = fig.add_subplot(gs[:, 0])
ax.hist(nn_cos, bins=60, color=C1, edgecolor="white", linewidth=0.3)
ax.axvline(cos_thr, color=INK2, ls="--", lw=1)
ax.text(cos_thr, ax.get_ylim()[1] * 0.92, f" threshold {cos_thr:.3f}", color=INK2, fontsize=7, ha="right")
ax.set_xlabel("Cosine similarity to nearest other image")
ax.set_ylabel("Images")
ax.set_yscale("log")
ax.set_title("(a) Nearest-neighbour similarity")
cross = [(a, b, v) for a, b, v in cp_all if let_v[int(a)] != let_v[int(b)]]
cross = sorted(cross, key=lambda t: -t[2])[:4] or sorted([tuple(t) for t in cp_all], key=lambda t: -t[2])[:4]
G_arr = np.load(GPATH, mmap_mode="r")
for c, (a, b, v) in enumerate(cross):
    for r, i in enumerate((int(a), int(b))):
        axi = fig.add_subplot(gs[r, 1 + c])
        axi.imshow(G_arr[VALID[i]])
        axi.set_xticks([]); axi.set_yticks([]); axi.grid(False)
        for sp in axi.spines.values():
            sp.set_visible(False)
        axi.set_title(f"{ALL_SHORT[let_v[i]]} ({src_v[i]})" + (f"\ncos {v:.3f}" if r == 0 else ""), fontsize=6.2)
fig.suptitle("(b) Most similar image pairs filed under different classes (top row vs bottom row)",
             x=0.66, y=1.02, fontsize=7.5, color=INK2)
save_fig(fig, "fig_duplicate_audit")

# Figure: label-conflict examples (same image, two labels) and images that are not fundus photographs.
rng_q = np.random.default_rng(SEED)
pairs_show = []
for g in rng_q.permutation(conf_task_groups)[:4]:
    mem = np.where(comp == g)[0]
    a = int(mem[0])
    b = int(next(j for j in mem if let_v[j] != let_v[a]))
    pairs_show.append((a, b))
nf_show = rng_q.permutation(np.where(nonfundus_v)[0])[:4]
n_rows = (2 if pairs_show else 0) + (1 if len(nf_show) else 0)
if n_rows:
    fig, axes = plt.subplots(n_rows, 4, figsize=(PAGE_W * 0.75, 1.6 * n_rows), squeeze=False, layout="constrained")
    for ax in axes.flat:
        ax.axis("off")
    for c_, (a, b) in enumerate(pairs_show):
        for r_, i in enumerate((a, b)):
            axes[r_, c_].imshow(G_arr[VALID[i]])
            axes[r_, c_].set_title(f"{ALL_SHORT[let_v[i]]} ({src_v[i]})", fontsize=6.2)
    for c_, i in enumerate(nf_show):
        axes[n_rows - 1, c_].imshow(G_arr[VALID[i]])
        axes[n_rows - 1, c_].set_title(f"{ALL_SHORT[let_v[i]]}: not fundus", fontsize=6.5)
    save_fig(fig, "fig_data_quality")

# Figure: images removed as ungradable (most extreme first). Check visually that they are really unusable; if good
# images appear, make blur_z_global more negative (e.g. -3.5) or set drop_ungradable=False.
_ug = np.where(reason == "ungradable")[0]
if len(_ug):
    _ug = _ug[np.argsort(np.minimum(blur_z[_ug], 0) - np.abs(br_v[_ug] - 128) / 32)][:16]
    fig, axes = plt.subplots(math.ceil(len(_ug) / 8), 8, figsize=(PAGE_W, 1.15 * math.ceil(len(_ug) / 8)),
                             squeeze=False, layout="constrained")
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, _ug):
        why = "too dark/bright" if exposure_v[i] else f"blur z {blur_z[i]:.1f}"
        ax.imshow(G_arr[VALID[i]])
        ax.set_title(f"{SHORT[lab_v[i]]} ({src_v[i]})\n{why}", fontsize=5.2)
    save_fig(fig, "fig_removed_ungradable")
    _ugt = pd.crosstab(src_v[reason == "ungradable"], np.array(SHORT, dtype=object)[lab_v[reason == "ungradable"]])
    print("\nRemoved as ungradable (source x class):\n" + _ugt.to_string())
    _share = pd.Series(reason == "ungradable")[task_v].groupby([src_v[task_v], lab_v[task_v]]).mean()
    if (_share > 0.15).any():
        print("WARNING: more than 15% of some source x class cell is ungradable - check fig_removed_ungradable.")

# Figure: what each source looks like after preprocessing (one random image per class x source cell).
_cells = [(s, c) for s in SOURCES for c in range(NC) if (used_v & (src_v == s) & (lab_v == c)).any()]
if _cells:
    nc_ = min(8, len(_cells))
    nr_ = math.ceil(len(_cells) / nc_)
    fig, axes = plt.subplots(nr_, nc_, figsize=(PAGE_W, PAGE_W / nc_ * 1.2 * nr_), squeeze=False, layout="constrained")
    for ax in axes.flat:
        ax.axis("off")
    for ax, (s, c) in zip(axes.flat, _cells):
        i = int(rng_q.choice(np.where(used_v & (src_v == s) & (lab_v == c))[0]))
        ax.imshow(G_arr[VALID[i]])
        ax.set_title(f"{SHORT[c]}\n{s}", fontsize=6)
    save_fig(fig, "fig_sources")

# Visual check (shown here, not saved): every image flagged as "not fundus", up to 24.
nf_all = np.where(nonfundus_v)[0][:24]
if len(nf_all):
    print(f"\nCHECK: {int(nonfundus_v.sum())} images flagged as NOT fundus. They should all be non-fundus photos; "
          f"if real fundus images appear here, lower CFG['non_fundus_dark'].")
    nc_ = 8
    fig, axes = plt.subplots(math.ceil(len(nf_all) / nc_), nc_, figsize=(PAGE_W, 1.05 * math.ceil(len(nf_all) / nc_)),
                             squeeze=False, layout="constrained")
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, nf_all):
        ax.imshow(G_arr[VALID[i]])
        ax.set_title(f"{ALL_SHORT[let_v[i]]} {dark_v[i]:.2f}", fontsize=5.5)
    plt.show()
    plt.close(fig)

# Apply the cleaning: excluded images get no fold, so they are never trained on or tested.
EXCLUDED = np.full(N, "", dtype=object)
EXCLUDED[VALID] = reason
CONFLICT = np.zeros(N, bool)
CONFLICT[VALID[conflict_v]] = True
VALID = VALID[reason == ""]
step(f"Part 4 done: audit saved (audit.json, data_quality.csv, figures). Task images {AUDIT['task_images_before_cleaning']}: "
     f"excluded {AUDIT['excluded_label_conflict']} label-conflict + {AUDIT['excluded_non_fundus']} non-fundus + "
     f"{AUDIT['excluded_ungradable']} ungradable + {AUDIT['excluded_dr_below_referable']} DR below referable + "
     f"{AUDIT['excluded_multi_disease']} multi-disease + {AUDIT['excluded_uncertain_label']} uncertain-label + "
     f"{AUDIT['excluded_duplicate_copies']} duplicate copies; {len(VALID)} images go to cross-validation.")

## Part 05: Stratified group K-fold splits (patients + duplicate clusters) + protocol-comparison split (`part05_cv_splits.py`)

Stratified **group** K-fold splits (`n_folds` = 5 in presets `fast`/`full`, 3 in `lite`; strata = class; groups = ODIR patients
(both eyes always in the same fold) + duplicate clusters). 1/8 of each training part is the validation set. Also a **random
image-level split** (patients ignored, as in many published papers) for the protocol comparison of Part 14.

In [ ]:
# ==============================================================================
# PART 5/16: Stratified group K-fold splits (patients + duplicate clusters) + source-shift split (splits.csv)
# Kaggle: paste into a NEW code cell directly below Part 4. Run the parts in order.
# ==============================================================================
SRC_CODE = pd.factorize(SRC)[0].astype(np.int64)


def strata(idx):
    """Stratification label = class x source dataset, so every fold (and every validation part) gets the same
    source mix inside each class (e.g. DR: ODIR vs. Mendeley, which differ a lot in difficulty). Class-source
    combinations too small to spread over the folds fall back to the class alone."""
    idx = np.asarray(idx)
    s = Y[idx] * 1000 + SRC_CODE[idx]
    vals, cnt = np.unique(s, return_counts=True)
    rare = np.isin(s, vals[cnt < 2 * CFG["inner_splits"]])
    s[rare] = Y[idx][rare] * 1000 + 999
    return s


def make_splits():
    fold = np.full(N, -1, np.int64)
    sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=SEED)
    for k, (_, te) in enumerate(sgkf.split(np.zeros(len(VALID)), strata(VALID), GROUP[VALID])):
        fold[VALID[te]] = k
    roles = {}
    for k in range(K):
        tr = np.where((fold != k) & (fold >= 0))[0]
        inner = StratifiedGroupKFold(n_splits=CFG["inner_splits"], shuffle=True, random_state=SEED + k)
        a, b = next(inner.split(np.zeros(len(tr)), strata(tr), GROUP[tr]))
        role = np.array(["-"] * N, dtype=object)
        role[tr[a]], role[tr[b]], role[fold == k] = "train", "val", "test"
        roles[k] = role
    return fold, roles


def make_source_shift(fold):
    """Test = every ODIR image of the held-out patients: all ODIR patients with a disease image plus a random
    share of the ODIR patients with only normal images. Training never sees an ODIR image of any disease, so the
    model must recognise diseases across cameras/sources. Duplicates of test images are removed from training."""
    role = np.array(["-"] * N, dtype=object)
    used = fold >= 0
    odir = used & (CAM == "ODIR")                       # ODIR files + ODIR copies from other sets
    pat = DF.patient.values
    dis = odir & (Y != NORMAL)                          # every ODIR-camera image of a disease is held out
    if not dis.any():
        return role
    dis_pat = set(pat[dis]) - {""}
    nrm_pat = sorted(set(pat[odir & (pat != "")]) - dis_pat)
    rng_s = np.random.default_rng(SEED)
    n_norm = int(round(CFG["source_shift_normal_frac"] * len(nrm_pat)))
    held = dis_pat | set(rng_s.choice(nrm_pat, n_norm, replace=False) if n_norm else [])
    te = np.where(dis | (odir & (pat != "") & np.isin(pat, list(held))))[0]   # + all other eyes of those patients
    pool = np.where(used & ~np.isin(GROUP, np.unique(GROUP[te])))[0]
    inner = StratifiedGroupKFold(n_splits=CFG["inner_splits"], shuffle=True, random_state=SEED + 99)
    a, b = next(inner.split(np.zeros(len(pool)), strata(pool), GROUP[pool]))
    role[pool[a]], role[pool[b]], role[te] = "train", "val", "test"
    return role


step(f"Part 5: building stratified group {K}-fold splits ...")
split_csv = WORK / "splits.csv"
_old = pd.read_csv(split_csv) if split_csv.exists() else None
if (_old is not None and _old.path.tolist() == DF.path.tolist()
        and np.array_equal(np.where(_old.fold.values >= 0)[0], np.sort(VALID))):
    S_DF = _old
    FOLD = S_DF.fold.values.astype(np.int64)
    ROLES = {k: S_DF[f"role_f{k}"].values for k in range(K)}
    GROUP = S_DF.group.values.astype(np.int64)
    ROLE_SS = S_DF["role_ss"].values if "role_ss" in S_DF else make_source_shift(FOLD)
    print("Loaded existing splits.csv (resume).")
else:
    FOLD, ROLES = make_splits()
    ROLE_SS = make_source_shift(FOLD)
    S_DF = DF.assign(group=GROUP, fold=FOLD, excluded=EXCLUDED, role_ss=ROLE_SS,
                     **{f"role_f{k}": ROLES[k] for k in range(K)})
    S_DF.to_csv(split_csv, index=False)
print("Excluded before splitting (see data_quality.csv):", pd.Series(EXCLUDED[EXCLUDED != ""]).value_counts().to_dict())

INNER = {k: (np.where(ROLES[k] == "train")[0], np.where(ROLES[k] == "val")[0]) for k in range(K)}
for k in range(K):
    tr, va = INNER[k]
    te = np.where(FOLD == k)[0]
    assert not (set(GROUP[tr]) & set(GROUP[te])) and not (set(GROUP[va]) & set(GROUP[te])), "group leak"
    assert not (set(GROUP[tr]) & set(GROUP[va])), "group leak (train/val)"
split_tab = pd.DataFrame({f"fold {k}": [len(INNER[k][0]), len(INNER[k][1]), int((FOLD == k).sum())]
                          for k in range(K)}, index=["train", "val", "test"])
print(split_tab)
if split_tab.loc["test"].min() < 0.6 * len(VALID) / K:
    print("WARNING: unbalanced folds - inspect fig_duplicate_audit and consider lowering dup_max_group.")
print("test-fold class counts:\n", pd.crosstab(FOLD[VALID], DF.cls.values[VALID]))
if not HAS_ANN:
    print("WARNING: no patient ids: folds are grouped by duplicate clusters only.")

# Protocol comparison (Part 14): a random image-level split of the SAME cleaned images, ignoring patients (the other eye of
# a test patient can be in training), as in many published fundus papers.
NAIVE_SPLIT = None
if CFG.get("naive_split_test"):
    _tr_n, _te_n = train_test_split(np.sort(VALID), test_size=CFG["naive_test_frac"], stratify=Y[np.sort(VALID)],
                                    random_state=SEED)
    _tr_n, _va_n = train_test_split(_tr_n, test_size=1 / CFG["inner_splits"], stratify=Y[_tr_n], random_state=SEED)
    NAIVE_SPLIT = (np.sort(_tr_n), np.sort(_va_n), np.sort(_te_n))
    _pt = set(DF.patient.values[NAIVE_SPLIT[0]]) - {""}
    NAIVE_LEAK = float(np.mean([DF.patient.values[i] in _pt for i in NAIVE_SPLIT[2]]))
    print(f"Protocol-comparison split (random, image level): train {len(NAIVE_SPLIT[0])}, val {len(NAIVE_SPLIT[1])}, "
          f"test {len(NAIVE_SPLIT[2])}; {NAIVE_LEAK * 100:.1f}% of its test images have the patient's other eye in training.")

SS_SPLIT = None
if CFG["source_shift_test"] and (ROLE_SS == "test").any():
    SS_SPLIT = tuple(np.where(ROLE_SS == r)[0] for r in ("train", "val", "test"))
    assert not (set(GROUP[SS_SPLIT[0]]) | set(GROUP[SS_SPLIT[1]])) & set(GROUP[SS_SPLIT[2]]), "source-shift leak"
    print("\nSource-shift split (test = held-out ODIR patients; training has no ODIR image of any disease):")
    print(pd.crosstab(ROLE_SS[ROLE_SS != "-"], [DF.cls.values[ROLE_SS != "-"]]))
    print("ODIR-camera images per class in training (must be Normal only):",
          pd.Series(DF.cls.values[SS_SPLIT[0]][CAM[SS_SPLIT[0]] == "ODIR"]).value_counts().to_dict())
elif CFG["source_shift_test"]:
    print("Source-shift test skipped: no ODIR disease images found.")
step("Part 5 done: splits saved to splits.csv.")

## Part 06: Optional external test sets (folder format: <root>/<class folder>/*.jpg) (`part06_external_data.py`)

Optional external folder sets (`CFG['external']`). Overlap with the internal data is removed automatically.

In [ ]:
# ==============================================================================
# PART 6/16: Optional external test sets (folder format: <root>/<class folder>/*.jpg)
# Kaggle: paste into a NEW code cell directly below Part 5. Run the parts in order.
# ==============================================================================
EXT = {}
step(f"Part 6: {len(CFG['external'])} external test set(s) configured.")
for spec in CFG["external"]:
    try:
        df_e = scan_dataset(spec["path"])
    except Exception as e:  # keep going without this external set
        print(f"[external] {spec['name']}: skipped ({e})")
        continue
    df_e = df_e.drop_duplicates("path").reset_index(drop=True) if len(df_e) else df_e
    if df_e.empty:
        print(f"[external] {spec['name']}: no usable images"); continue
    gp_e, dp_e, meta_e = build_cache(df_e.path.tolist(), "ext_" + re.sub(r"\W", "", spec["name"]))
    ok_e = meta_e["ok"]
    # Remove external images that duplicate any internal image (many public fundus sets share images).
    hp_e, _ = hash_pairs(meta_e["hashes"], H_valid, thr=max(0, hash_thr))
    E_e = deep_embed(gp_e, np.arange(len(df_e)))
    cp_e, _ = cos_pairs(E_e, E_valid, thr=min(cos_thr, 0.995))
    overlap = np.zeros(len(df_e), bool)
    overlap[hp_e[:, 0].astype(int)] = True
    overlap[cp_e[:, 0].astype(int)] = True
    keep = ok_e & ~overlap
    EXT[spec["name"]] = dict(df=df_e, g=gp_e, d=dp_e, disc=meta_e["disc"], y=df_e.label.values.astype(np.int64),
                             keep=keep)
    print(f"[external] {spec['name']}: {len(df_e)} images, {overlap.sum()} overlap with internal data "
          f"(excluded), {keep.sum()} used")
    print(pd.Series(df_e.cls[keep]).value_counts())
if not CFG["external"]:
    print("No external folder set configured.")
step(f"Part 6 done: {len(EXT)} external set(s) ready.")

## Part 07: Preprocessing figure (fig_preprocessing) (`part07_preprocessing_figure.py`)

Preprocessing figure (one example per class).

In [ ]:
# ==============================================================================
# PART 7/16: Preprocessing figure (fig_preprocessing)
# Kaggle: paste into a NEW code cell directly below Part 6. Run the parts in order.
# ==============================================================================
step("Part 7: drawing the preprocessing figure ...")
rng = np.random.default_rng(SEED)
if "G_arr" not in globals():
    G_arr = np.load(GPATH, mmap_mode="r")
D_arr = np.load(DPATH, mmap_mode="r")
fig, axes = plt.subplots(NC, 4, figsize=(PAGE_W * 0.62, PAGE_W * 0.62 * NC / 4 * 1.02))
for r, c in enumerate(CLASSES):
    i = int(rng.choice(VALID[Y[VALID] == IDX[c]]))
    raw = cv2.cvtColor(read_bgr(DF.path[i]), cv2.COLOR_BGR2RGB)
    sq = cv2.cvtColor(fundus_square(harmonise(remove_white_pad(read_bgr(DF.path[i]))[0]))[0], cv2.COLOR_BGR2RGB)
    g = G_arr[i].copy()
    cx, cy, _ = META["disc"][i]
    S = g.shape[0]
    half = CFG["disc_frac"] * S / 2
    x0, y0 = int(cx * S - half), int(cy * S - half)
    cv2.rectangle(g, (x0, y0), (int(x0 + 2 * half), int(y0 + 2 * half)), (255, 255, 255), max(1, S // 128))
    for cidx, (im, title) in enumerate([(raw, "Raw"), (sq, "FOV crop"), (g, "CLAHE + disc box"),
                                        (D_arr[i], "Disc ROI")]):
        ax = axes[r, cidx]
        ax.imshow(im); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for sp in ax.spines.values():
            sp.set_visible(False)
        if r == 0:
            ax.set_title(title, fontsize=7)
    axes[r, 0].set_ylabel(SHORT[r], fontsize=7.5)
fig.tight_layout(pad=0.3)
save_fig(fig, "fig_preprocessing")
step("Part 7 done.")

## Part 08: Model zoo + training functions (definitions only, nothing trains yet) (`part08_models_training_engine.py`)

Model zoo and training engine (**class-balanced sampling + class-weighted loss**, augmentation batched on the GPU, EMA, early
stopping on the class-balanced validation NLL). Nothing trains here. At the end every model's pre-trained weights are
downloaded and checked with one forward pass: if this cell fails, fix it before committing.

In [ ]:
# ==============================================================================
# PART 8/16: Model zoo + training functions (definitions only, nothing trains yet)
# Kaggle: paste into a NEW code cell directly below Part 7. Run the parts in order.
# ==============================================================================
NICE = {"resnet50": "ResNet50", "vgg16": "VGG16", "densenet121": "DenseNet121",
        "mobilenet_v3_large": "MobileNetV3-L", "convnext_tiny": "ConvNeXt-T (IN-1k)", "vit_b_16": "ViT-B/16",
        "efficientnet_v2_s": "EfficientNetV2-S", "swin_t": "Swin-T",
        "convnext_small_in22k": "ConvNeXt-S (IN-22k)", "effnetv2_s_in21k": "EffNetV2-S (IN-21k)",
        "dinov2_vitb14": "DINOv2 ViT-B/14", "mobilenetv4_conv_m": "MobileNetV4-M",
        "convnext_tiny_dinov3": "ConvNeXt-T (DINOv3)", "convnextv2_tiny_in22k": "ConvNeXt-V2-T (IN-22k)",
        "dinov3_vitb16": "DINOv3 ViT-B/16"}
# timm checkpoints: larger-scale (IN-22k) or self-supervised (DINOv2/DINOv3, web-scale LVD) pre-training, and
# MobileNetV4. All use the ImageNet mean/std of MEAN/STD.
TIMM = {"convnext_small_in22k": "convnext_small.fb_in22k_ft_in1k_384",
        "effnetv2_s_in21k": "tf_efficientnetv2_s.in21k_ft_in1k",
        "dinov2_vitb14": "vit_base_patch14_dinov2.lvd142m",
        "convnext_tiny_dinov3": "convnext_tiny.dinov3_lvd1689m",       # needs timm >= 1.0.20 (Part 1)
        "convnextv2_tiny_in22k": "convnextv2_tiny.fcmae_ft_in22k_in1k",
        "mobilenetv4_conv_m": "mobilenetv4_conv_medium.e500_r256_in1k",
        "dinov3_vitb16": "vit_base_patch16_dinov3.lvd1689m"}     # DINOv3 ViT-B/16 (2025), needs timm >= 1.0.20
# Used only if the DINOv3 ViT weights cannot be downloaded (Part 8 check): DINOv2 ViT-B/14 with 4 registers.
DINOV3_FALLBACK = ("vit_base_patch14_reg4_dinov2.lvd142m", "DINOv2-reg ViT-B/14 (DINOv3 fallback)")
NO_CAM = {"vit_b_16", "swin_t", "dinov2_vitb14", "dinov3_vitb16"}   # transformer token maps: excluded from CAM metrics
LLRD = {"dinov2_vitb14", "dinov3_vitb16"}           # layer-wise LR decay


def nice(view, name):
    return NICE[name] + (" [disc]" if view == "disc" else "")


def build_model(name, pretrained, size=224):
    if name in TIMM:
        import timm
        kw = dict(pretrained=pretrained, num_classes=NC, drop_rate=0.3, drop_path_rate=CFG["drop_path"])
        if name.startswith(("dinov2", "dinov3")):
            kw["img_size"] = size                       # DINOv2: positional embeddings resampled; DINOv3: RoPE
        m = timm.create_model(TIMM[name], **kw)
        # Grad-CAM layer = last spatial feature map (MobileNetV4: its conv_head runs after global pooling).
        layer = (m.stages[-1] if name.startswith("convnext") else m.bn2 if name.startswith("effnet")
                 else m.blocks[-1] if name.startswith("mobilenetv4") else None)
        return m, m.get_classifier(), layer
    w = "DEFAULT" if pretrained else None

    def head(in_f):
        return nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, NC))

    if name == "resnet50":
        m = models.resnet50(weights=w); m.fc = head(m.fc.in_features); return m, m.fc, m.layer4[-1]
    if name == "vgg16":
        m = models.vgg16(weights=w); m.classifier[6] = head(4096); return m, m.classifier[6], m.features[-1]
    if name == "densenet121":
        m = models.densenet121(weights=w); m.classifier = head(m.classifier.in_features)
        return m, m.classifier, m.features[-1]
    if name == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=w); m.classifier[3] = head(m.classifier[3].in_features)
        return m, m.classifier[3], m.features[-1]
    if name == "convnext_tiny":
        m = models.convnext_tiny(weights=w); m.classifier[2] = head(m.classifier[2].in_features)
        return m, m.classifier[2], m.features[-1]
    if name == "efficientnet_v2_s":
        m = models.efficientnet_v2_s(weights=w); m.classifier[1] = head(m.classifier[1].in_features)
        return m, m.classifier[1], m.features[-1]
    if name == "vit_b_16":
        m = models.vit_b_16(weights=w); m.heads.head = head(m.heads.head.in_features); return m, m.heads.head, None
    if name == "swin_t":
        m = models.swin_t(weights=w); m.head = head(m.head.in_features); return m, m.head, None
    raise ValueError(name)


def in_size(view, name):
    if name == "vit_b_16":
        return 224                                       # torchvision ViT-B/16 is fixed at 224
    s = CFG["disc_size"] if view == "disc" else CFG["size_override"].get(name, CFG["img_size"])
    if name.startswith("dinov2"):
        s = max(56, int(round(s / 14)) * 14)            # patch size 14
    if name.startswith("dinov3"):
        s = max(64, int(round(s / 16)) * 16)            # patch size 16
    return s


def param_groups(model, head, name):
    """AdamW groups: backbone LR, head LR, and layer-wise decay for transformer blocks (DINOv2)."""
    base, head_ids = CFG["lr_override"].get(name, CFG["lr"]), {id(p) for p in head.parameters()}
    nb = len(model.blocks) if name in LLRD and hasattr(model, "blocks") else 0
    groups = {}
    for pname, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if id(p) in head_ids:
            key, lr = "head", CFG["head_lr"]
        elif nb:
            mt = re.match(r"blocks\.(\d+)\.", pname)
            depth = int(mt.group(1)) + 1 if mt else (0 if pname.startswith(
                ("patch_embed", "cls_token", "pos_embed", "reg_token", "mask_token")) else nb + 1)
            key, lr = f"d{depth}", base * CFG["layer_decay"] ** (nb + 1 - depth)
        else:
            key, lr = "body", base
        groups.setdefault(key, {"params": [], "lr": lr})["params"].append(p)
    return list(groups.values())


# ---- Data loading + augmentation ----
# Kaggle gives 4 CPU cores. Augmenting every image on the CPU (rotation, colour jitter, blur, JPEG) was the
# bottleneck: the GPUs waited for data (ResNet50 ~40 img/s at 384 px; MobileNetV4, a much lighter model, was
# even slower because its many small layers also need CPU time). Now the CPU workers only copy bytes from the
# cache (+ JPEG re-compression of ~30% of the training images) and all other augmentations run batched on the GPU.
_RGB2YIQ = torch.tensor([[0.299, 0.587, 0.114], [0.596, -0.274, -0.322], [0.211, -0.523, 0.312]])
_YIQ2RGB = torch.linalg.inv(_RGB2YIQ)


def _uni(n, lo, hi, dev):
    return lo + (hi - lo) * torch.rand(n, device=dev)


def _gauss_blur(x, sig):
    """Separable 5-tap Gaussian blur with one sigma per image (reflect padding, like torchvision)."""
    B, C, H, W = x.shape
    k = torch.arange(-2, 3, device=x.device, dtype=x.dtype)
    ker = torch.exp(-(k[None] ** 2) / (2 * sig[:, None] ** 2))
    ker = (ker / ker.sum(1, keepdim=True)).repeat_interleave(C, 0)            # (B*C, 5)
    y = x.contiguous().view(1, B * C, H, W)
    y = F.conv2d(F.pad(y, (2, 2, 0, 0), mode="reflect"), ker.view(B * C, 1, 1, 5), groups=B * C)
    y = F.conv2d(F.pad(y, (0, 0, 2, 2), mode="reflect"), ker.view(B * C, 1, 5, 1), groups=B * C)
    return y.view(B, C, H, W)


@torch.no_grad()
def gpu_augment(x, view):
    """Training augmentation of a whole batch on the GPU; x = float (B, 3, S, S) in [0, 1]. Same operations and
    ranges as the former per-image torchvision pipeline: horizontal/vertical flip, rotation +-25 deg, translation
    (4% of the width; disc view 8% = also simulates localisation error), scale 0.92-1.08, colour jitter
    (p 0.8: brightness/contrast 0.25, saturation 0.2, hue 0.02), Gaussian blur (p 0.2, sigma 0.1-1.2), random
    resolution (p 0.25, 60-100%: hides resolution differences between sources) and Gaussian noise (p 0.2)."""
    B, dev = x.shape[0], x.device
    # flips + affine as one resampling step (affine_grid maps output -> input coordinates, range -1..1)
    shift = 0.08 if view == "disc" else 0.04
    ang = _uni(B, -25, 25, dev) * (math.pi / 180)
    sc = _uni(B, 0.92, 1.08, dev)
    fx = 1 - 2 * (torch.rand(B, device=dev) < 0.5).float()
    fy = 1 - 2 * (torch.rand(B, device=dev) < 0.5).float()
    c, s = torch.cos(ang) / sc, torch.sin(ang) / sc
    tx, ty = _uni(B, -2 * shift, 2 * shift, dev), _uni(B, -2 * shift, 2 * shift, dev)
    theta = torch.stack([torch.stack([c * fx, -s * fy, tx], 1), torch.stack([s * fx, c * fy, ty], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    # colour jitter (p 0.8); factors are 1 (no change) for the other images
    on = (torch.rand(B, device=dev) < 0.8).float()
    v = lambda t: t.view(B, 1, 1, 1)
    gray = lambda t: (0.2989 * t[:, 0] + 0.587 * t[:, 1] + 0.114 * t[:, 2]).unsqueeze(1)
    x = (x * v(1 + on * _uni(B, -0.25, 0.25, dev))).clamp(0, 1)                       # brightness
    ct = v(1 + on * _uni(B, -0.25, 0.25, dev))
    x = (ct * x + (1 - ct) * gray(x).mean((2, 3), keepdim=True)).clamp(0, 1)          # contrast
    sa = v(1 + on * _uni(B, -0.2, 0.2, dev))
    x = (sa * x + (1 - sa) * gray(x)).clamp(0, 1)                                      # saturation
    h = on * _uni(B, -0.02, 0.02, dev) * (2 * math.pi)                                 # hue: rotate chroma (YIQ)
    R = torch.zeros(B, 3, 3, device=dev)
    R[:, 0, 0], R[:, 1, 1], R[:, 1, 2], R[:, 2, 1], R[:, 2, 2] = 1, h.cos(), -h.sin(), h.sin(), h.cos()
    M = _YIQ2RGB.to(dev) @ R @ _RGB2YIQ.to(dev)
    x = torch.einsum("bij,bjhw->bihw", M, x).clamp(0, 1)
    # Gaussian blur (p 0.2)
    sel = (torch.rand(B, device=dev) < 0.2).nonzero().squeeze(1)
    if sel.numel():
        x[sel] = _gauss_blur(x[sel], _uni(sel.numel(), 0.1, 1.2, dev))
    # random resolution (p 0.25; mild: blur itself is a cataract sign)
    if CFG["source_robust_aug"]:
        sel = (torch.rand(B, device=dev) < 0.25).nonzero().squeeze(1)
        if sel.numel():
            S = x.shape[-1]
            r = max(32, int(S * random.uniform(0.6, 1.0)))
            lo = F.interpolate(x[sel], size=(r, r), mode="bilinear", antialias=True, align_corners=False)
            x[sel] = F.interpolate(lo, size=(S, S), mode="bilinear", antialias=True, align_corners=False)
    # Gaussian noise (p 0.2)
    on = v((torch.rand(B, device=dev) < 0.2).float())
    return (x + on * 0.02 * torch.randn_like(x)).clamp(0, 1)


def class_weights(idx):
    """Loss weights against class imbalance, from the training images only (mean weight per image = 1)."""
    if CFG["imbalance"] == "none":
        return None
    cnt = np.bincount(Y[idx], minlength=NC).astype(float)
    w = np.maximum(cnt, 1) ** (-0.5 if CFG["imbalance"] == "sqrt_inv" else -1.0)
    w = w * cnt.sum() / max((w * cnt).sum(), 1e-9)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def eval_weights(idx):
    """Class weights of the TOTAL training objective (balanced sampling x loss weight), for the validation loss that
    drives early stopping and checkpoint choice. None = unweighted."""
    e = float(CFG.get("sampler_power", 0)) + {"none": 0.0, "sqrt_inv": 0.5, "inv": 1.0}[CFG["imbalance"]]
    if e == 0:
        return None
    cnt = np.bincount(Y[idx], minlength=NC).astype(float)
    w = np.maximum(cnt, 1) ** (-e)
    w = w * cnt.sum() / max((w * cnt).sum(), 1e-9)
    return torch.tensor(w, dtype=torch.float32)


def sample_weights(idx, labels):
    """Per-image draw probability ~ 1 / (class size)^sampler_power (None = plain shuffling)."""
    p = float(CFG.get("sampler_power", 0))
    if p <= 0:
        return None
    y = np.asarray(labels)[np.asarray(idx)]
    cnt = np.bincount(y[y >= 0], minlength=NC).astype(float)
    return torch.as_tensor(np.maximum(cnt[np.clip(y, 0, None)], 1) ** (-p), dtype=torch.double)


class BatchDS(Dataset):
    """One item = one whole batch of uint8 images (B, S, S, 3) from the memory-mapped .npy cache. The CPU only
    copies bytes, plus JPEG re-compression (p 0.3, quality 40-95: hides compression differences between sources)
    when `jpeg` is on; everything else runs on the GPU (GPUBatches / gpu_augment)."""

    def __init__(self, npy_path, idx, labels, jpeg):
        self.npy_path, self.idx, self.labels, self.jpeg = str(npy_path), np.asarray(idx), labels, jpeg
        self.arr = None

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, pos):
        if self.arr is None:
            self.arr = np.load(self.npy_path, mmap_mode="r")
        j = self.idx[np.asarray(pos)]
        o = np.argsort(j)                              # read the memory map in file order
        x = np.empty((len(j),) + self.arr.shape[1:], np.uint8)
        x[o] = self.arr[j[o]]
        if self.jpeg:
            for i in np.where(np.random.rand(len(j)) < 0.3)[0]:
                q = int(np.random.randint(40, 96))
                enc = cv2.imencode(".jpg", np.ascontiguousarray(x[i, :, :, ::-1]), [cv2.IMWRITE_JPEG_QUALITY, q])[1]
                x[i] = cv2.imdecode(enc, cv2.IMREAD_COLOR)[:, :, ::-1]
        return (torch.from_numpy(x), torch.from_numpy(np.asarray(self.labels)[j].astype(np.int64)),
                torch.from_numpy(j.astype(np.int64)))


class GPUBatches:
    """Iterates a batch DataLoader and yields (x, y, j): x = normalised float images on the GPU (channels-last),
    resized to the model's input size and augmented there when training; y, j stay on the CPU."""

    def __init__(self, dl, train, view, size):
        self.dl, self.train, self.view, self.size = dl, train, view, size

    def __len__(self):
        return len(self.dl)

    def __iter__(self):
        mean = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
        std = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)
        for x, y, j in self.dl:
            x = x.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255)
            if x.shape[-1] != self.size:
                x = F.interpolate(x, size=(self.size, self.size), mode="bilinear", antialias=True,
                                  align_corners=False)
            if self.train:
                x = gpu_augment(x, self.view)
            yield ((x - mean) / std).contiguous(memory_format=CL), y, j


def loader(npy, idx, labels, train, view, size, bs, persistent=False):
    ds = BatchDS(npy, idx, labels, jpeg=train and CFG["source_robust_aug"])
    sw = sample_weights(idx, labels) if train else None
    base = (torch.utils.data.WeightedRandomSampler(sw, num_samples=len(idx), replacement=True) if sw is not None
            else torch.utils.data.RandomSampler(ds) if train else torch.utils.data.SequentialSampler(ds))
    smp = torch.utils.data.BatchSampler(base, bs, drop_last=train and len(idx) > bs)
    dl = DataLoader(ds, sampler=smp, batch_size=None, num_workers=NW, pin_memory=DEVICE.type == "cuda",
                    persistent_workers=persistent and NW > 0, prefetch_factor=4 if NW > 0 else None)
    return GPUBatches(dl, train, view, size)


@torch.no_grad()
def infer(model, dl, tta):
    model.eval()
    L, LT, I = [], [], []
    for x, _, j in dl:
        x = x.to(DEVICE, non_blocking=True, memory_format=CL)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            lg = model(x).float()
            lt = lg
            if tta:
                lt = (lg + model(torch.flip(x, dims=[3])).float() + model(torch.flip(x, dims=[2])).float()
                      + model(torch.flip(x, dims=[2, 3])).float()) / 4
        L.append(lg.cpu()); LT.append(lt.cpu()); I.append(j)
    return torch.cat(L).numpy(), torch.cat(LT).numpy(), torch.cat(I).numpy()


def macro_f1(y, pred):
    return f1_score(y, pred, average="macro", labels=list(range(NC)), zero_division=0)


def par(m):
    """Multi-GPU wrapper (Kaggle T4 x2): every batch is split over the GPUs. The wrapper shares the parameters of
    `m`, so EMA, gradient clipping and the saved state_dict keep working on `m` itself."""
    return nn.DataParallel(m) if USE_DP else m


def train_eval(view, name, k, bs, split=None):
    """split=None -> CV fold k; otherwise an explicit (train, val, test) index triple (source-shift test)."""
    tr, va = INNER[k] if split is None else split[:2]
    te = np.where(FOLD == k)[0] if split is None else split[2]
    npy, size = (GPATH if view == "global" else DPATH), in_size(view, name)
    model, head, _ = build_model(name, CFG["pretrained"], size)
    model = model.to(DEVICE, memory_format=CL)
    fwd = par(model)
    opt = torch.optim.AdamW(param_groups(model, head, name), weight_decay=CFG["weight_decay"])
    # EMA of the weights: evaluated every epoch, and the best EMA checkpoint is the one kept.
    ema = copy.deepcopy(model).eval() if CFG["ema"] else None
    if ema is not None:
        for p in ema.parameters():
            p.requires_grad_(False)
        pairs = list(zip(ema.state_dict().values(), model.state_dict().values()))
        ema_f = [(e, m) for e, m in pairs if e.dtype.is_floating_point]
        ema_i = [(e, m) for e, m in pairs if not e.dtype.is_floating_point]
    step = 0
    dl_tr = loader(npy, tr, Y, True, view, size, bs, persistent=True)
    dl_va = loader(npy, va, Y, False, view, size, bs * 2, persistent=True)
    steps, warm = CFG["epochs"] * len(dl_tr), CFG["warmup_epochs"] * len(dl_tr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / max(1, warm) if s < warm else
                                              0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    cw = class_weights(tr)
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=CFG["label_smoothing"])
    # Validation loss with the class weights of the whole training objective (balanced sampling x loss weight): early
    # stopping and the kept checkpoint follow the objective that is optimised (unweighted NLL would favour Normal).
    cw_cpu = eval_weights(tr)
    hist, best_nll, best_state, best_ep, bad = [], float("inf"), None, 0, 0
    a = CFG["mixup_alpha"]
    for ep in range(CFG["epochs"]):
        t_ep = time.time()
        model.train()
        tl, nb = 0.0, 0
        for x, yb, _ in pbar(dl_tr, desc=f"  epoch {ep + 1}/{CFG['epochs']}", unit="batch", leave=False):
            x = x.to(DEVICE, non_blocking=True, memory_format=CL)
            yb = yb.to(DEVICE, non_blocking=True)
            mix = a > 0 and random.random() < CFG["mixup_prob"]
            if mix:
                lam = float(np.random.beta(a, a))
                perm = torch.randperm(x.size(0), device=DEVICE)
                x = lam * x + (1 - lam) * x[perm]
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out = fwd(x)
                loss = lam * crit(out, yb) + (1 - lam) * crit(out, yb[perm]) if mix else crit(out, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                d = min(CFG["ema_decay"], (1 + step) / (10 + step))
                with torch.no_grad():
                    torch._foreach_mul_([e for e, _ in ema_f], d)
                    torch._foreach_add_([e for e, _ in ema_f], [m.detach() for _, m in ema_f], alpha=1 - d)
                    for e, m in ema_i:
                        e.copy_(m)
            step += 1
            tl += loss.item(); nb += 1
        eval_model = ema if ema is not None else model
        lg, _, vi = infer(par(eval_model), dl_va, tta=False)
        yv = torch.from_numpy(Y[vi])
        lgt = torch.from_numpy(lg)
        v_obj = F.cross_entropy(lgt, yv, weight=cw_cpu, label_smoothing=CFG["label_smoothing"]).item()
        v_nll = F.cross_entropy(lgt, yv, weight=cw_cpu).item()
        pred = lg.argmax(1)
        hist.append(dict(epoch=ep + 1, train_loss=tl / max(1, nb), val_loss=v_obj, val_nll=v_nll,
                         val_acc=float((pred == Y[vi]).mean()), val_f1=float(macro_f1(Y[vi], pred)),
                         lr=opt.param_groups[0]["lr"], sec=time.time() - t_ep))
        if v_nll < best_nll - 1e-4:
            best_nll, best_ep, bad = v_nll, ep + 1, 0
            best_state = {kk: vv.detach().cpu().clone() for kk, vv in eval_model.state_dict().items()}
        else:
            bad += 1
        print(f"  ep {ep + 1:2d} train {tl / max(1, nb):.4f} | val loss {v_obj:.4f} nll {v_nll:.4f} "
              f"acc {hist[-1]['val_acc']:.4f} f1 {hist[-1]['val_f1']:.4f} | {hist[-1]['sec']:.0f}s", flush=True)
        if ep + 1 >= CFG["min_epochs"] and bad >= CFG["patience"]:
            break
    if best_state is None:                    # validation NLL was never finite: keep the last weights
        print("  WARNING: no finite validation loss; using last-epoch weights")
        best_state, best_ep = (ema if ema is not None else model).state_dict(), len(hist)
    model.load_state_dict(best_state)
    del ema, fwd
    res = {}
    for tag, idx_, npy_, lab_ in [("val", va, npy, Y), ("test", te, npy, Y)]:
        l, lt, ii = infer(par(model), loader(npy_, idx_, lab_, False, view, size, bs * 2), CFG["tta"])
        res[f"{tag}_idx"], res[f"{tag}_logits"], res[f"{tag}_logits_tta"] = ii, l, lt
    for ename, e in (EXT.items() if split is None else []):
        _, lt, _ = infer(par(model), loader(e["g"] if view == "global" else e["d"], np.arange(len(e["y"])), e["y"],
                                            False, view, size, bs * 2), CFG["tta"])
        res[f"ext__{ename}"] = lt
    del dl_tr, dl_va
    return model, res, hist, best_ep


def run_job(view, name, k, split=None):
    """k = CV fold; with split=(train, val, test) and k="ss" this is the source-shift run."""
    out = RUNS / f"{view}__{name}__f{k}.npz" if split is None else RUNS_SS / f"{view}__{name}__{k}.npz"
    if out.exists():
        return "skip"
    step(f"=== {'fold ' + str(k) if split is None else str(k).upper() + ' SPLIT'} | {view} | {nice(view, name)} | "
         f"input {in_size(view, name)} px ===")
    seed_everything(SEED + 1000 * (k if split is None else 99) + zlib.crc32(f"{view}{name}".encode()) % 1000)
    t = time.time()
    bs = CFG["bs_override"].get(name, CFG["batch_size"])
    while True:
        try:
            model, res, hist, best_ep = train_eval(view, name, k, bs, split)
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if bs <= 8:
                raise
            bs //= 2
            print(f"  OOM -> retrying with batch size {bs}")
    np.savez_compressed(out, **res, hist=np.array(json.dumps(hist)), best_epoch=best_ep,
                        batch_size=bs, train_sec=time.time() - t)
    if split is None and k == CFG["save_weights_fold"]:
        torch.save({kk: (vv.half() if vv.is_floating_point() else vv) for kk, vv in model.state_dict().items()},
                   WTS / f"{view}__{name}.pt")
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    key = "test_logits_tta" if CFG["tta"] else "test_logits"
    acc_t = float((res[key].argmax(1) == Y[res["test_idx"]]).mean())
    print(f"  done in {(time.time() - t) / 60:.1f} min (best epoch {best_ep}) | "
          f"{'this fold' if split is None else str(k) + ' split'} test accuracy {acc_t * 100:.2f}%, macro-F1 "
          f"{macro_f1(Y[res['test_idx']], res[key].argmax(1)) * 100:.2f}% (single model, before ensembling)")
    return "done"


def preflight():
    """Download every model's pre-trained weights once and run one forward pass, so that a missing package, an
    old timm or a failed download shows up now (interactive Parts 1-8), not hours into the commit run."""
    todo = [("global", m) for m in CFG["global_models"]] + [("disc", m) for m in CFG["disc_models"]]
    if CFG["source_shift_test"]:
        todo += [("global", m) for m in CFG["source_shift_models"]]
    if CFG.get("naive_split_test"):
        todo += [("global", m) for m in CFG["naive_split_models"]]
    bad = []
    for view, name in dict.fromkeys(todo):
        s = in_size(view, name)
        try:
            try:
                m, _, layer = build_model(name, CFG["pretrained"], s)
            except Exception as e0:
                if name != "dinov3_vitb16":
                    raise
                print(f"  DINOv3 ViT weights unavailable ({type(e0).__name__}: {e0}); using {DINOV3_FALLBACK[1]} instead. "
                      "Name the model accordingly in the paper.")
                TIMM[name], NICE[name] = DINOV3_FALLBACK
                m, _, layer = build_model(name, CFG["pretrained"], s)
            with torch.no_grad():
                out = m.eval()(torch.zeros(1, 3, s, s))
            assert tuple(out.shape) == (1, NC), f"output shape {tuple(out.shape)}"
            assert name in NO_CAM or layer is not None, "no Grad-CAM layer"
            n_par = sum(p.numel() for p in m.parameters()) / 1e6
            print(f"  ok  {nice(view, name):28s} input {s} px, {n_par:5.1f} M params")
            del m
        except Exception as e:
            bad.append(f"{nice(view, name)}: {type(e).__name__}: {e}")
            print(f"  FAILED  {bad[-1]}")
    gc.collect()
    if bad:
        raise RuntimeError("Model check failed (fix before committing; Internet must be ON for the downloads):\n"
                           + "\n".join(bad))


step("Part 8: checking every model (downloads pre-trained weights, one forward pass on CPU) ...")
preflight()
step("Part 8 done: models and training functions defined (nothing is trained in this part).")

## Part 09: TRAIN all models on all folds + the protocol-comparison run (the long part, resumable) (`part09_train_all.py`)

**Training (~3.3 h with preset `fast`, ~1.2 h with `lite`, ~4.6 h with `full`).** Every model x fold, then one run per model in
`naive_split_models` on the random image-level split. Each run prints its test accuracy and macro-F1 when it finishes, and after
every fold the projected end of training. A run that would end after `time_budget_h` is not started, so Parts 10-16 always
finish; re-run with this output attached to resume.

In [ ]:
# ==============================================================================
# PART 9/16: TRAIN all models on all folds + the source-shift runs  (the long part: hours, resumable)
# Kaggle: paste into a NEW code cell directly below Part 8. Run the parts in order.
# ==============================================================================
import traceback

JOBS = [("global", m) for m in CFG["global_models"]] + [("disc", m) for m in CFG["disc_models"]]
FOLDS_TO_RUN = CFG["run_folds"] if CFG["run_folds"] is not None else list(range(K))
FAILED = {}                                   # (view, name) -> error; a failing model is skipped, the rest go on


def expected_sec(view, name):
    """Duration of earlier runs of the same model (any fold); 15 min if none yet."""
    secs = [float(np.load(f)["train_sec"]) for f in RUNS.glob(f"{view}__{name}__f*.npz")]
    return float(np.max(secs)) if secs else 15 * 60.0


def projection(todo_left):
    """Clock time at which training would end, from the measured run times (printed after every fold)."""
    sec = sum(expected_sec(v, n) for v, n, _ in todo_left)
    if SS_SPLIT is not None:
        sec += sum(expected_sec("global", m) for m in CFG["source_shift_models"]
                   if not (RUNS_SS / f"global__{m}__ss.npz").exists())
    if NAIVE_SPLIT is not None:
        sec += sum(expected_sec("global", m) for m in CFG["naive_split_models"]
                   if not (RUNS_SS / f"global__{m}__naive.npz").exists())
    end_h = (time.time() - T0 + sec) / 3600
    step(f"Projected end of training: {time.strftime('%H:%M', time.localtime(time.time() + sec))} "
         f"({end_h:.2f} h after Part 2; budget {CFG['time_budget_h']} h"
         f"{', some runs will be skipped' if end_h > CFG['time_budget_h'] else ''}).")


def safe_run(view, name, k, split=None):
    """run_job, but one broken model must not abort a 10-hour commit: log the error and skip that model."""
    if (view, name) in FAILED:
        return "failed"
    try:
        return run_job(view, name, k, split)
    except Exception as e:
        FAILED[(view, name)] = f"{type(e).__name__}: {e}"
        print(f"  !!! {nice(view, name)} FAILED (fold {k}); its remaining runs are skipped in this session:")
        traceback.print_exc()
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return "failed"


TODO = [(v, n, k) for k in FOLDS_TO_RUN for v, n in JOBS if not (RUNS / f"{v}__{n}__f{k}.npz").exists()]
step(f"Part 9: {len(TODO)} training runs to do, {K * len(JOBS) - len(TODO)} already finished "
     f"({len(JOBS)} models x {K} folds). Each run prints one line per epoch.")
overall = pbar(total=len(TODO), desc="ALL TRAINING RUNS", unit="run", mininterval=0)
for idx, (view, name, k) in enumerate(TODO, 1):
    if (time.time() - T0 + expected_sec(view, name)) / 3600 > CFG["time_budget_h"]:
        step("Time budget reached: stopping cleanly so the outputs are saved.")
        break
    step(f"--> [Run {idx}/{len(TODO)}] Starting fold {k} | {view} | {nice(view, name)}")
    overall.set_postfix_str(f"fold {k} | {nice(view, name)}")
    safe_run(view, name, k)
    overall.update(1)
    if idx == len(TODO) or TODO[idx][2] != k:          # a fold has just finished
        projection(TODO[idx:])
overall.close()
remaining = [(v, n, k) for k in range(K) for v, n in JOBS if not (RUNS / f"{v}__{n}__f{k}.npz").exists()]
step(f"Part 9: CV runs finished: {K * len(JOBS) - len(remaining)}/{K * len(JOBS)}. " +
     ("Re-run a new version with this output attached to continue." if remaining else "All training complete."))

# Source-shift test: one extra run per model in CFG["source_shift_models"], trained without any ODIR image of a
# disease (see Part 5).
if SS_SPLIT is not None:
    for _m in CFG["source_shift_models"]:
        _ss = ("global", _m)
        if (RUNS_SS / f"{_ss[0]}__{_ss[1]}__ss.npz").exists():
            step(f"Source-shift run of {nice(*_ss)} already finished.")
        elif (time.time() - T0 + expected_sec(*_ss)) / 3600 > CFG["time_budget_h"]:
            step(f"Time budget reached: source-shift run of {nice(*_ss)} postponed to the next version.")
        else:
            step(f"--> Source-shift run: {nice(*_ss)} trained on {len(SS_SPLIT[0])} images, tested on "
                 f"{len(SS_SPLIT[2])} held-out ODIR images")
            safe_run(*_ss, "ss", split=SS_SPLIT)
# Protocol comparison: the same model trained once on the random image-level split of Part 5 (patients ignored).
if NAIVE_SPLIT is not None:
    for _m in CFG["naive_split_models"]:
        if (RUNS_SS / f"global__{_m}__naive.npz").exists():
            step(f"Protocol-comparison run of {nice('global', _m)} already finished.")
        elif (time.time() - T0 + expected_sec("global", _m)) / 3600 > CFG["time_budget_h"]:
            step(f"Time budget reached: protocol-comparison run of {nice('global', _m)} postponed to the next version.")
        else:
            step(f"--> Protocol-comparison run: {nice('global', _m)} on a random image-level split "
                 f"({len(NAIVE_SPLIT[0])} train / {len(NAIVE_SPLIT[2])} test images)")
            safe_run("global", _m, "naive", split=NAIVE_SPLIT)
if FAILED:
    print("\n*** FAILED MODELS (results below are interim without them):")
    for (v, n), err in FAILED.items():
        print(f"   {nice(v, n)}: {err}")
step("Part 9 done.")

## Part 10: Model complexity table (params, GFLOPs, latency) (`part10_complexity.py`)

Parameters, GFLOPs and latency of each model.

In [ ]:
# ==============================================================================
# PART 10/16: Model complexity table (params, GFLOPs, latency)
# Kaggle: paste into a NEW code cell directly below Part 9. Run the parts in order.
# ==============================================================================
def complexity_row(view, name):
    s = in_size(view, name)
    m, _, _ = build_model(name, pretrained=False, size=s)
    m = m.to(DEVICE).eval()
    x = torch.randn(1, 3, s, s, device=DEVICE)
    params = sum(p.numel() for p in m.parameters()) / 1e6
    gflops = float("nan")
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc, torch.no_grad():
            m(x)
        gflops = fc.get_total_flops() / 1e9
    except Exception:
        pass
    reps = 50 if DEVICE.type == "cuda" else 3
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        for _ in range(5):
            m(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t = time.time()
        for _ in range(reps):
            m(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    ms = (time.time() - t) / reps * 1000
    del m
    return dict(Model=nice(view, name), Input=f"{s}x{s}", **{"Params (M)": round(params, 2),
                "GFLOPs": round(gflops, 2), "Latency (ms)": round(ms, 2)})


step("Part 10: measuring model complexity ...")
CPLX = pd.DataFrame([complexity_row(v, n) for v, n in pbar(JOBS, desc="Complexity", unit="model")])
print(CPLX.to_string(index=False))
to_latex(CPLX, "tab_complexity", "Model complexity (batch size 1, " +
         ("GPU, FP16" if USE_AMP else "CPU") + ").", "tab:complexity")

## Part 11: Ensembles (soft vote, stacking = proposed), decision prior, calibration, metrics (`part11_ensembles_metrics.py`)

Every model's probabilities are moved from its balanced training prior to the decision prior (class frequency ^
`decision_prior_power`, post-hoc logit adjustment), then soft voting, stacking (= proposed, same prior), temperature scaling and
pooled metrics.

In [ ]:
# ==============================================================================
# PART 11/16: Ensembles (soft vote, stacking = proposed), calibration, metrics
# Kaggle: paste into a NEW code cell directly below Part 10. Run the parts in order.
# ==============================================================================
def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def log_p(P):
    return np.log(np.clip(P, 1e-7, 1.0))


def fit_temperature(z, y):
    """Scalar temperature minimising validation NLL of softmax(z / T)."""
    def nll(t):
        s = z / t
        s = s - s.max(1, keepdims=True)
        lp = s - np.log(np.exp(s).sum(1, keepdims=True))
        return -lp[np.arange(len(y)), y].mean()
    return minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded").x


def ece_score(P, y, n_bins=15):
    conf, pred = P.max(1), P.argmax(1)
    e = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e


def compute_metrics(y, P, w=None):
    P = P / P.sum(1, keepdims=True)
    pred = P.argmax(1)
    labels = list(range(NC))
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", labels=labels, zero_division=0,
                                                 sample_weight=w)
    out = dict(acc=accuracy_score(y, pred, sample_weight=w), bacc=balanced_accuracy_score(y, pred, sample_weight=w),
               prec=p, rec=r, f1=f, kappa=cohen_kappa_score(y, pred, sample_weight=w),
               mcc=matthews_corrcoef(y, pred, sample_weight=w))
    try:
        out["auc"] = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=labels, sample_weight=w)
    except ValueError:
        out["auc"] = float("nan")
    if w is None:
        out["nll"] = float(-np.log(np.clip(P[np.arange(len(y)), y], 1e-7, 1)).mean())
        out["ece"] = ece_score(P, y)
        cm = confusion_matrix(y, pred, labels=labels)
        for c in range(NC):
            tp, fn = cm[c, c], cm[c].sum() - cm[c, c]
            fp, tn = cm[:, c].sum() - tp, cm.sum() - cm[c].sum() - (cm[:, c].sum() - tp)
            out[f"sens_{SHORT[c]}"] = tp / max(1, tp + fn)
            out[f"spec_{SHORT[c]}"] = tn / max(1, tn + fp)
        out["normal_as_glaucoma"] = int(cm[NORMAL, GLAUCOMA])
        out["glaucoma_as_normal"] = int(cm[GLAUCOMA, NORMAL])
        abn = (y != NORMAL).astype(int)
        out["referable_sens"] = float(((pred != NORMAL) & (abn == 1)).sum() / max(1, abn.sum()))
        out["referable_spec"] = float(((pred == NORMAL) & (abn == 0)).sum() / max(1, (1 - abn).sum()))
        try:
            out["referable_auc"] = roc_auc_score(abn, 1 - P[:, NORMAL])
        except ValueError:
            out["referable_auc"] = float("nan")
    return out


def load_runs():
    R = {}
    for f in sorted(RUNS.glob("*.npz")):
        view, name, fk = f.stem.split("__")
        if (view, name) not in JOBS:
            continue
        z = np.load(f, allow_pickle=False)
        R[(view, name, int(fk[1:]))] = {kk: z[kk] for kk in z.files}
    return R


step("Part 11: loading finished runs ...")
R = load_runs()
done = {k: {(v, n) for (v, n, kk) in R if kk == k} for k in range(K)}
FOLDS_DONE = [k for k in range(K) if set(JOBS) <= done[k]]
CANDS = list(JOBS)
if not FOLDS_DONE:
    ks = [k for k in range(K) if done[k]]
    common = set.intersection(*[done[k] for k in ks]) if ks else set()
    FOLDS_DONE, CANDS = ks, [j for j in JOBS if j in common]
assert CANDS and FOLDS_DONE, "No finished runs yet - run section 7 first."
INTERIM = len(FOLDS_DONE) < K or len(CANDS) < len(JOBS)
if INTERIM:
    print(f"*** INTERIM RESULTS: folds {FOLDS_DONE}, {len(CANDS)}/{len(JOBS)} models. Do not report these. ***")
G_C = [c for c in CANDS if c[0] == "global"]
D_C = [c for c in CANDS if c[0] == "disc"]
SINGLE_NAMES = {c: nice(*c) for c in CANDS}
M_BEST, M_TOP2, M_ALLG = "Best single (val-selected)", "Soft vote: top-2 (submitted strategy)", "Soft vote: all global"
M_ALLGD, M_STG = "Soft vote: global + disc", "Stacking: global only"
M_PROP = "DA-MVE (proposed)" if D_C else "Stacking (proposed)"
ENS_ORDER = [M_BEST, M_TOP2, M_ALLG] + ([M_ALLGD] if D_C else []) + ([M_STG] if D_C else []) + [M_PROP]
METHODS = [SINGLE_NAMES[c] for c in CANDS] + ENS_ORDER


def _full(p, classes):
    """Place predict_proba columns into all NC classes (robust if a class is absent from validation)."""
    out = np.full((len(p), NC), 1e-6)
    out[:, classes] = p
    return out / out.sum(1, keepdims=True)


def stack_class_weight(yv):
    """The training loss's class weighting (CFG['imbalance']), so the stacker's implied class prior matches its
    members' and 'proposed vs. single model' is not confounded by a different prior."""
    e = 1.0 - float(CFG["decision_prior_power"])     # validation images are naturally distributed: weight n^-(1-d)
    if e == 0:
        return None
    cls, cnt = np.unique(yv, return_counts=True)
    w = cnt.astype(float) ** (-e)
    w = w * cnt.sum() / (w * cnt).sum()
    return {int(c): float(v) for c, v in zip(cls, w)}


def train_prior_power():
    """Exponent of the class prior the models were trained under: class frequency ^ (1 - sampler_power - loss exponent)."""
    return 1.0 - float(CFG.get("sampler_power", 0)) - {"none": 0.0, "sqrt_inv": 0.5, "inv": 1.0}[CFG["imbalance"]]


def prior_adjust(P, train_idx):
    """Post-hoc logit adjustment: from the training prior to the decision prior (class frequency ^ decision_prior_power,
    frequencies of the fold's training images). Changes only the class prior, not the image evidence."""
    cnt = np.maximum(np.bincount(Y[train_idx], minlength=NC).astype(float), 1)
    Q = P * cnt ** (float(CFG["decision_prior_power"]) - train_prior_power())
    return Q / Q.sum(1, keepdims=True)


def fit_stacker(Xv, yv, gv):
    """Multinomial LR on members' log-probabilities. Returns (predict_fn, out-of-fold validation probs)."""
    present = np.unique(yv)
    cw = stack_class_weight(yv)
    nspl = int(min(5, np.bincount(yv)[present].min()))
    if nspl >= 2:
        cv = list(StratifiedGroupKFold(n_splits=nspl, shuffle=True, random_state=SEED).split(Xv, yv, gv))
        clf = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=np.logspace(-3, 2, 11), cv=cv,
                                                                   scoring="neg_log_loss", max_iter=5000,
                                                                   class_weight=cw))
        clf.fit(Xv, yv)
        c_best = float(np.atleast_1d(clf[-1].C_)[0])
        oof = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(C=c_best, max_iter=5000,
                                                                                   class_weight=cw)),
                                Xv, yv, cv=cv, method="predict_proba")
    else:
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000, class_weight=cw)).fit(Xv, yv)
        oof = clf.predict_proba(Xv)
    return (lambda X: _full(clf.predict_proba(X), clf[-1].classes_)), _full(oof, present)


OOF = {m: np.full((N, NC), np.nan) for m in METHODS}          # calibrated
OOF_RAW = {m: np.full((N, NC), np.nan) for m in METHODS}      # before temperature scaling
EXT_P = {m: {e: [] for e in EXT} for m in METHODS}
FOLD_ROWS, CHOICES, TEMPS = [], [], []
for k in pbar(FOLDS_DONE, desc="Ensembles + calibration", unit="fold"):
    rv = R[CANDS[0] + (k,)]
    vi, ti = rv["val_idx"], rv["test_idx"]
    for c in CANDS:
        assert np.array_equal(R[c + (k,)]["val_idx"], vi) and np.array_equal(R[c + (k,)]["test_idx"], ti)
    yv = Y[vi]
    key = "logits_tta" if CFG["tta"] else "logits"
    trk = INNER[k][0]                                       # this fold's training images (class frequencies)
    PV = {c: prior_adjust(softmax_np(R[c + (k,)][f"val_{key}"]), trk) for c in CANDS}
    PT = {c: prior_adjust(softmax_np(R[c + (k,)][f"test_{key}"]), trk) for c in CANDS}
    PE = {c: {e: prior_adjust(softmax_np(R[c + (k,)][f"ext__{e}"]), trk) for e in EXT if f"ext__{e}" in R[c + (k,)]}
          for c in CANDS}
    M = {}                                                  # method -> (val P, test P, {ext: P})
    for c in CANDS:
        M[SINGLE_NAMES[c]] = (PV[c], PT[c], PE[c])

    def avg(cs):
        return (np.mean([PV[c] for c in cs], 0), np.mean([PT[c] for c in cs], 0),
                {e: np.mean([PE[c][e] for c in cs], 0) for e in EXT if all(e in PE[c] for c in cs)})

    def stack(cs):
        Xv = np.concatenate([log_p(PV[c]) for c in cs], 1)
        predict, oof = fit_stacker(Xv, yv, GROUP[vi])
        f = lambda Ps: predict(np.concatenate([log_p(p) for p in Ps], 1))
        return (oof, f([PT[c] for c in cs]),
                {e: f([PE[c][e] for c in cs]) for e in EXT if all(e in PE[c] for c in cs)})

    rank = sorted(G_C, key=lambda c: (-macro_f1(yv, PV[c].argmax(1)), -PV[c][np.arange(len(yv)), yv].mean()))
    M[M_BEST] = M[SINGLE_NAMES[rank[0]]]
    M[M_TOP2] = avg(rank[:2])
    M[M_ALLG] = avg(G_C)
    if D_C:
        M[M_ALLGD] = avg(G_C + D_C)
        M[M_STG] = stack(G_C)
        M[M_PROP] = stack(G_C + D_C)
    else:
        M[M_PROP] = stack(G_C)
    CHOICES.append(dict(fold=k, best_single=nice(*rank[0]), top2=" + ".join(nice(*c) for c in rank[:2])))
    for m in METHODS:
        pv, pt, pe = M[m]
        t_m = fit_temperature(log_p(pv), yv)
        TEMPS.append(dict(fold=k, method=m, T=t_m))
        OOF_RAW[m][ti] = pt
        OOF[m][ti] = softmax_np(log_p(pt) / t_m)
        for e, P in pe.items():
            EXT_P[m][e].append(softmax_np(log_p(P) / t_m))
        FOLD_ROWS.append(dict(method=m, fold=k, **compute_metrics(Y[ti], OOF[m][ti])))

FOLD_DF = pd.DataFrame(FOLD_ROWS)
EVAL = np.where(~np.isnan(OOF[M_PROP][:, 0]))[0]           # images with an out-of-fold prediction
YE = Y[EVAL]
POOLED = pd.DataFrame([dict(method=m, **compute_metrics(YE, OOF[m][EVAL])) for m in METHODS]).set_index("method")
print(pd.DataFrame(CHOICES).to_string(index=False))
print(POOLED[["acc", "f1", "auc", "kappa", "ece", "normal_as_glaucoma"]].round(4).to_string())
step("Part 11 done.")

## Part 12: Statistics, per-class tables, shortcut baselines, DR grades, decision prior (`part12_statistics.py`)

Statistics (mean +/- SD, bootstrap CIs, McNemar + Holm, corrected t-test, Friedman), per-class table, **shortcut baselines**
(acquisition metadata only; **image quality only**), **DR recall by severity grade** and the proposed model under three
**decision priors** (natural / square root / balanced).

In [ ]:
# ==============================================================================
# PART 12/16: Statistics, per-class tables, within-source table, shortcut baselines
# Kaggle: paste into a NEW code cell directly below Part 11. Run the parts in order.
# ==============================================================================
def fmt_ms(m, s, pct=True):
    return f"{m * 100:.2f} ± {s * 100:.2f}" if pct else f"{m:.3f} ± {s:.3f}"


step("Part 12: statistics tables ...")
agg = FOLD_DF.groupby("method", sort=False)
MAIN = pd.DataFrame({
    "Model": METHODS,
    "Accuracy (%)": [fmt_ms(agg.get_group(m).acc.mean(), agg.get_group(m).acc.std(ddof=1) if len(FOLDS_DONE) > 1 else 0) for m in METHODS],
    "Bal. Acc. (%)": [fmt_ms(agg.get_group(m).bacc.mean(), agg.get_group(m).bacc.std(ddof=1) if len(FOLDS_DONE) > 1 else 0) for m in METHODS],
    "Macro-P (%)": [fmt_ms(agg.get_group(m).prec.mean(), agg.get_group(m).prec.std(ddof=1) if len(FOLDS_DONE) > 1 else 0) for m in METHODS],
    "Macro-R (%)": [fmt_ms(agg.get_group(m).rec.mean(), agg.get_group(m).rec.std(ddof=1) if len(FOLDS_DONE) > 1 else 0) for m in METHODS],
    "Macro-F1 (%)": [fmt_ms(agg.get_group(m).f1.mean(), agg.get_group(m).f1.std(ddof=1) if len(FOLDS_DONE) > 1 else 0) for m in METHODS],
    "Kappa": [fmt_ms(agg.get_group(m).kappa.mean(), agg.get_group(m).kappa.std(ddof=1) if len(FOLDS_DONE) > 1 else 0, False) for m in METHODS],
    "Macro-AUC": [fmt_ms(agg.get_group(m).auc.mean(), agg.get_group(m).auc.std(ddof=1) if len(FOLDS_DONE) > 1 else 0, False) for m in METHODS],
    "ECE (%)": [f"{POOLED.loc[m, 'ece'] * 100:.2f}" for m in METHODS],
})
print(MAIN.to_string(index=False))
to_latex(MAIN, "tab_main_results", f"{len(FOLDS_DONE)}-fold stratified group cross-validation "
         "(mean ± SD over test folds). ECE is computed on pooled out-of-fold predictions after temperature scaling.",
         "tab:results")

# Cluster bootstrap (resample duplicate clusters, not images) for 95% CIs and paired differences.
g_eval = GROUP[EVAL]
_, g_inv = np.unique(g_eval, return_inverse=True)
n_g = g_inv.max() + 1
rng_b = np.random.default_rng(SEED)
B = CFG["n_boot"]
BOOT = {m: [] for m in METHODS}
for b in pbar(range(B), desc="Bootstrap 95% CIs", unit="resample"):
    w = rng_b.multinomial(n_g, np.full(n_g, 1.0 / n_g))[g_inv].astype(float)
    for m in METHODS:
        r_ = compute_metrics(YE, OOF[m][EVAL], w=w)
        BOOT[m].append((r_["acc"], r_["f1"], r_["auc"]))
BOOT = {m: np.array(v) for m, v in BOOT.items()}
ci = lambda a: (np.nanpercentile(a, 2.5), np.nanpercentile(a, 97.5))

rows, pvals = [], []
corr_pred = {m: OOF[m][EVAL].argmax(1) == YE for m in METHODS}
n_tr = np.mean([len(INNER[k][0]) for k in FOLDS_DONE])
n_te = np.mean([(FOLD == k).sum() for k in FOLDS_DONE])
for m in METHODS:
    a_ci, f_ci, u_ci = ci(BOOT[m][:, 0]), ci(BOOT[m][:, 1]), ci(BOOT[m][:, 2])
    row = {"Model": m, "Acc. [95% CI]": f"{POOLED.loc[m, 'acc'] * 100:.2f} [{a_ci[0] * 100:.2f}, {a_ci[1] * 100:.2f}]",
           "Macro-F1 [95% CI]": f"{POOLED.loc[m, 'f1'] * 100:.2f} [{f_ci[0] * 100:.2f}, {f_ci[1] * 100:.2f}]",
           "Macro-AUC [95% CI]": f"{POOLED.loc[m, 'auc']:.4f} [{u_ci[0]:.4f}, {u_ci[1]:.4f}]"}
    if m != M_PROP:
        b_ = int((corr_pred[M_PROP] & ~corr_pred[m]).sum())
        c_ = int((~corr_pred[M_PROP] & corr_pred[m]).sum())
        p_mc = stats.binomtest(min(b_, c_), b_ + c_, 0.5).pvalue if b_ + c_ > 0 else 1.0
        d = np.array([agg.get_group(M_PROP).set_index("fold").f1[k] - agg.get_group(m).set_index("fold").f1[k]
                      for k in FOLDS_DONE])
        if len(d) > 1 and d.std(ddof=1) > 0:
            t_stat = d.mean() / math.sqrt((1 / len(d) + n_te / n_tr) * d.var(ddof=1))
            p_t = 2 * stats.t.sf(abs(t_stat), len(d) - 1)
        else:
            p_t = float("nan")
        diff_f1 = BOOT[M_PROP][:, 1] - BOOT[m][:, 1]
        row.update({"ΔF1 vs proposed [95% CI]": f"{(POOLED.loc[M_PROP, 'f1'] - POOLED.loc[m, 'f1']) * 100:+.2f} "
                    f"[{np.percentile(diff_f1, 2.5) * 100:+.2f}, {np.percentile(diff_f1, 97.5) * 100:+.2f}]",
                    "McNemar b/c": f"{b_}/{c_}", "p_mcnemar": p_mc, "p_corr_t": p_t})
        pvals.append(p_mc)
    rows.append(row)
STATS = pd.DataFrame(rows)
# Holm-Bonferroni over the McNemar comparisons.
mask = STATS.Model != M_PROP
order = np.argsort(pvals)
holm = np.empty(len(pvals))
running = 0.0
for rank_i, j in enumerate(order):
    running = max(running, min(1.0, (len(pvals) - rank_i) * pvals[j]))
    holm[j] = running
STATS.loc[mask, "p_mcnemar (Holm)"] = holm
for col in ("p_mcnemar", "p_mcnemar (Holm)", "p_corr_t"):
    STATS[col] = STATS[col].map(lambda v: "" if pd.isna(v) else ("<0.001" if v < 0.001 else f"{v:.3f}"))
print(STATS.to_string(index=False))
to_latex(STATS, "tab_statistics", "Pooled out-of-fold performance with cluster-bootstrap 95\\% CIs "
         f"(B={B}) and significance tests against the proposed model (McNemar with Holm correction; "
         "corrected resampled t-test on per-fold macro-F1).", "tab:stats")

sing = [SINGLE_NAMES[c] for c in G_C] + [M_PROP]
if len(FOLDS_DONE) >= 3 and len(sing) >= 3:                 # Friedman needs >= 3 methods
    mat = np.array([[agg.get_group(m).set_index("fold").f1[k] for m in sing] for k in FOLDS_DONE])
    fr = stats.friedmanchisquare(*mat.T)
    ranks = pd.Series(stats.rankdata(-mat, axis=1).mean(0), index=sing).sort_values()
    print(f"Friedman chi2={fr.statistic:.2f}, p={fr.pvalue:.4g}\nAverage rank (1 = best):\n{ranks.round(2)}")
    ranks.rename("avg_rank").to_csv(TABS / "friedman_ranks.csv")

# Per-class report of the proposed model and the Normal-vs-Glaucoma confusion of every method.
pm = POOLED.loc[M_PROP]
P_prop = OOF[M_PROP][EVAL]
PER_CLASS = pd.DataFrame({
    "Class": SHORT,
    "Sensitivity (%)": [f"{pm[f'sens_{s}'] * 100:.2f}" for s in SHORT],
    "Specificity (%)": [f"{pm[f'spec_{s}'] * 100:.2f}" for s in SHORT],
    "F1 (%)": [f"{f1_score(YE == c, P_prop.argmax(1) == c) * 100:.2f}" for c in range(NC)],
    "AUC": [f"{roc_auc_score(YE == c, P_prop[:, c]):.4f}" for c in range(NC)],
    "Support": [int((YE == c).sum()) for c in range(NC)]})
print(PER_CLASS.to_string(index=False))
to_latex(PER_CLASS, "tab_per_class", f"Per-class performance of {M_PROP} (pooled out-of-fold).", "tab:perclass")
NG = pd.DataFrame({"Model": METHODS,
                   "Normal→Glaucoma": [int(POOLED.loc[m, "normal_as_glaucoma"]) for m in METHODS],
                   "Glaucoma→Normal": [int(POOLED.loc[m, "glaucoma_as_normal"]) for m in METHODS],
                   "Glaucoma sens. (%)": [f"{POOLED.loc[m, 'sens_Glaucoma'] * 100:.2f}" for m in METHODS],
                   "Referable sens. (%)": [f"{POOLED.loc[m, 'referable_sens'] * 100:.2f}" for m in METHODS],
                   "Referable spec. (%)": [f"{POOLED.loc[m, 'referable_spec'] * 100:.2f}" for m in METHODS],
                   "Referable AUC": [f"{POOLED.loc[m, 'referable_auc']:.4f}" for m in METHODS]})
print(NG.to_string(index=False))
to_latex(NG, "tab_normal_glaucoma", "Normal--glaucoma confusion and referable-disease screening "
         "(any disease vs. normal; specificity = normal eyes correctly kept as normal), pooled over all test folds.",
         "tab:ng")

# Sensitivity analysis, only when label-conflict images were kept (CFG drop_label_conflicts=False).
LABEL_NOISE = None
if CONFLICT[EVAL].any():
    cf = ~CONFLICT[EVAL]
    LABEL_NOISE = pd.DataFrame([{
        "Model": m,
        "Acc. all (%)": f"{accuracy_score(YE, OOF[m][EVAL].argmax(1)) * 100:.2f}",
        "Acc. conflict-free (%)": f"{accuracy_score(YE[cf], OOF[m][EVAL][cf].argmax(1)) * 100:.2f}",
        "Macro-F1 conflict-free (%)": f"{f1_score(YE[cf], OOF[m][EVAL][cf].argmax(1), average='macro') * 100:.2f}",
        "Acc. label-conflict images (%)": f"{accuracy_score(YE[~cf], OOF[m][EVAL][~cf].argmax(1)) * 100:.2f}"}
        for m in METHODS])
    print(f"\nLabel-conflict sensitivity analysis ({int((~cf).sum())} of {len(cf)} test images have a near-identical "
          "copy with a different label):")
    print(LABEL_NOISE.to_string(index=False))
    to_latex(LABEL_NOISE, "tab_label_conflicts", "Accuracy on all images vs. images without a label conflict "
             "(a near-identical copy filed under another class).", "tab:labelconflicts")

# ---- Within-source evaluation: the same out-of-fold predictions, split by source dataset. Inside one source
# (one camera / pipeline) acquisition cues cannot separate the classes, so these are the honest numbers. ----
src_e = CAM[EVAL]                                  # camera source (ODIR copies count as ODIR)
best_single_m = POOLED.loc[[SINGLE_NAMES[c] for c in G_C], "f1"].idxmax()
show_m = [best_single_m, M_TOP2, M_PROP]
rows_s = []
for s in sorted({s_ for s_ in src_e if (src_e == s_).sum() >= 20}, key=lambda s_: (s_ != "ODIR", s_)):
    msk = src_e == s
    present = np.unique(YE[msk])
    row = {"Source": s, "Images": int(msk.sum()), "Classes": " ".join(ABBR[c] for c in present)}
    for m in show_m:
        pr = OOF[m][EVAL][msk].argmax(1)
        row[f"{m.split(' (')[0]}: Acc. (%)"] = f"{accuracy_score(YE[msk], pr) * 100:.2f}"
        if len(present) > 1:
            row[f"{m.split(' (')[0]}: Macro-F1 (%)"] = \
                f"{f1_score(YE[msk], pr, labels=present, average='macro', zero_division=0) * 100:.2f}"
    rows_s.append(row)
PER_SOURCE = pd.DataFrame(rows_s) if rows_s else pd.DataFrame({"Source": []})
print("\nWithin-source performance (same out-of-fold predictions, split by source):")
print(PER_SOURCE.to_string(index=False))
to_latex(PER_SOURCE, "tab_per_source", "Out-of-fold performance inside each source dataset. Within one source the "
         "classes share camera and processing, so acquisition shortcuts cannot explain the result.", "tab:persource")

# ---- Shortcut baselines: how much of the label is predictable WITHOUT looking at the retina? ----
from sklearn.ensemble import HistGradientBoostingClassifier
_sh = META["shapes"].astype(float)
_cs = META["crop_side"].astype(float) if "crop_side" in META else np.maximum(_sh[:, 0], _sh[:, 1])
X_acq = np.c_[np.log(_sh[:, 1] + 1), np.log(_sh[:, 0] + 1), _sh[:, 1] / np.maximum(_sh[:, 0], 1),
              np.log(FBYTES + 1), FBYTES / np.maximum(_sh[:, 0] * _sh[:, 1], 1),
              np.array([p.lower().endswith(".png") for p in DF.path], float), META["corner_dark"],
              _cs / np.maximum(_sh[:, 1], 1), _cs / np.maximum(_sh[:, 0], 1)]
X_q = np.c_[META["bright"], np.log1p(META["sharp"]), META["white_pad"]]     # global image quality, no lesion detail
pred_acq = np.full(N, -1)
pred_src = np.full(N, -1)
pred_q = np.full(N, -1)
for k in FOLDS_DONE:
    tr_k = np.where((FOLD != k) & (FOLD >= 0))[0]
    te_k = np.where(FOLD == k)[0]
    hgb = HistGradientBoostingClassifier(max_iter=300, random_state=SEED).fit(X_acq[tr_k], Y[tr_k])
    pred_acq[te_k] = hgb.predict(X_acq[te_k])
    pred_q[te_k] = HistGradientBoostingClassifier(max_iter=200, random_state=SEED, class_weight="balanced").fit(
        X_q[tr_k], Y[tr_k]).predict(X_q[te_k])
    maj = pd.Series(Y[tr_k]).groupby(SRC[tr_k]).agg(lambda s: s.value_counts().idxmax())
    pred_src[te_k] = [maj.get(s, np.bincount(Y[tr_k]).argmax()) for s in SRC[te_k]]
SHORTCUT = pd.DataFrame([
    {"Predictor": name_, "Uses retina pixels": uses_,
     "Accuracy (%)": f"{accuracy_score(YE, p_) * 100:.2f}", "Bal. Acc. (%)": f"{balanced_accuracy_score(YE, p_) * 100:.2f}",
     "Macro-F1 (%)": f"{f1_score(YE, p_, average='macro', labels=list(range(NC)), zero_division=0) * 100:.2f}"}
    for name_, uses_, p_ in [
        ("Majority class", "no", np.full(len(YE), np.bincount(Y[VALID]).argmax())),
        ("Source dataset -> its most frequent class", "no", pred_src[EVAL]),
        ("Acquisition metadata only (size, file, format, FOV)", "no", pred_acq[EVAL]),
        ("Image quality only (brightness, sharpness, white border)", "global statistics only", pred_q[EVAL]),
        (M_PROP, "yes", OOF[M_PROP][EVAL].argmax(1))]])
print("\nShortcut baselines (same folds):")
print(SHORTCUT.to_string(index=False))
to_latex(SHORTCUT, "tab_shortcut", "Shortcut baselines that never look at the retina, evaluated with the same "
         "folds. The gap to the proposed model is the part of the performance that needs retinal features.",
         "tab:shortcut")

# ---- DR recall by ICDR severity grade (keywords of the official ODIR-5K table) ----
DR_GRADE = None
if HAS_ANN and "D" in IDX:
    _gr = np.full(N, -1)
    for i in EVAL:
        if Y[i] == IDX["D"] and KW[i]:
            _gr[i] = dr_grade(KW[i])
    _pp = OOF[M_PROP][EVAL].argmax(1)
    names_ = {1: "Mild NPDR", 2: "Moderate NPDR", 3: "Severe NPDR", 4: "Proliferative DR", 0: "DR, grade not given"}
    rows_g = []
    for g_, nm_ in names_.items():
        msk = _gr[EVAL] == g_
        if msk.any():
            rows_g.append({"DR grade": nm_, "Images": int(msk.sum()),
                           "Recall as DR (%)": f"{(_pp[msk] == IDX['D']).mean() * 100:.1f}",
                           "Called normal (%)": f"{(_pp[msk] == NORMAL).mean() * 100:.1f}"})
    if rows_g:
        DR_GRADE = pd.DataFrame(rows_g)
        print("\nDR recall by severity grade (proposed model, pooled out-of-fold):")
        print(DR_GRADE.to_string(index=False))
        to_latex(DR_GRADE, "tab_dr_grade", "Recall of the DR class by ICDR grade (from the ODIR-5K diagnostic keywords).",
                 "tab:drgrade")

# ---- Decision prior (sensitivity analysis, post-hoc on the proposed model's out-of-fold probabilities). The reported
# headline uses the pre-specified CFG["decision_prior_power"]; the other rows show the accuracy / balanced-accuracy trade-off.
_cnt_used = np.maximum(np.bincount(Y[EVAL], minlength=NC).astype(float), 1)
rows_p = []
for e_, nm_ in ((1.0, "natural (class frequencies)"), (0.5, "square root"), (0.0, "balanced (all classes equal)")):
    Q_ = OOF[M_PROP][EVAL] * _cnt_used ** (e_ - float(CFG["decision_prior_power"]))
    r_ = compute_metrics(YE, Q_ / Q_.sum(1, keepdims=True))
    rows_p.append({"Decision prior": nm_ + (" [pre-specified]" if abs(e_ - CFG["decision_prior_power"]) < 1e-9 else ""),
                   "Accuracy (%)": f"{r_['acc'] * 100:.2f}", "Bal. acc. (%)": f"{r_['bacc'] * 100:.2f}",
                   "Macro-F1 (%)": f"{r_['f1'] * 100:.2f}",
                   **{f"Sens. {s} (%)": f"{r_[f'sens_{s}'] * 100:.1f}" for s in SHORT}})
PRIOR_TAB = pd.DataFrame(rows_p)
print("\nDecision prior of the proposed model (post-hoc; headline = pre-specified row):")
print(PRIOR_TAB.to_string(index=False))
to_latex(PRIOR_TAB, "tab_decision_prior", f"{M_PROP} under three class priors (post-hoc logit adjustment of the same "
         "out-of-fold probabilities). The pre-specified prior is used for all other tables.", "tab:prior")
step("Part 12 done.")

## Part 13: Result figures: learning curves, confusion, ROC, calibration/referral, benchmark (`part13_figures.py`)

Result figures (learning curves, confusion matrices, ROC, calibration/referral, benchmark, complexity).

In [ ]:
# ==============================================================================
# PART 13/16: Result figures: learning curves, confusion, ROC, calibration/referral, benchmark
# Kaggle: paste into a NEW code cell directly below Part 12. Run the parts in order.
# ==============================================================================
# (a) Real learning curves of every global model on the first finished fold (no hand-typed values).
step("Part 13: drawing result figures ...")
k0 = FOLDS_DONE[0]
nG = len(G_C)
ncol = 4
nrow = math.ceil(nG / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(PAGE_W, 1.55 * nrow), sharey=True, squeeze=False)
for a_i, c in enumerate(G_C):
    ax = axes.flat[a_i]
    h = pd.DataFrame(json.loads(str(R[c + (k0,)]["hist"])))
    ax.plot(h.epoch, h.train_loss, color=C1, label="Train (MixUp + LS)")
    ax.plot(h.epoch, h.val_loss, color=C2, label="Validation (LS)")
    be = int(R[c + (k0,)]["best_epoch"])
    ax.axvline(be, color=MUTED, lw=0.8, ls=":")
    ax.set_title(nice(*c))
    ax.set_xlabel("Epoch")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    if a_i % ncol == 0:
        ax.set_ylabel("Loss")
for ax in list(axes.flat)[nG:]:
    ax.set_visible(False)
fig.legend([plt.Line2D([], [], color=C1), plt.Line2D([], [], color=C2), plt.Line2D([], [], color=MUTED, lw=0.8, ls=":")],
           ["Train (MixUp + label smoothing)", "Validation (label smoothing)", "Kept checkpoint (lowest val. NLL)"],
           loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.0 + 0.12 / nrow))
fig.tight_layout(pad=0.4)
save_fig(fig, "fig_learning_curves")

# (b) Confusion matrices: proposed vs. best single (pooled out-of-fold).
best_single_m = POOLED.loc[[SINGLE_NAMES[c] for c in G_C], "f1"].idxmax()
fig, axes = plt.subplots(1, 2, figsize=(PAGE_W, 2.9 if NC <= 4 else 3.6))
fs_cm = 7 if NC <= 4 else 5.2
for ax, m in zip(axes, [best_single_m, M_PROP]):
    cm = confusion_matrix(YE, OOF[m][EVAL].argmax(1), labels=list(range(NC)))
    pct = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(pct, cmap=BLUES, vmin=0, vmax=1)
    ax.grid(False)
    for i in range(NC):
        for j in range(NC):
            ax.text(j, i, f"{cm[i, j]}\n{pct[i, j] * 100:.1f}%", ha="center", va="center", fontsize=fs_cm,
                    color="white" if pct[i, j] > 0.55 else INK)
    tick = SHORT if NC <= 4 else ABBR
    ax.set_xticks(range(NC), tick); ax.set_yticks(range(NC), tick)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{m}\nacc {POOLED.loc[m, 'acc'] * 100:.2f}%, macro-F1 {POOLED.loc[m, 'f1'] * 100:.2f}%")
fig.tight_layout()
save_fig(fig, "fig_confusion")

# (c) One-vs-rest ROC, small multiples per class: proposed vs. best single vs. top-2 soft vote.
cmp_methods = [(M_PROP, C1, "-"), (best_single_m, C2, "--"), (M_TOP2, C3, ":")]
ncol_r = min(NC, 4)
nrow_r = math.ceil(NC / ncol_r)
fig, axes = plt.subplots(nrow_r, ncol_r, figsize=(PAGE_W, 2.0 * nrow_r + 0.2), sharey=True, squeeze=False)
for c, ax in enumerate(axes.flat[:NC]):
    for m, col, ls in cmp_methods:
        fpr, tpr, _ = roc_curve(YE == c, OOF[m][EVAL][:, c])
        ax.plot(fpr, tpr, color=col, ls=ls, lw=1.4,
                label=f"AUC {roc_auc_score(YE == c, OOF[m][EVAL][:, c]):.3f}")
    ax.plot([0, 1], [0, 1], color=AXIS, lw=0.8)
    ax.set_title(SHORT[c]); ax.set_xlabel("1 - specificity")
    ax.set_xlim(-0.01, 1); ax.set_ylim(0, 1.01); ax.set_xticks([0, 0.5, 1])
    ax.legend(loc="lower right", fontsize=6, handlelength=1.8)
for ax in axes.flat[NC:]:
    ax.set_visible(False)
for r in range(nrow_r):
    axes[r, 0].set_ylabel("Sensitivity")
fig.legend([plt.Line2D([], [], color=col, ls=ls, lw=1.4) for _, col, ls in cmp_methods],
           [m for m, _, _ in cmp_methods], loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.0 + 0.08 / nrow_r),
           fontsize=7)
fig.tight_layout(pad=0.4)
save_fig(fig, "fig_roc")

# (d) Reliability diagram (before / after temperature scaling) and (e) risk-coverage referral curve.
fig, axes = plt.subplots(1, 2, figsize=(PAGE_W, 2.5))
ax = axes[0]
for P, col, lab in [(OOF_RAW[M_PROP][EVAL], C2, "Before scaling"), (OOF[M_PROP][EVAL], C1, "After scaling")]:
    P = P / P.sum(1, keepdims=True)
    conf, ok_ = P.max(1), P.argmax(1) == YE
    edges = np.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m_ = (conf > lo) & (conf <= hi)
        if m_.sum() >= 5:
            xs.append(conf[m_].mean()); ys.append(ok_[m_].mean())
    ax.plot(xs, ys, marker="o", ms=3.5, color=col, label=f"{lab} (ECE {ece_score(P, YE) * 100:.2f}%)")
ax.plot([0, 1], [0, 1], color=AXIS, lw=0.8)
ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.set_title(f"(a) Calibration, {M_PROP}")
ax.legend(loc="upper left")
ax = axes[1]
REFER = []
for m, col, ls in cmp_methods:
    P = OOF[m][EVAL]
    order_ = np.argsort(-P.max(1))
    ok_ = (P.argmax(1) == YE)[order_]
    cov = np.arange(1, len(ok_) + 1) / len(ok_)
    acc_cov = np.cumsum(ok_) / np.arange(1, len(ok_) + 1)
    shown = cov >= 0.5                          # below 50% coverage the curve rests on too few images
    ax.plot(cov[shown] * 100, acc_cov[shown] * 100, color=col, ls=ls, label=m.split(" (")[0][:28])
    REFER.append({"Model": m, **{f"Acc@{round(c_ * 100)}% cov.": f"{acc_cov[max(0, int(c_ * len(ok_)) - 1)] * 100:.2f}"
                                 for c_ in (1.0, 0.95, 0.9, 0.8)},
                  "AURC (x100)": f"{np.mean(1 - acc_cov) * 100:.2f}"})
ax.set_xlabel("Coverage (% of images decided automatically)"); ax.set_ylabel("Accuracy (%)")
ax.set_title("(b) Uncertainty-based referral"); ax.set_xlim(101, 49); ax.legend(loc="lower right")
fig.tight_layout()
save_fig(fig, "fig_calibration_referral")
REFER = pd.DataFrame(REFER)
print(REFER.to_string(index=False))
to_latex(REFER, "tab_referral", "Accuracy when the least-confident cases are referred to an ophthalmologist "
         "(coverage = share of images decided automatically).", "tab:referral")

# (f) Benchmark: macro-F1 mean +/- SD across folds, proposed highlighted.
f1m = agg.f1.mean().reindex(METHODS)
f1s = (agg.f1.std(ddof=1) if len(FOLDS_DONE) > 1 else agg.f1.mean() * 0).reindex(METHODS).fillna(0)
order_m = f1m.sort_values().index
fig, ax = plt.subplots(figsize=(COL_W * 1.25, 0.2 * len(METHODS) + 0.6))
cols = [C1 if m == M_PROP else "#9ec5f4" for m in order_m]
ax.barh(range(len(order_m)), f1m[order_m] * 100, xerr=f1s[order_m] * 100, color=cols, height=0.7,
        error_kw=dict(ecolor=INK2, lw=0.8, capsize=1.5))
ax.set_yticks(range(len(order_m)), order_m)
lo = max(0, (f1m.min() - 3 * f1s.max()) * 100 - 2)
hi = min(100, (f1m + f1s).max() * 100)
span = hi - lo
ax.set_xlim(lo, hi + span * 0.16)
ax.set_xticks([t for t in ax.get_xticks() if lo <= t <= hi + span * 0.05])
for i, m in enumerate(order_m):
    ax.text(f1m[m] * 100 + f1s[m] * 100 + span * 0.02, i, f"{f1m[m] * 100:.2f}", va="center", fontsize=6.5,
            color=INK2)
ax.set_xlabel("Macro-F1 (%) — mean ± SD over test folds")
ax.grid(axis="y", visible=False)
fig.tight_layout()
save_fig(fig, "fig_benchmark")

# (g) Accuracy vs. complexity (one colour; labels identify the models).
fig, ax = plt.subplots(figsize=(COL_W, 2.4))
cp_ = CPLX.set_index("Model")
for m in [SINGLE_NAMES[c] for c in CANDS]:
    if m in cp_.index:
        ax.scatter(cp_.loc[m, "Params (M)"], f1m[m] * 100, s=22, color=C1 if "[disc]" not in m else C2,
                   edgecolor="white", linewidth=0.8, zorder=3)
        ax.annotate(m, (cp_.loc[m, "Params (M)"], f1m[m] * 100), fontsize=5.8, color=INK2,
                    xytext=(3, 2), textcoords="offset points")
ax.set_xscale("log"); ax.set_xlabel("Parameters (M, log scale)"); ax.set_ylabel("Macro-F1 (%)")
ax.set_title("Single models: macro-F1 vs. size")
fig.tight_layout()
save_fig(fig, "fig_complexity")
step("Part 13 done.")

## Part 14: Protocol comparison (+ external sets, if configured) (`part14_external_results.py`)

**Protocol comparison** (`tab_protocol`): the same model under patient-level grouped CV (this work) and under a random
image-level split (patients ignored, as in many published papers), with the share of test images whose other eye was in
training. External folder sets, if any. The Version 2 source-shift test is off (ODIR-5K is one source).

In [ ]:
# ==============================================================================
# PART 14/16: Source-shift test results (+ external sets, if configured)
# Kaggle: paste into a NEW code cell directly below Part 13. Run the parts in order.
# ==============================================================================
step("Part 14: protocol comparison and external validation ...")
EXT_TABLE = []
for e, info in EXT.items():
    keep = info["keep"]
    ye = info["y"][keep]
    for m in METHODS:
        if not EXT_P[m][e]:
            continue
        P = np.mean(EXT_P[m][e], 0)[keep]
        r_ = compute_metrics(ye, P)
        EXT_TABLE.append({"Dataset": e, "Model": m, "N": int(keep.sum()), "Accuracy (%)": f"{r_['acc'] * 100:.2f}",
                          "Macro-F1 (%)": f"{r_['f1'] * 100:.2f}", "Macro-AUC": f"{r_['auc']:.4f}",
                          **{f"Sens. {s} (%)": f"{r_[f'sens_{s}'] * 100:.2f}" for s in SHORT}})
if EXT_TABLE:
    EXT_TABLE = pd.DataFrame(EXT_TABLE)
    print(EXT_TABLE.to_string(index=False))
    to_latex(EXT_TABLE, "tab_external", "External validation (fold models averaged; images that duplicate "
             "internal data removed).", "tab:external")

# Source-shift test: the model never saw an ODIR image of any disease. Reference = the same architecture in
# cross-validation, on the same test images (there, ODIR disease images of OTHER patients were in training).
SS_TABLE, SS_SUM, SS_MODELS = None, None, None
_kk = "logits_tta" if CFG["tta"] else "logits"


def ss_eval(name):
    """Source-shift predictions of one model + CV predictions of the same architecture on the same images."""
    c = ("global", name)
    f = RUNS_SS / f"{c[0]}__{c[1]}__ss.npz"
    if SS_SPLIT is None or not f.exists():
        return None
    z = np.load(f, allow_pickle=False)
    ti = z["test_idx"]
    y = Y[ti]
    pr = z[f"test_{_kk}"].argmax(1)                  # temperature scaling does not change the arg-max
    ref_name = SINGLE_NAMES.get(c)
    have_ref = ref_name is not None and not np.isnan(OOF[ref_name][ti, 0]).any()
    pr_ref = OOF[ref_name][ti].argmax(1) if have_ref else None
    present = np.unique(y)
    dis = y != NORMAL
    f1_ = lambda p: float(f1_score(y, p, labels=present, average="macro", zero_division=0))
    return dict(c=c, ti=ti, y=y, pr=pr, pr_ref=pr_ref, have_ref=have_ref, present=present,
                acc=float(accuracy_score(y, pr)), f1=f1_(pr), bacc=float(balanced_accuracy_score(y, pr)),
                called_normal=float((pr[dis] == NORMAL).mean()) if dis.any() else float("nan"),
                ref_acc=float(accuracy_score(y, pr_ref)) if have_ref else None,
                ref_f1=f1_(pr_ref) if have_ref else None,
                ref_called_normal=float((pr_ref[dis] == NORMAL).mean()) if have_ref and dis.any() else None)


SS_RES = {}
for _m in CFG["source_shift_models"] if CFG["source_shift_test"] else []:
    _r = ss_eval(_m)
    if _r is not None:
        SS_RES[_m] = _r

if SS_RES:
    # ---- detailed per-class table + figure for the first (primary) source-shift model ----
    r0 = next(iter(SS_RES.values()))
    ti_ss, y_ss, pr_ss, pr_ref, have_ref, present = (r0[k_] for k_ in ("ti", "y", "pr", "pr_ref", "have_ref",
                                                                        "present"))
    rows_ = []
    for c in present:
        msk = y_ss == c
        r_ = {"Class": SHORT[c], "Test images": int(msk.sum())}
        if have_ref:
            r_["Recall, CV (%)"] = f"{(pr_ref[msk] == c).mean() * 100:.1f}"
        r_["Recall, source-shift (%)"] = f"{(pr_ss[msk] == c).mean() * 100:.1f}"
        if c != NORMAL:
            if have_ref:
                r_["Called normal, CV (%)"] = f"{(pr_ref[msk] == NORMAL).mean() * 100:.1f}"
            r_["Called normal, source-shift (%)"] = f"{(pr_ss[msk] == NORMAL).mean() * 100:.1f}"
        rows_.append(r_)
    SS_TABLE = pd.DataFrame(rows_)
    SS_SUM = dict(n_test=int(len(ti_ss)), n_train=int(len(SS_SPLIT[0])), model=nice(*r0["c"]),
                  acc=r0["acc"], f1=r0["f1"], bacc=r0["bacc"], ref_acc=r0["ref_acc"], ref_f1=r0["ref_f1"])
    print(f"Source-shift test ({SS_SUM['model']}, {SS_SUM['n_test']} held-out ODIR images): accuracy "
          f"{SS_SUM['acc'] * 100:.2f}%, macro-F1 {SS_SUM['f1'] * 100:.2f}%" +
          (f"  | same images in CV: accuracy {SS_SUM['ref_acc'] * 100:.2f}%, macro-F1 {SS_SUM['ref_f1'] * 100:.2f}%"
           if have_ref else ""))
    print(SS_TABLE.to_string(index=False))
    to_latex(SS_TABLE, "tab_source_shift", f"Source-shift test with {SS_SUM['model']}: all ODIR images of the "
             "disease classes (and 20\\% of ODIR normal patients) are held out, so training contains ODIR images only "
             "of normal eyes. CV = the same architecture in cross-validation on the same images. 'Called normal' = "
             "diseased eyes predicted as normal.", "tab:sourceshift")
    # Figure: per-class recall, CV vs source-shift (two series -> legend + colour; values labelled at bar ends).
    labels_ = [SHORT[c] for c in present]
    rec_ss = np.array([(pr_ss[y_ss == c] == c).mean() * 100 for c in present])
    rec_cv = np.array([(pr_ref[y_ss == c] == c).mean() * 100 for c in present]) if have_ref else None
    fig, ax = plt.subplots(figsize=(COL_W, 0.42 * len(present) + 0.7))
    yy = np.arange(len(present))
    hgt = 0.36
    if have_ref:
        ax.barh(yy - hgt / 2 - 0.02, rec_cv, height=hgt, color=C1, label="Cross-validation (ODIR disease seen)")
    ax.barh(yy + (hgt / 2 + 0.02 if have_ref else 0), rec_ss, height=hgt, color=C2,
            label="Source-shift (ODIR disease never seen)")
    for i in range(len(present)):
        if have_ref:
            ax.text(rec_cv[i] + 1, yy[i] - hgt / 2 - 0.02, f"{rec_cv[i]:.0f}", va="center", fontsize=6, color=INK2)
        ax.text(rec_ss[i] + 1, yy[i] + (hgt / 2 + 0.02 if have_ref else 0), f"{rec_ss[i]:.0f}", va="center",
                fontsize=6, color=INK2)
    ax.set_yticks(yy, labels_); ax.invert_yaxis()
    ax.set_xlim(0, 112); ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xlabel("Recall on held-out ODIR images (%)")
    ax.grid(axis="y", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.28 if len(present) > 3 else -0.4), ncol=1, fontsize=6.5)
    fig.tight_layout()
    save_fig(fig, "fig_source_shift")

    # ---- all source-shift models side by side (supervised vs. self-supervised pre-training) ----
    pct = lambda v: "" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v * 100:.1f}"
    SS_MODELS = pd.DataFrame([{
        "Model": nice(*r["c"]), "Test images": int(len(r["ti"])),
        "Macro-F1, CV (%)": pct(r["ref_f1"]), "Macro-F1, source-shift (%)": pct(r["f1"]),
        "Bal. acc., source-shift (%)": pct(r["bacc"]),
        "Disease called normal, CV (%)": pct(r["ref_called_normal"]),
        "Disease called normal, source-shift (%)": pct(r["called_normal"])} for r in SS_RES.values()])
    print("\nSource-shift test, all models:")
    print(SS_MODELS.to_string(index=False))
    to_latex(SS_MODELS, "tab_source_shift_models", "Source-shift test for each model (training contains no ODIR "
             "image of any disease; test = held-out ODIR patients). CV = the same architecture in cross-validation "
             "on the same images. A smaller drop from CV to source-shift = less reliance on the image source.",
             "tab:sourceshiftmodels")
    if len(SS_RES) > 1:
        # Grouped bars: per-class recall under source shift, one series per model (fixed colour order C1, C2, C3,
        # legend + value labels at the bar ends, 2 px gap between adjacent bars).
        cols_ = [C1, C2, C3]
        res_ = list(SS_RES.values())[:3]
        n_s = len(res_)
        slot = 0.8 / n_s
        fig, ax = plt.subplots(figsize=(COL_W, 0.55 * len(present) + 0.9))
        for j, r in enumerate(res_):
            rec_j = np.array([(r["pr"][r["y"] == c] == c).mean() * 100 for c in present])
            yj = yy - 0.4 + slot * (j + 0.5)
            ax.barh(yj, rec_j, height=slot * 0.88, color=cols_[j],
                    label=f"{nice(*r['c'])} (macro-F1 {r['f1'] * 100:.1f}%)")
            for i in range(len(present)):
                ax.text(rec_j[i] + 1, yj[i], f"{rec_j[i]:.0f}", va="center", fontsize=5.5, color=INK2)
        ax.set_yticks(yy, labels_); ax.invert_yaxis()
        ax.set_xlim(0, 112); ax.set_xticks([0, 25, 50, 75, 100])
        ax.set_xlabel("Source-shift recall on held-out ODIR images (%)")
        ax.grid(axis="y", visible=False)
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=1, fontsize=6.5)
        fig.tight_layout()
        save_fig(fig, "fig_source_shift_models")
elif SS_SPLIT is not None:
    print("Source-shift runs not finished yet (Part 9).")
if not len(EXT_TABLE):
    print("No external folder set configured.")
PROTO = None
if NAIVE_SPLIT is not None:
    rows_pr = []
    for _m in CFG["naive_split_models"]:
        c_ = ("global", _m)
        f_ = RUNS_SS / f"global__{_m}__naive.npz"
        if not f_.exists() or c_ not in SINGLE_NAMES:
            continue
        z_ = np.load(f_, allow_pickle=False)
        ti_ = z_["test_idx"]
        r_n = compute_metrics(Y[ti_], prior_adjust(softmax_np(z_[f"test_{_kk}"]), NAIVE_SPLIT[0]))
        r_c = POOLED.loc[SINGLE_NAMES[c_]]
        for prot, n_, lk, r_ in [("Patient-level grouped CV (this work)", len(EVAL), 0.0, r_c),
                                 ("Random image-level split (common in literature)", len(ti_), NAIVE_LEAK, r_n)]:
            rows_pr.append({"Model": nice(*c_), "Protocol": prot, "Test images": int(n_),
                            "Test images with the other eye in training (%)": f"{lk * 100:.1f}",
                            "Accuracy (%)": f"{r_['acc'] * 100:.2f}", "Bal. acc. (%)": f"{r_['bacc'] * 100:.2f}",
                            "Macro-F1 (%)": f"{r_['f1'] * 100:.2f}", "Macro-AUC": f"{r_['auc']:.4f}"})
    if rows_pr:
        PROTO = pd.DataFrame(rows_pr)
        print("\nProtocol comparison (same model, same cleaned images, same decision prior):")
        print(PROTO.to_string(index=False))
        to_latex(PROTO, "tab_protocol", "Effect of the evaluation protocol. Patient-level grouped cross-validation (this "
                 "work) never has the other eye of a test patient in training; a random image-level split, as used by many "
                 "published fundus studies, does. The difference is the optimism of the image-level protocol.",
                 "tab:protocol")
    else:
        print("Protocol-comparison run not finished yet (Part 9).")
step("Part 14 done.")

## Part 15: Grad-CAM: qualitative figure + quantitative XAI metrics (`part15_xai.py`)

Grad-CAM / Grad-CAM++ / LayerCAM with deletion/insertion, drop/increase, optic-disc pointing game.

In [ ]:
# ==============================================================================
# PART 15/16: Grad-CAM: qualitative figure + quantitative XAI metrics
# Kaggle: paste into a NEW code cell directly below Part 14. Run the parts in order.
# ==============================================================================
import importlib.util, subprocess, sys
if importlib.util.find_spec("pytorch_grad_cam") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "grad-cam"], check=False)

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, LayerCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

FX = CFG["save_weights_fold"]


def load_trained(view, name):
    m, _, layer = build_model(name, pretrained=False, size=in_size(view, name))
    sd = torch.load(WTS / f"{view}__{name}.pt", map_location="cpu")
    m.load_state_dict({kk: (vv.float() if vv.is_floating_point() else vv) for kk, vv in sd.items()})
    return m.to(DEVICE).eval(), layer


class ProbAverage(nn.Module):
    def __init__(self, members):
        super().__init__()
        self.members = nn.ModuleList(members)

    def forward(self, x):
        p = torch.stack([F.softmax(m(x).float(), 1) for m in self.members]).mean(0)
        return torch.log(p.clamp_min(1e-8))


step("Part 15: Grad-CAM explanations and XAI metrics (~10-20 min on GPU) ...")
cam_members = [c for c in G_C if c[1] not in NO_CAM and (WTS / f"{c[0]}__{c[1]}.pt").exists()]
disc_cnn = [c for c in D_C if c[1] not in NO_CAM and (WTS / f"{c[0]}__{c[1]}.pt").exists()]
XAI_OK = len(cam_members) > 0 and FX in FOLDS_DONE
if not XAI_OK:
    print("Grad-CAM skipped: no saved CNN weights for the XAI fold yet.")

if XAI_OK:
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    loaded = [load_trained(*c) for c in cam_members]
    ens = ProbAverage([m for m, _ in loaded]).eval()
    ens_layers = [l for _, l in loaded]
    best_cnn = max(cam_members, key=lambda c: f1m[SINGLE_NAMES[c]])
    best_model, best_layer = loaded[cam_members.index(best_cnn)]
    XAI_MODELS = {f"Ensemble of {len(cam_members)} CNNs": (ens, ens_layers)} if len(cam_members) > 1 else {}
    XAI_MODELS[nice(*best_cnn)] = (best_model, [best_layer])
    CAM_METHODS = {"Grad-CAM": GradCAM, "Grad-CAM++": GradCAMPlusPlus, "LayerCAM": LayerCAM}
    norm_t = T.Compose([T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])
    te_fx = np.where(FOLD == FX)[0]
    rng_x = np.random.default_rng(SEED)
    per_cls = max(1, CFG["xai_n"] // NC)
    xai_idx = np.concatenate([rng_x.choice(te_fx[Y[te_fx] == c], size=min(per_cls, int((Y[te_fx] == c).sum())),
                                           replace=False) for c in range(NC) if (Y[te_fx] == c).any()])

    def batch_tensor(idx, npy=GPATH):
        arr = np.load(npy, mmap_mode="r")
        x = torch.stack([norm_t(torch.from_numpy(np.ascontiguousarray(arr[i])).permute(2, 0, 1)) for i in idx])
        return x.to(DEVICE)

    def denorm(x):
        return (x.cpu() * torch.tensor(STD).view(1, 3, 1, 1) + torch.tensor(MEAN).view(1, 3, 1, 1)).clamp(0, 1)

    @torch.no_grad()
    def probs(model, x):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            return F.softmax(model(x).float(), 1)

    @torch.no_grad()
    def del_ins(model, x, cams, target, steps=20):
        Bn, _, H, W = x.shape
        order_ = torch.from_numpy(np.argsort(-cams.reshape(Bn, -1), axis=1).copy()).to(DEVICE)
        rank_ = torch.empty_like(order_)
        rank_.scatter_(1, order_, torch.arange(H * W, device=DEVICE).expand(Bn, -1).contiguous())
        blurred = T.functional.gaussian_blur(x, kernel_size=[51, 51], sigma=[15.0, 15.0])
        zero = torch.zeros_like(x)                         # dataset mean after normalisation
        dl_s, in_s = [], []
        tg = torch.as_tensor(target, device=DEVICE)
        for s in range(steps + 1):
            m_ = (rank_ < int(round(H * W * s / steps))).view(Bn, 1, H, W)
            dl_s.append(probs(model, torch.where(m_, zero, x))[torch.arange(Bn), tg].cpu().numpy())
            in_s.append(probs(model, torch.where(m_, x, blurred))[torch.arange(Bn), tg].cpu().numpy())
        dl_s, in_s = np.stack(dl_s, 1), np.stack(in_s, 1)
        auc_ = lambda a: ((a[:, :-1] + a[:, 1:]) / 2).mean(1)
        return auc_(dl_s), auc_(in_s), dl_s, in_s

    XAI_ROWS, CURVES = [], {}
    disc_box = META["disc"]
    for mname, (model, layers) in XAI_MODELS.items():
        for cname, cls_ in list(CAM_METHODS.items()) + [("Random", None)]:
            dels, inss, drops, incs, energy, point, cur_d, cur_i, glau = [], [], [], [], [], [], [], [], []
            camobj = cls_(model=model, target_layers=layers) if cls_ else None
            xai_bs = 2
            for s in pbar(range(0, len(xai_idx), xai_bs), desc=f"XAI {mname} | {cname}", unit="batch"):
                idx = xai_idx[s:s + xai_bs]
                x = batch_tensor(idx)
                p0 = probs(model, x)
                tgt = p0.argmax(1).cpu().numpy()
                if camobj is not None:
                    try:
                        cams = camobj(input_tensor=x, targets=[ClassifierOutputTarget(int(t)) for t in tgt])
                    except torch.cuda.OutOfMemoryError:
                        gc.collect()
                        if DEVICE.type == "cuda":
                            torch.cuda.empty_cache()
                        cams = np.concatenate([camobj(input_tensor=x[bi:bi + 1],
                                                      targets=[ClassifierOutputTarget(int(tgt[bi]))])
                                               for bi in range(len(x))], 0)
                else:
                    cams = rng_x.random((len(idx),) + tuple(x.shape[-2:])).astype(np.float32)
                d_, i_, dc_, ic_ = del_ins(model, x, cams, tgt)
                dels += list(d_); inss += list(i_); cur_d.append(dc_); cur_i.append(ic_)
                expl = ((denorm(x) * torch.from_numpy(cams)[:, None]) - torch.tensor(MEAN).view(1, 3, 1, 1)) \
                    / torch.tensor(STD).view(1, 3, 1, 1)
                p1 = probs(model, expl.to(DEVICE))
                y0 = p0[torch.arange(len(idx)), torch.as_tensor(tgt, device=DEVICE)].cpu().numpy()
                y1 = p1[torch.arange(len(idx)), torch.as_tensor(tgt, device=DEVICE)].cpu().numpy()
                drops += list(np.maximum(0, y0 - y1) / np.maximum(y0, 1e-6)); incs += list(y1 > y0)
                Hc = cams.shape[-1]
                for b, i in enumerate(idx):
                    if Y[i] != GLAUCOMA:
                        continue
                    cx, cy, _ = disc_box[i]
                    half = CFG["disc_frac"] * Hc / 2
                    x0, x1 = int(max(0, cx * Hc - half)), int(min(Hc, cx * Hc + half))
                    y0_, y1_ = int(max(0, cy * Hc - half)), int(min(Hc, cy * Hc + half))
                    cm_ = cams[b]
                    frac_area = (x1 - x0) * (y1_ - y0_) / (Hc * Hc)
                    energy.append((cm_[y0_:y1_, x0:x1].sum() / max(cm_.sum(), 1e-8)) / max(frac_area, 1e-8))
                    py, px = np.unravel_index(int(np.argmax(cm_)), cm_.shape)
                    point.append(x0 <= px < x1 and y0_ <= py < y1_)
            if camobj is not None:
                camobj.activations_and_grads.release()
                del camobj
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            CURVES[(mname, cname)] = (np.concatenate(cur_d).mean(0), np.concatenate(cur_i).mean(0))
            XAI_ROWS.append({"Model": mname, "Method": cname, "Deletion AUC ↓": f"{np.mean(dels):.3f}",
                             "Insertion AUC ↑": f"{np.mean(inss):.3f}", "Avg. Drop (%) ↓": f"{np.mean(drops) * 100:.2f}",
                             "Increase (%) ↑": f"{np.mean(incs) * 100:.2f}",
                             "Disc enrichment (glaucoma) ↑": f"{np.mean(energy):.2f}" if energy else "-",
                             "Pointing game (glaucoma, %) ↑": f"{np.mean(point) * 100:.1f}" if point else "-"})
    XAI_DF = pd.DataFrame(XAI_ROWS)
    print(XAI_DF.to_string(index=False))
    to_latex(XAI_DF, "tab_xai", f"Quantitative explanation quality on {len(xai_idx)} stratified test images of "
             f"fold {FX}. Disc enrichment = share of saliency inside the optic-disc ROI divided by the ROI's area share "
             "(1.0 = chance).", "tab:xai")

    mname0 = next(iter(XAI_MODELS))
    fig, axes = plt.subplots(1, 2, figsize=(PAGE_W * 0.75, 2.2))
    xs = np.linspace(0, 100, 21)
    for (cname, col, ls) in [("Grad-CAM", C1, "-"), ("Grad-CAM++", C2, "--"), ("LayerCAM", C3, "-."),
                              ("Random", MUTED, ":")]:
        d_c, i_c = CURVES[(mname0, cname)]
        axes[0].plot(xs, d_c, color=col, ls=ls, label=cname)
        axes[1].plot(xs, i_c, color=col, ls=ls, label=cname)
    axes[0].set_title("(a) Deletion (lower AUC = more faithful)")
    axes[1].set_title("(b) Insertion (higher AUC = more faithful)")
    for ax in axes:
        ax.set_xlabel("Pixels removed / inserted (%)"); ax.set_ylabel("Class probability")
    axes[1].legend(loc="lower right")
    fig.tight_layout()
    save_fig(fig, "fig_xai_faithfulness")

    # Qualitative figure: randomly chosen correct cases per class + one Normal->Glaucoma error, if any.
    P_fx = OOF[M_PROP]
    pred_fx = np.nan_to_num(P_fx).argmax(1)
    rows_q = []
    for c in range(NC):
        cand = te_fx[(Y[te_fx] == c) & (pred_fx[te_fx] == c)]
        if len(cand):
            rows_q.append((int(rng_x.choice(cand)), "correct"))
    err = te_fx[(Y[te_fx] == NORMAL) & (pred_fx[te_fx] == GLAUCOMA)]
    if len(err):
        rows_q.append((int(rng_x.choice(err)), "error"))
    disc_model = load_trained(*disc_cnn[0]) if disc_cnn else None
    ncols = 5 if disc_model else 3
    rh = min(1.45, 8.8 / len(rows_q))                 # 7 classes + 1 error row must fit on one page
    fig, axes = plt.subplots(len(rows_q), ncols, figsize=(ncols * rh * 0.95 + 0.45, rh * len(rows_q)), squeeze=False)
    _who = "ensemble" if len(cam_members) > 1 else nice(*best_cnn)
    titles = ["Input (CLAHE)", f"Grad-CAM ({_who})", f"Grad-CAM++ ({_who})", "Disc ROI", "Grad-CAM (disc expert)"]
    with GradCAM(model=ens, target_layers=ens_layers) as gc_e, GradCAMPlusPlus(model=ens, target_layers=ens_layers) as gpp_e:
        for r, (i, kind) in enumerate(rows_q):
            x = batch_tensor([i])
            p_ = P_fx[i]
            tgt = int(np.argmax(p_))
            img = denorm(x)[0].permute(1, 2, 0).numpy()
            panels = [img,
                      show_cam_on_image(img, gc_e(input_tensor=x, targets=[ClassifierOutputTarget(tgt)])[0], use_rgb=True,
                                        colormap=cv2.COLORMAP_INFERNO, image_weight=0.55),
                      show_cam_on_image(img, gpp_e(input_tensor=x, targets=[ClassifierOutputTarget(tgt)])[0], use_rgb=True,
                                        colormap=cv2.COLORMAP_INFERNO, image_weight=0.55)]
            if disc_model:
                xd = batch_tensor([i], DPATH)
                imgd = denorm(xd)[0].permute(1, 2, 0).numpy()
                with GradCAM(model=disc_model[0], target_layers=[disc_model[1]]) as gc_d:
                    camd = gc_d(input_tensor=xd, targets=[ClassifierOutputTarget(tgt)])[0]
                panels += [imgd, show_cam_on_image(imgd, camd, use_rgb=True, colormap=cv2.COLORMAP_INFERNO,
                                                   image_weight=0.55)]
            for cidx, im in enumerate(panels):
                ax = axes[r, cidx]
                ax.imshow(im); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
                for sp in ax.spines.values():
                    sp.set_visible(False)
                if r == 0:
                    ax.set_title(titles[cidx], fontsize=6.8)
            axes[r, 0].set_ylabel(f"True {SHORT[Y[i]]}\nPred {SHORT[tgt]} ({p_[tgt] * 100:.0f}%)"
                                  + ("\n(error)" if kind == "error" else ""), fontsize=6.5)
    fig.tight_layout(pad=0.3)
    save_fig(fig, "fig_gradcam")
step("Part 15 done.")

## Part 16: REPORT.md + paper_numbers.tex + v3_results.zip (`part16_report_bundle.py`)

Writes `REPORT.md`, `paper_numbers.tex` (LaTeX macros) and `v3_results.zip` (all tables, 600 dpi PNG + PDF figures).

In [ ]:
# ==============================================================================
# PART 16/16: REPORT.md + paper_numbers.tex + v3_results.zip
# Kaggle: paste into a NEW code cell directly below Part 15. Run the parts in order.
# ==============================================================================
step("Part 16: writing REPORT.md, paper_numbers.tex and the zip bundle ...")
SUMMARY = dict(
    interim=INTERIM, folds=FOLDS_DONE, models=[nice(*c) for c in CANDS], audit=AUDIT,
    proposed=M_PROP,
    proposed_pooled={k_: (float(v) if isinstance(v, (float, np.floating, int, np.integer)) else v)
                     for k_, v in POOLED.loc[M_PROP].items()},
    proposed_mean_sd={m_: [float(FOLD_DF[FOLD_DF.method == M_PROP][m_].mean()),
                           float(FOLD_DF[FOLD_DF.method == M_PROP][m_].std(ddof=1)) if len(FOLDS_DONE) > 1 else 0.0]
                      for m_ in ("acc", "f1", "auc", "kappa", "mcc")},
    best_single=best_single_m, best_single_f1=float(POOLED.loc[best_single_m, "f1"]),
    source_shift=SS_SUM, classes=CLASSES,
    protocol_comparison=PROTO.to_dict("records") if isinstance(PROTO, pd.DataFrame) else None,
    decision_prior=PRIOR_TAB.to_dict("records"),
    source_shift_models=SS_MODELS.to_dict("records") if isinstance(SS_MODELS, pd.DataFrame) else None,
    failed_models={nice(v_, n_): e_ for (v_, n_), e_ in globals().get("FAILED", {}).items()},
    ensemble_choices=CHOICES, temperatures=TEMPS, runtime_h=(time.time() - T0) / 3600, config=CFG)
(WORK / "summary.json").write_text(json.dumps(SUMMARY, indent=2, default=str))
FOLD_DF.to_csv(TABS / "per_fold_metrics.csv", index=False)
POOLED.to_csv(TABS / "pooled_metrics.csv")
pd.DataFrame({"path": DF.path, "label": Y, "fold": FOLD, "source": SRC,
              **{f"P_{s}": OOF[M_PROP][:, c] for c, s in enumerate(SHORT)}}).to_csv(TABS / "oof_predictions_proposed.csv",
                                                                                   index=False)


def md_table(df):
    cols = [str(c) for c in df.columns]
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    cell = lambda v: "" if (not isinstance(v, str) and pd.isna(v)) else str(v)
    return "\n".join(out + ["| " + " | ".join(cell(v) for v in r) + " |" for r in df.values])


ps = SUMMARY["proposed_mean_sd"]
st_best = STATS.set_index("Model").loc[best_single_m]
rf = REFER.set_index("Model").loc[M_PROP]
pp = POOLED.loc[M_PROP]
rep = [f"# Version 3 ({NC}-class ODIR-5K, referable DR >= grade {CFG['dr_min_grade']}) results report", "",
       "> **INTERIM RESULTS: not all folds/models finished. Do not report these numbers.**" if INTERIM
       else "> Final run: all folds and models finished.", "",
       f"Generated {time.strftime('%Y-%m-%d %H:%M')} · preset `{PRESET}` · {len(FOLDS_DONE)} test folds · "
       f"{len(EVAL)} images evaluated out-of-fold · models: {', '.join(nice(*c) for c in CANDS)}", "",
       "## Headline numbers", "",
       f"- **{M_PROP}**: accuracy **{ps['acc'][0] * 100:.2f} ± {ps['acc'][1] * 100:.2f}%**, macro-F1 "
       f"**{ps['f1'][0] * 100:.2f} ± {ps['f1'][1] * 100:.2f}%**, macro-AUC {ps['auc'][0]:.4f} ± {ps['auc'][1]:.4f}, "
       f"Cohen's kappa {ps['kappa'][0]:.3f} (mean ± SD over test folds); balanced accuracy {pp['bacc'] * 100:.2f}% "
       f"(pooled; decision prior power {CFG['decision_prior_power']}).",
       f"- Best single model, **{best_single_m}**: accuracy {POOLED.loc[best_single_m, 'acc'] * 100:.2f}%, macro-F1 "
       f"{POOLED.loc[best_single_m, 'f1'] * 100:.2f}% (pooled). Proposed minus best single, macro-F1: "
       f"{st_best['ΔF1 vs proposed [95% CI]']} points; McNemar p (Holm) = {st_best['p_mcnemar (Holm)']}.",
       f"- Uncertainty-based referral: accuracy **{rf['Acc@95% cov.']}%** at 95% coverage, **{rf['Acc@90% cov.']}%** "
       f"at 90% and **{rf['Acc@80% cov.']}%** at 80% (the remaining least-confident images go to an ophthalmologist).",
       f"- Referable disease (any disease vs. normal): sensitivity {pp['referable_sens'] * 100:.2f}%, specificity "
       f"{pp['referable_spec'] * 100:.2f}%, AUC {pp['referable_auc']:.4f}.",
       f"- Normal→Glaucoma errors: {int(pp['normal_as_glaucoma'])} (proposed) vs. "
       f"{int(POOLED.loc[best_single_m, 'normal_as_glaucoma'])} (best single).",
       f"- Duplicate audit: {AUDIT['n_images_in_clusters_gt1']} of {AUDIT['n_images']} images are in near-duplicate "
       f"clusters; a naive image-level 70/15/15 split would have put a duplicate of "
       f"{AUDIT['naive_70_15_15_test_with_dup_in_train'] * 100:.1f}% of its test images into training.",
       f"- Data quality: {AUDIT['task_images_before_cleaning']} task images; removed before splitting: "
       f"{AUDIT['excluded_label_conflict']} label conflicts ({AUDIT['label_conflict_clusters']} clusters), "
       f"{AUDIT['excluded_non_fundus']} non-fundus, {AUDIT['excluded_ungradable']} ungradable, "
       f"{AUDIT['excluded_dr_below_referable']} DR below the referable grade, {AUDIT['excluded_multi_disease']} "
       f"multi-disease eyes, {AUDIT['excluded_uncertain_label']} uncertain labels, {AUDIT['excluded_duplicate_copies']} "
       f"duplicate copies; {AUDIT['n_used']} images used.",
       (f"- Protocol comparison ({PROTO.Model.iloc[0]}): patient-level CV accuracy {PROTO['Accuracy (%)'].iloc[0]}% vs. "
        f"{PROTO['Accuracy (%)'].iloc[1]}% on a random image-level split, where "
        f"{PROTO['Test images with the other eye in training (%)'].iloc[1]}% of the test images had the other eye in training."
        if isinstance(PROTO, pd.DataFrame) else "- Protocol comparison: not run."),
       f"- Patients: {AUDIT['odir_patients']} ODIR patients ({AUDIT['odir_patients_with_both_eyes']} with both eyes) "
       f"are kept in one fold; a naive image-level split would have put the other eye or a duplicate of "
       f"{AUDIT['naive_70_15_15_test_with_dup_or_patient_in_train'] * 100:.1f}% of its test images into training.",
       f"- Shortcut baseline (no retina pixels, acquisition metadata only): macro-F1 "
       f"{SHORTCUT.set_index('Predictor').iloc[2]['Macro-F1 (%)']}% vs. {pp['f1'] * 100:.2f}% for {M_PROP}.",
       f"- Image-quality-only baseline (brightness, sharpness, white border): macro-F1 "
       f"{SHORTCUT.set_index('Predictor').iloc[3]['Macro-F1 (%)']}%.", "",
       "## Data quality (tab_data_quality.tex)", "", md_table(DQ_TABLE), "",
       "## Image quality per class (class_quality.csv)", "", md_table(QUALITY), "",
       "## Class viability (class_viability.csv)", "", md_table(VIABILITY.drop(columns=["In shared sources (%)"])), "",
       "## Shortcut baselines (tab_shortcut.tex)", "", md_table(SHORTCUT), "",
       "## Decision prior (tab_decision_prior.tex)", "", md_table(PRIOR_TAB), ""]
if isinstance(PROTO, pd.DataFrame):
    rep += ["## Protocol comparison (tab_protocol.tex)", "", md_table(PROTO), ""]
if isinstance(DR_GRADE, pd.DataFrame):
    rep += ["## DR recall by severity grade (tab_dr_grade.tex)", "", md_table(DR_GRADE), ""]
if isinstance(SS_TABLE, pd.DataFrame):
    rep += ["## Source-shift test (tab_source_shift.tex)", "", md_table(SS_TABLE), ""]
if isinstance(SS_MODELS, pd.DataFrame):
    rep += ["## Source-shift test, all models (tab_source_shift_models.tex)", "", md_table(SS_MODELS), ""]
rep += [
       "## Main results (Table: tab_main_results.tex)", "", md_table(MAIN), "",
       "## Statistics (tab_statistics.tex)", "", md_table(STATS), "",
       "## Per-class performance of the proposed model (tab_per_class.tex)", "", md_table(PER_CLASS), "",
       "## Normal vs. glaucoma and referable disease (tab_normal_glaucoma.tex)", "", md_table(NG), "",
       "## Referral (tab_referral.tex)", "", md_table(REFER), ""]
if isinstance(LABEL_NOISE, pd.DataFrame):
    rep += ["## Label-conflict sensitivity analysis (tab_label_conflicts.tex)", "", md_table(LABEL_NOISE), ""]
if isinstance(EXT_TABLE, pd.DataFrame):
    rep += ["## External validation (tab_external.tex)", "", md_table(EXT_TABLE), ""]
if XAI_OK:
    rep += ["## Explainability metrics (tab_xai.tex)", "", md_table(XAI_DF), ""]
rep += ["## Model complexity (tab_complexity.tex)", "", md_table(CPLX), "",
        "## Ensemble members chosen on validation data, per fold", "", md_table(pd.DataFrame(CHOICES)), "",
        "## Duplicate audit", "", "```json", json.dumps(AUDIT, indent=2), "```", "", "## Figures", ""]
rep += [f"### {f.stem}\n\n![{f.stem}](figures/{f.name})\n" for f in sorted(FIGS.glob("*.png"))]
(WORK / "REPORT.md").write_text("\n".join(rep), encoding="utf-8")

# LaTeX macros: put \input{paper_numbers} in the preamble and write \PropAcc instead of typing numbers.
tex_esc = lambda s: str(s).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")
macros = {
    "PropName": M_PROP, "NFolds": len(FOLDS_DONE), "NImages": len(EVAL),
    "PropAcc": f"{ps['acc'][0] * 100:.2f}", "PropAccSD": f"{ps['acc'][1] * 100:.2f}",
    "PropFOne": f"{ps['f1'][0] * 100:.2f}", "PropFOneSD": f"{ps['f1'][1] * 100:.2f}",
    "PropAUC": f"{ps['auc'][0]:.4f}", "PropKappa": f"{ps['kappa'][0]:.3f}",
    "PropECE": f"{pp['ece'] * 100:.2f}",
    "BestSingleName": best_single_m, "BestSingleAcc": f"{POOLED.loc[best_single_m, 'acc'] * 100:.2f}",
    "BestSingleFOne": f"{POOLED.loc[best_single_m, 'f1'] * 100:.2f}",
    "PValueVsBest": st_best["p_mcnemar (Holm)"],
    "AccAtNinetyFiveCov": rf["Acc@95% cov."], "AccAtNinetyCov": rf["Acc@90% cov."], "AccAtEightyCov": rf["Acc@80% cov."],
    "RefSens": f"{pp['referable_sens'] * 100:.2f}", "RefSpec": f"{pp['referable_spec'] * 100:.2f}",
    "RefAUC": f"{pp['referable_auc']:.4f}",
    "NormAsGlauProp": int(pp["normal_as_glaucoma"]),
    "NormAsGlauBest": int(POOLED.loc[best_single_m, "normal_as_glaucoma"]),
    "DupImages": AUDIT["n_images_in_clusters_gt1"],
    "NaiveLeakPct": f"{AUDIT['naive_70_15_15_test_with_dup_in_train'] * 100:.1f}",
    "OfficialLeakPct": f"{(AUDIT['official_eval_with_dup_in_train'] or 0) * 100:.1f}",
    "ConflictImages": AUDIT["label_conflict_images"], "ConflictClusters": AUDIT["label_conflict_clusters"],
    "NonFundusImages": AUDIT["non_fundus_images"],
    "ExcludedImages": AUDIT["excluded_label_conflict"] + AUDIT["excluded_non_fundus"] + AUDIT["excluded_ungradable"]
    + AUDIT["excluded_duplicate_copies"] + AUDIT["excluded_multi_disease"] + AUDIT["excluded_uncertain_label"]
    + AUDIT["excluded_dr_below_referable"], "UsedImages": AUDIT["n_used"],
    "UngradableImages": AUDIT["excluded_ungradable"], "DupCopiesRemoved": AUDIT["excluded_duplicate_copies"],
    "TaskImagesBefore": AUDIT["task_images_before_cleaning"],
    "NClasses": NC, "ODIRPatients": AUDIT["odir_patients"],
    "NaivePatientLeakPct": f"{AUDIT['naive_70_15_15_test_with_dup_or_patient_in_train'] * 100:.1f}",
    "PropBalAcc": f"{pp['bacc'] * 100:.2f}",
    "ShortcutFOne": SHORTCUT.set_index("Predictor").iloc[2]["Macro-F1 (%)"],
    "QualityOnlyFOne": SHORTCUT.set_index("Predictor").iloc[3]["Macro-F1 (%)"],
    "MultiDiseaseImages": AUDIT["excluded_multi_disease"], "UncertainImages": AUDIT["excluded_uncertain_label"],
    "MildDRRemoved": AUDIT["excluded_dr_below_referable"], "DecisionPriorPower": CFG["decision_prior_power"],
    "SourceOnlyFOne": SHORTCUT.set_index("Predictor").iloc[1]["Macro-F1 (%)"],
}
if isinstance(PROTO, pd.DataFrame):
    macros.update({"CVAccSameModel": PROTO["Accuracy (%)"].iloc[0], "NaiveSplitAcc": PROTO["Accuracy (%)"].iloc[1],
                   "NaiveSplitFOne": PROTO["Macro-F1 (%)"].iloc[1],
                   "NaiveSplitLeakPct": PROTO["Test images with the other eye in training (%)"].iloc[1]})
for _r in PRIOR_TAB.itertuples(index=False):
    _k = "Natural" if _r[0].startswith("natural") else "Sqrt" if _r[0].startswith("square") else "Balanced"
    macros[f"PriorAcc{_k}"], macros[f"PriorBalAcc{_k}"] = _r[1], _r[2]
if isinstance(DR_GRADE, pd.DataFrame):
    for _r in DR_GRADE.itertuples(index=False):
        _k = {"Mild NPDR": "Mild", "Moderate NPDR": "Moderate", "Severe NPDR": "Severe", "Proliferative DR": "PDR"}.get(_r[0])
        if _k:
            macros[f"DRRecall{_k}"] = _r[2]
if SS_SUM:
    macros.update({"SSAcc": f"{SS_SUM['acc'] * 100:.2f}", "SSFOne": f"{SS_SUM['f1'] * 100:.2f}",
                   "SSTestImages": SS_SUM["n_test"]})
    if SS_SUM["ref_acc"] is not None:
        macros.update({"SSRefAcc": f"{SS_SUM['ref_acc'] * 100:.2f}", "SSRefFOne": f"{SS_SUM['ref_f1'] * 100:.2f}"})
(WORK / "paper_numbers.tex").write_text(
    "% Auto-generated by the Version 3 notebook" + (" -- INTERIM, do not use" if INTERIM else "") + "\n" +
    "\n".join(f"\\newcommand{{\\{k}}}{{{tex_esc(v)}}}" for k, v in macros.items()) + "\n", encoding="utf-8")

tmp = WORK / "_bundle"
if tmp.exists():
    shutil.rmtree(tmp)
tmp.mkdir()
for sub in ("figures", "tables"):
    shutil.copytree(WORK / sub, tmp / sub)
for f in ("audit.json", "summary.json", "splits.csv", "data_quality.csv", "fingerprint.txt", "REPORT.md",
          "paper_numbers.tex"):
    if (WORK / f).exists():
        shutil.copy(WORK / f, tmp / f)
bundle = shutil.make_archive(str(WORK / "v3_results"), "zip", root_dir=tmp)
shutil.rmtree(tmp)
if IS_KAGGLE:
    try:
        shutil.copy(bundle, "/kaggle/working/v3_results.zip")
    except Exception:
        pass
print(f"\nProposed ({M_PROP}): acc {SUMMARY['proposed_mean_sd']['acc'][0] * 100:.2f} ± "
      f"{SUMMARY['proposed_mean_sd']['acc'][1] * 100:.2f}%, macro-F1 {SUMMARY['proposed_mean_sd']['f1'][0] * 100:.2f} ± "
      f"{SUMMARY['proposed_mean_sd']['f1'][1] * 100:.2f}%  |  best single: {best_single_m}")
print("Bundle:", bundle, "| total runtime %.2f h" % SUMMARY["runtime_h"])
step(f"ALL DONE. Download {WORK.name}/v3_results.zip (or /kaggle/working/v3_results.zip) from the Output tab.")
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(bundle), result_html_prefix="Download all results (600 dpi PNG + PDF figures, tables): "))
except Exception:
    pass
if INTERIM:
    print("*** INTERIM: finish all folds/models before reporting numbers. ***")